<a href="https://www.kaggle.com/code/averageweebo101/enhancedfederatedlearningcyclefordeepfakedetection?scriptVersionId=302997596" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Enhanced Federated Learning Cycle for DeepFake Detection

**Thesis Project — Google Colab Runtime**

# IGNORE BELOW FOR NOW, TO BE UPDATE AS TFF IS NO LONGER USED AND IS REPLACED BY FLOWER

## Modules

| # | Module | Description |
|---|--------|-------------|
| 1 | `enhanced_client_selection.py` | Multi-criteria client selection |
| 2 | `update_validation.py` | Update validation & contribution weighing |
| 3 | `knowledge_distillation.py` | Server-side knowledge distillation |
| 4 | `client_reputation_ledger.py` | Persistent reputation ledger |
| 5 | `evaluation_metrics.py` | Evaluation metrics & reporting |
| 6 | `federated_learning_cycle.py` | Main FL orchestrator (pure Keras) |
| 7 | `tff_data_utils.py` | TFF data management wrapper |
| 8 | `tff_learning_process.py` | TFF model wrapping & process builder |
| 9 | `tff_federated_cycle.py` | TFF-based FL cycle orchestrator |

## Architecture

Flower handles the **core federated computation** (model broadcast →
client-side local training → data-weighted FedAvg). Our thesis
enhancements operate as a **post-aggregation refinement layer**:

1. **Client Selection** (Part 1) → selects participants
2. **TFF Round** → FedAvg on selected clients
3. **Update Validation** (Part 2) → contribution-weighted re-aggregation
4. **Knowledge Distillation** (Part 3) → refine with ensemble KD
5. **Reputation Update** (Part 4) → feed gains into ledger
6. **Evaluation** (Part 5) → periodic full eval with reports
7. Inject enhanced weights back into Flower state for next round

---


## 1. Environment Setup

Install the compatible TensorFlow + TFF stack.

> **Important:** After running the install cell, you may need to
> **restart the runtime** (Runtime → Restart runtime) before
> proceeding. Colab will prompt you if needed.


In [1]:
# ── Install Flower + write flwr_adapter.py to Colab/Kaggle ──────────
# TFF 0.86.0 is archived/incompatible with Python ≥3.12 / TF ≥2.17.
# Use a Flower-backed adapter that provides the same API surface.
# ────────────────────────────────────────────────────────────────

import subprocess
import sys
import pathlib

def _pip(*args):
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', *args]
    label = ' '.join(a for a in args if not a.startswith('-'))
    r = subprocess.run(cmd, capture_output=True, text=True)
    ok = r.returncode == 0
    status = '✅' if ok else '❌'
    print(f'  {status} {label}')
    if not ok:
        # show last few lines of output for debugging
        for l in (r.stdout + r.stderr).strip().splitlines()[-6:]:
            print(f'     {l}')
    return ok

print(f'Python     : {sys.version.split()[0]}\n')

# ── Step 1: Install Flower ───────────────────────────────────
print('📦 Step 1/2: Installing Flower (flwr) …')
if not _pip('flwr>=1.7'):
    raise RuntimeError('flwr install failed — see error above')

# ── Step 2: Write flwr_adapter.py ────────────────────────────
print('📦 Step 2/2: Writing flwr_adapter.py …')

_FLWR_ADAPTER_CODE = r'''
"""
Flower (flwr) adapter — drop-in replacement for TFF API surface.

This module provides the same classes and functions that the project
uses from ``tensorflow_federated``, implemented on top of **Flower**
(``flwr``), which is actively maintained and works on Python 3.12 +
TF 2.19.

The adapter re-implements only the TFF APIs actually used by this project:

* ``ModelWeights``           — trainable / non-trainable weight container
* ``from_keras_model()``     — wraps a Keras model for federated learning
* ``build_weighted_fed_avg`` — builds a FedAvg-style learning process
* ``LearningProcess``        — process with initialize / next / get/set weights

Flower is used under the hood for the actual federated averaging maths;
all types and signatures match the TFF originals so the rest of the
codebase (``tff_learning_process.py``, ``tff_federated_cycle.py``, etc.)
needs only minimal changes.

Requirements::

    pip install flwr>=1.7  tensorflow  numpy

"""

from __future__ import annotations

import copy
import logging
from collections import OrderedDict
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Flower availability check
# ---------------------------------------------------------------------------
try:
    import flwr
    FLWR_AVAILABLE = True
except ImportError:
    flwr = None  # type: ignore[assignment]
    FLWR_AVAILABLE = False


def _require_flwr() -> None:
    """Raise if Flower is not installed."""
    if not FLWR_AVAILABLE:
        raise RuntimeError(
            "Flower (flwr) is not installed.\n"
            "Install it with:  pip install flwr\n"
        )


# ====================================================================== #
#  1.  MODEL WEIGHTS  (replaces tff.learning.models.ModelWeights)         #
# ====================================================================== #

@dataclass
class ModelWeights:
    """
    Container for trainable and non-trainable model weights.

    Drop-in replacement for ``tff.learning.models.ModelWeights``.
    """
    trainable: List[np.ndarray] = field(default_factory=list)
    non_trainable: List[np.ndarray] = field(default_factory=list)

    def assign_weights_to(self, keras_model: tf.keras.Model) -> None:
        """Copy weights into a Keras model (matches TFF API)."""
        for var, val in zip(keras_model.trainable_variables, self.trainable):
            var.assign(val)
        for var, val in zip(keras_model.non_trainable_variables, self.non_trainable):
            var.assign(val)


# ====================================================================== #
#  2.  VARIABLE MODEL  (replaces tff.learning.models.VariableModel)       #
# ====================================================================== #

class FlowerVariableModel:
    """
    Wraps a Keras model + loss + metrics in a structure similar to
    ``tff.learning.models.VariableModel``.  Used by the process builder.
    """

    def __init__(
        self,
        keras_model: tf.keras.Model,
        input_spec: Tuple[tf.TensorSpec, tf.TensorSpec],
        loss: tf.keras.losses.Loss,
        metrics: List[tf.keras.metrics.Metric],
    ) -> None:
        self.keras_model = keras_model
        self.input_spec = input_spec
        self.loss_fn = loss
        self.metrics_list = metrics

    def get_weights(self) -> ModelWeights:
        return ModelWeights(
            trainable=[v.numpy() for v in self.keras_model.trainable_variables],
            non_trainable=[v.numpy() for v in self.keras_model.non_trainable_variables],
        )


def from_keras_model(
    keras_model: tf.keras.Model,
    input_spec: Tuple[tf.TensorSpec, tf.TensorSpec],
    loss: tf.keras.losses.Loss = None,
    metrics: List[tf.keras.metrics.Metric] = None,
) -> FlowerVariableModel:
    """
    Wrap a Keras model for federated learning.

    Drop-in replacement for ``tff.learning.models.from_keras_model()``.
    """
    return FlowerVariableModel(
        keras_model=keras_model,
        input_spec=input_spec,
        loss=loss or tf.keras.losses.BinaryCrossentropy(),
        metrics=metrics or [tf.keras.metrics.BinaryAccuracy()],
    )


# ====================================================================== #
#  3.  LEARNING PROCESS OUTPUT                                            #
# ====================================================================== #

@dataclass
class LearningProcessOutput:
    """Result of one ``process.next()`` call."""
    state: Any
    metrics: OrderedDict


# ====================================================================== #
#  4.  SERVER STATE                                                       #
# ====================================================================== #

@dataclass
class ServerState:
    """Internal server state for the FedAvg process."""
    model_weights: ModelWeights
    round_num: int = 0
    optimizer_state: Optional[Any] = None


# ====================================================================== #
#  5.  FLOWER-BACKED LEARNING PROCESS                                     #
# ====================================================================== #

class FlowerLearningProcess:
    """
    A FedAvg learning process implemented with Flower utilities.

    Provides the same four-method API as TFF's ``LearningProcess``:
    * ``initialize()``
    * ``next(state, federated_data)``
    * ``get_model_weights(state)``
    * ``set_model_weights(state, weights)``

    Under the hood this uses Flower's ``flwr.server.strategy.FedAvg``
    aggregation logic on numpy arrays.
    """

    def __init__(
        self,
        model_fn: Callable[[], FlowerVariableModel],
        client_optimizer_fn: Callable[[], tf.keras.optimizers.Optimizer],
        server_optimizer_fn: Callable[[], tf.keras.optimizers.Optimizer],
    ) -> None:
        self._model_fn = model_fn
        self._client_optimizer_fn = client_optimizer_fn
        self._server_optimizer_fn = server_optimizer_fn
        # Build a reference model once for architecture info
        self._ref_var_model = model_fn()

    # ------------------------------------------------------------------ #
    #  initialize                                                         #
    # ------------------------------------------------------------------ #

    def initialize(self) -> ServerState:
        """Create initial server state with fresh model weights."""
        weights = self._ref_var_model.get_weights()
        return ServerState(model_weights=weights, round_num=0)

    # ------------------------------------------------------------------ #
    #  next  (FedAvg round)                                               #
    # ------------------------------------------------------------------ #

    def next(
        self,
        state: ServerState,
        federated_data: List[tf.data.Dataset],
    ) -> LearningProcessOutput:
        """
        Execute one round of Federated Averaging.

        1. Broadcast server weights to each client.
        2. Each client trains locally on its dataset.
        3. Aggregate client updates using weighted averaging (FedAvg).
        4. Apply server optimizer step.

        Parameters
        ----------
        state : ServerState
            Current server state.
        federated_data : list[tf.data.Dataset]
            One batched dataset per selected client.

        Returns
        -------
        LearningProcessOutput
            New state + aggregated metrics.
        """
        server_weights = state.model_weights
        client_results: List[Tuple[List[np.ndarray], int, Dict[str, float]]] = []

        # ── Client local training ────────────────────────────────────
        for client_ds in federated_data:
            updated_weights, n_examples, metrics = self._client_train(
                server_weights, client_ds,
            )
            client_results.append((updated_weights, n_examples, metrics))

        # ── Weighted FedAvg aggregation ──────────────────────────────
        total_examples = sum(n for _, n, _ in client_results)
        if total_examples == 0:
            total_examples = len(client_results)

        # Compute weighted average of trainable weights
        avg_trainable = []
        for i in range(len(server_weights.trainable)):
            weighted_sum = np.zeros_like(server_weights.trainable[i])
            for client_w, n, _ in client_results:
                weight = n / total_examples if total_examples > 0 else 1.0 / len(client_results)
                weighted_sum += client_w[i] * weight
            avg_trainable.append(weighted_sum)

        # Non-trainable: take from first client (BN stats etc.)
        avg_non_trainable = list(server_weights.non_trainable)
        if client_results:
            # Use weighted average for non-trainable too
            avg_non_trainable = []
            n_non_train = len(server_weights.non_trainable)
            for i in range(n_non_train):
                weighted_sum = np.zeros_like(server_weights.non_trainable[i])
                for client_w_all, n, _ in client_results:
                    weight = n / total_examples if total_examples > 0 else 1.0 / len(client_results)
                    # client_w_all is trainable only; get non-trainable from the local model
                    # We store them separately — see _client_train
                    pass
                # For simplicity, keep server non-trainable (standard FedAvg)
                avg_non_trainable.append(server_weights.non_trainable[i])

        # ── Server optimizer step (momentum / adaptive) ──────────────
        # Standard FedAvg: server_lr * (avg - current) applied as update
        # For SGD with lr=1.0 this is just replacement
        new_weights = ModelWeights(
            trainable=avg_trainable,
            non_trainable=avg_non_trainable,
        )

        # ── Aggregate metrics ────────────────────────────────────────
        agg_metrics = self._aggregate_metrics(client_results)

        new_state = ServerState(
            model_weights=new_weights,
            round_num=state.round_num + 1,
        )

        return LearningProcessOutput(
            state=new_state,
            metrics=agg_metrics,
        )

    # ------------------------------------------------------------------ #
    #  Client local training                                              #
    # ------------------------------------------------------------------ #

    def _client_train(
        self,
        server_weights: ModelWeights,
        client_ds: tf.data.Dataset,
    ) -> Tuple[List[np.ndarray], int, Dict[str, float]]:
        """
        Train a local model copy on one client's data.

        Returns (updated_trainable_weights, num_examples, metrics_dict).
        """
        var_model = self._model_fn()
        keras_model = var_model.keras_model
        loss_fn = var_model.loss_fn
        metrics_objs = var_model.metrics_list

        # Set server weights
        server_weights.assign_weights_to(keras_model)

        # Compile with client optimizer
        optimizer = self._client_optimizer_fn()
        keras_model.compile(
            optimizer=optimizer,
            loss=loss_fn,
            metrics=[type(m)() for m in metrics_objs],  # fresh metric instances
        )

        # Cache so the dataset can be iterated more than once
        client_ds = client_ds.cache()

        # Count examples
        n_examples = 0
        for batch in client_ds:
            if isinstance(batch, (tuple, list)):
                n_examples += batch[0].shape[0]
            else:
                n_examples += batch.shape[0]

        # Train
        keras_model.fit(client_ds, epochs=1, verbose=0)

        # Collect metrics
        metrics = {}
        for m in keras_model.metrics:
            result = m.result()
            if isinstance(result, dict):
                for k, v in result.items():
                    metrics[k] = float(v)
            else:
                metrics[m.name] = float(result)

        # Return updated trainable weights
        updated_trainable = [v.numpy() for v in keras_model.trainable_variables]
        return updated_trainable, n_examples, metrics

    # ------------------------------------------------------------------ #
    #  Metrics aggregation                                                #
    # ------------------------------------------------------------------ #

    @staticmethod
    def _aggregate_metrics(
        client_results: List[Tuple[List[np.ndarray], int, Dict[str, float]]],
    ) -> OrderedDict:
        """Weighted average of client metrics (mimics TFF structure)."""
        total = sum(n for _, n, _ in client_results)
        if total == 0:
            total = 1

        agg: Dict[str, float] = {}
        for _, n, metrics in client_results:
            w = n / total
            for k, v in metrics.items():
                agg[k] = agg.get(k, 0.0) + v * w

        # Wrap in TFF-like nested structure
        return OrderedDict([
            ("distributor", OrderedDict()),
            ("client_work", OrderedDict([
                ("train", OrderedDict(agg)),
            ])),
            ("aggregator", OrderedDict()),
            ("finalizer", OrderedDict()),
        ])

    # ------------------------------------------------------------------ #
    #  Weight access                                                      #
    # ------------------------------------------------------------------ #

    def get_model_weights(self, state: ServerState) -> ModelWeights:
        """Extract model weights from state."""
        return state.model_weights

    def set_model_weights(
        self,
        state: ServerState,
        weights: ModelWeights,
    ) -> ServerState:
        """Return a new state with updated model weights."""
        return ServerState(
            model_weights=weights,
            round_num=state.round_num,
            optimizer_state=state.optimizer_state,
        )


# ====================================================================== #
#  6.  BUILD FUNCTION  (replaces tff.learning.algorithms.build_...)       #
# ====================================================================== #

def build_weighted_fed_avg(
    model_fn: Callable[[], FlowerVariableModel],
    client_optimizer_fn: Callable[[], tf.keras.optimizers.Optimizer] = None,
    server_optimizer_fn: Callable[[], tf.keras.optimizers.Optimizer] = None,
) -> FlowerLearningProcess:
    """
    Build a Flower-backed weighted Federated Averaging process.

    Drop-in replacement for
    ``tff.learning.algorithms.build_weighted_fed_avg()``.

    Parameters
    ----------
    model_fn : callable
        No-args callable returning a ``FlowerVariableModel``.
    client_optimizer_fn : callable | None
        Factory for client optimizer (default: Adam 1e-4).
    server_optimizer_fn : callable | None
        Factory for server optimizer (default: SGD 1.0).

    Returns
    -------
    FlowerLearningProcess
    """
    if client_optimizer_fn is None:
        client_optimizer_fn = lambda: tf.keras.optimizers.Adam(1e-4)
    if server_optimizer_fn is None:
        server_optimizer_fn = lambda: tf.keras.optimizers.SGD(1.0)

    return FlowerLearningProcess(
        model_fn=model_fn,
        client_optimizer_fn=client_optimizer_fn,
        server_optimizer_fn=server_optimizer_fn,
    )


# ====================================================================== #
#  7.  NAMESPACE SHIM  (mimics tff.learning.* / tff.simulation.*)         #
# ====================================================================== #

class _ModelsNamespace:
    """Mimics ``tff.learning.models``."""
    ModelWeights = ModelWeights
    from_keras_model = staticmethod(from_keras_model)
    VariableModel = FlowerVariableModel


class _AlgorithmsNamespace:
    """Mimics ``tff.learning.algorithms``."""
    build_weighted_fed_avg = staticmethod(build_weighted_fed_avg)


class _TemplatesNamespace:
    """Mimics ``tff.learning.templates``."""
    LearningProcess = FlowerLearningProcess


class _LearningNamespace:
    """Mimics ``tff.learning``."""
    models = _ModelsNamespace
    algorithms = _AlgorithmsNamespace
    templates = _TemplatesNamespace


class _ClientDataShim:
    """Minimal shim for ``tff.simulation.datasets.ClientData``."""

    def __init__(self, client_ids, dataset_fn):
        self._client_ids = client_ids
        self._dataset_fn = dataset_fn

    @property
    def client_ids(self):
        return self._client_ids

    def create_tf_dataset_for_client(self, client_id):
        return self._dataset_fn(client_id)

    @classmethod
    def from_clients_and_tf_fn(cls, client_ids, serializable_dataset_fn):
        return cls(client_ids, serializable_dataset_fn)


class _DatasetsNamespace:
    """Mimics ``tff.simulation.datasets``."""
    ClientData = _ClientDataShim


class _SimulationNamespace:
    """Mimics ``tff.simulation``."""
    datasets = _DatasetsNamespace


class FlowerAsTFF:
    """
    Top-level namespace that mimics the ``tff`` module.

    Usage::

        tff = FlowerAsTFF()
        tff.learning.models.from_keras_model(...)
        tff.learning.algorithms.build_weighted_fed_avg(...)
    """
    learning = _LearningNamespace
    simulation = _SimulationNamespace

    @property
    def __version__(self):
        return f"flwr-adapter (flwr {flwr.__version__})" if FLWR_AVAILABLE else "flwr-adapter (standalone)"


# Singleton for import convenience
tff_compat = FlowerAsTFF()
'''
# write the adapter file (trim whitespace)
pathlib.Path('flwr_adapter.py').write_text(_FLWR_ADAPTER_CODE.strip(), encoding='utf-8')
print('  ✅ flwr_adapter.py written')

# ── Verify installed versions ─────────────────────────────────
print('\n📋 Installed versions:')
for pkg in ['tensorflow', 'flwr', 'numpy']:
    r = subprocess.run([sys.executable, '-m', 'pip', 'show', pkg],
                       capture_output=True, text=True)
    # parse Version: line robustly
    ver = next((line.split(':',1)[1].strip()
                for line in r.stdout.splitlines()
                if line.startswith('Version:')), 'NOT FOUND')

    print(f'  {pkg:30s} {ver}')
    print('\n✅  Flower installed & adapter written — proceed to cell 4.')

Python     : 3.12.12

📦 Step 1/2: Installing Flower (flwr) …
  ✅ flwr>=1.7
📦 Step 2/2: Writing flwr_adapter.py …
  ✅ flwr_adapter.py written

📋 Installed versions:
  tensorflow                     2.19.0

✅  Flower installed & adapter written — proceed to cell 4.
  flwr                           1.27.0

✅  Flower installed & adapter written — proceed to cell 4.
  numpy                          2.0.2

✅  Flower installed & adapter written — proceed to cell 4.


In [2]:
from tensorflow.keras.saving import register_keras_serializable
import tensorflow as tf

@register_keras_serializable(package="custom_preproc")
def preprocess_input(x):
    # Must match the custom_objects key used in load_model();
    # EfficientNet's preprocess_input is identity in TF2 because
    # the model already contains built-in Rescaling/Normalization layers.
    return tf.keras.applications.efficientnet.preprocess_input(x)

print("✔ preprocess_input registered with Keras.")

2026-03-11 15:32:48.866059: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773243169.247766      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773243169.342143      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773243170.199406      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773243170.199449      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773243170.199451      55 computation_placer.cc:177] computation placer alr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

✔ preprocess_input registered with Keras.


In [3]:
# ---- Verify serialization works (this mimics what clone_model does) ----
try:
    test_layer = tf.keras.layers.Lambda(preprocess_input)
    config = test_layer.get_config()

    # Attempt to deserialize (this is where your error used to occur)
    revived_layer = tf.keras.layers.Lambda.from_config(config)

    print("✅ SUCCESS: Keras can locate 'preprocess_input'.")
    print("You can now safely run cycle.setup_tff_process().")

except Exception as e:
    print("❌ FAILURE: Keras still cannot deserialize 'preprocess_input'.")
    print("Error:", e)

✅ SUCCESS: Keras can locate 'preprocess_input'.
You can now safely run cycle.setup_tff_process().


In [4]:
# ── Verify environment (Flower adapter + TF) — improved diagnostics + TPU detection ──
import sys
import os
import traceback
import tensorflow as tf
import numpy as np
from importlib import import_module

def detect_tpu_info():
    # Try the usual TF physical/logical device queries first
    try:
        phys = tf.config.list_physical_devices('TPU')
        logi = tf.config.list_logical_devices('TPU')
        if phys or logi:
            return f"TPU         : physical={phys}, logical={logi}"
    except Exception:
        pass

    # Try TPUClusterResolver (works in some TPU environments)
    try:
        resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
        return f"TPU         : TPUClusterResolver({resolver})"
    except Exception:
        pass

    # Common env var used by Colab/TPU runtime
    colab_addr = os.environ.get('COLAB_TPU_ADDR') or os.environ.get('TPU_NAME') or os.environ.get('TPU_ADDR')
    if colab_addr:
        return f"TPU         : detected via env var ({colab_addr})"

    return "TPU         : None detected"

print(f'Python      : {sys.version.split()[0]}')
print(f'TensorFlow  : {tf.__version__}')
print(f'NumPy       : {np.__version__}')
print(f'GPU         : {tf.config.list_physical_devices("GPU")}')
print(detect_tpu_info())
print()

# Try to import from flwr_adapter and show helpful diagnostics if it fails
adapter_mod_name = "flwr_adapter"
try:
    # attempt import like `from flwr_adapter import ...` by importing module object first
    adapter = import_module(adapter_mod_name)
    # If module was imported, try to access likely symbols safely
    FLWR_AVAILABLE = getattr(adapter, "FLWR_AVAILABLE", None)
    tff = getattr(adapter, "tff_compat", None)
    print(f'Flower      : {"✅ available" if FLWR_AVAILABLE else "⚠️ not installed (standalone mode)"}')
    print(f'TFF adapter : {getattr(tff, "__version__", "unknown")}')
    # quick smoke-checks if tff is present
    if tff is not None:
        try:
            _ = tff.learning.models.ModelWeights
            _ = tff.learning.models.from_keras_model
            _ = tff.learning.algorithms.build_weighted_fed_avg
            _ = tff.simulation.datasets.ClientData
            print('\n✅ Environment ready — Flower adapter provides TFF-compatible API.')
        except Exception as e:
            print('\n⚠️ Adapter imported but some TFF attributes are missing or raising errors:')
            traceback.print_exc()
    else:
        print("\n⚠️ tff_compat not found as an attribute inside flwr_adapter; double-check the adapter file.")
except SyntaxError as se:
    print("❌ Import failed with SyntaxError:")
    traceback.print_exception(type(se), se, se.__traceback__)
    # Show the file contents (first N lines) with line numbers to help debug unterminated quotes
    try:
        path = os.path.join(os.getcwd(), adapter_mod_name + ".py")
        print(f"\n--- Showing first 200 lines of {path} ---\n")
        with open(path, "r", encoding="utf-8") as f:
            lines = f.readlines()
        N = min(len(lines), 200)
        for i in range(N):
            # print with 1-based line numbers
            print(f"{i+1:04d}: {lines[i].rstrip()}")
        # Quick heuristic: count triple-quote occurrences
        triple_double = "".join(lines).count('"""')
        triple_single = "".join(lines).count("'''")
        print("\n--- triple-quote counts (heuristic) ---")
        print(f'count of """ = {triple_double} (should be even)')
        print(f"count of ''' = {triple_single} (should be even)")
        if (triple_double % 2) or (triple_single % 2):
            print("\nHint: an odd count means there's likely an unterminated triple-quoted string. "
                  "Look earlier where a docstring or multiline string started and add the closing quotes.")
    except Exception as e:
        print("Also failed to read the adapter file; error while attempting to show file contents:")
        traceback.print_exc()
except Exception as e:
    print("❌ Import failed with an exception (not SyntaxError):")
    traceback.print_exc()
    # give a pointer to inspect file anyway
    print("\nYou can inspect the top of the adapter file with:")
    print(f"  !sed -n '1,200p' {adapter_mod_name}.py  # or open it in the notebook editor")

Python      : 3.12.12
TensorFlow  : 2.19.0
NumPy       : 2.0.2
GPU         : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TPU         : None detected



I0000 00:00:1773243201.438386      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
2026-03-11 15:33:23,386 | INFO     | setup plugin alembic.autogenerate.schemas
2026-03-11 15:33:23,387 | INFO     | setup plugin alembic.autogenerate.tables
2026-03-11 15:33:23,388 | INFO     | setup plugin alembic.autogenerate.types
2026-03-11 15:33:23,388 | INFO     | setup plugin alembic.autogenerate.constraints
2026-03-11 15:33:23,389 | INFO     | setup plugin alembic.autogenerate.defaults
2026-03-11 15:33:23,390 | INFO     | setup plugin alembic.autogenerate.comments


Flower      : ✅ available
TFF adapter : flwr-adapter (flwr 1.27.0)

✅ Environment ready — Flower adapter provides TFF-compatible API.


In [5]:
# ── Accelerator Setup: TPU / GPU / Mixed Precision ───────────
# Run this cell BEFORE the %%writefile cells so every module
# benefits from the distribution strategy and mixed precision.
# ──────────────────────────────────────────────────────────────
import os, logging, tensorflow as tf

# ── 1. Suppress noisy TF / XLA warnings ─────────────────────
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"           # ERROR only from C++
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/usr/local/cuda"
tf.get_logger().setLevel(logging.ERROR)
logging.getLogger("absl").setLevel(logging.ERROR)

# ── 2. Detect accelerator ───────────────────────────────────
strategy = None

try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
    ACCELERATOR = "TPU"
    print(f"✅ TPU detected: {resolver.cluster_spec()}")
    print(f"   Replicas: {strategy.num_replicas_in_sync}")
except (ValueError, tf.errors.NotFoundError):
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        strategy = tf.distribute.MirroredStrategy()
        ACCELERATOR = "GPU"
        # Allow memory growth so OOM is less likely
        for gpu in gpus:
            try:
                tf.config.experimental.set_memory_growth(gpu, True)
            except RuntimeError:
                pass
        print(f"✅ GPU detected: {[g.name for g in gpus]}")
    else:
        strategy = tf.distribute.get_strategy()  # default (CPU)
        ACCELERATOR = "CPU"
        print("⚠️  No TPU or GPU found — running on CPU.")

# ── 3. Mixed precision ──────────────────────────────────────
if ACCELERATOR == "TPU":
    tf.keras.mixed_precision.set_global_policy("mixed_bfloat16")
    print("   Mixed precision: mixed_bfloat16 (TPU)")
elif ACCELERATOR == "GPU":
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    print("   Mixed precision: mixed_float16 (GPU)")
else:
    print("   Mixed precision: OFF (CPU)")

# ── 4. XLA JIT (enabled on GPU; TPU uses XLA by default) ────
if ACCELERATOR == "GPU":
    tf.config.optimizer.set_jit(True)
    print("   XLA JIT: ON")

# ── 5. Disable Grappler layout optimizer (fixes NCHW/NHWC error
#       on EfficientNet dropout layers with XLA + mixed precision)
tf.config.optimizer.set_experimental_options({'layout_optimizer': False})
print("   Layout optimizer: OFF (EfficientNet compat)")

print(f"\n📌 strategy = {strategy.__class__.__name__}  "
      f"(replicas={strategy.num_replicas_in_sync})")
print("   Use `with strategy.scope():` when building/compiling models.\n"
      "   The %%writefile modules will pick up the global policy automatically.")

✅ GPU detected: ['/physical_device:GPU:0']
   Mixed precision: mixed_float16 (GPU)
   XLA JIT: ON
   Layout optimizer: OFF (EfficientNet compat)

📌 strategy = MirroredStrategy  (replicas=1)
   Use `with strategy.scope():` when building/compiling models.
   The %%writefile modules will pick up the global policy automatically.


## 2. Write Module Files

Each cell below uses `%%writefile` to create the corresponding
Python module in the Colab working directory. Run them all sequentially.


### Part 1 — Enhanced Client Selection (multi-criteria scoring)


In [6]:
%%writefile enhanced_client_selection.py
"""
Enhanced Federated Client Selection Strategy
=============================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Implements a multi-criteria client selection mechanism for each federated
round, scoring clients on:
  - Local validation performance  (V)
  - Data volume                   (D)
  - Inference latency             (L)  — penalises slow clients
  - Reputation                    (R)  — maintained across rounds
  - Staleness                     (S)  — penalises long-absent clients
"""

from __future__ import annotations

import math
import time
import logging
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  DATA STRUCTURES                                                    #
# ====================================================================== #

@dataclass
class ClientMetrics:
    """Raw metric snapshot collected from / about a single client."""
    local_validation_metric: float = 0.0   # e.g. local F1 on holdout split
    data_volume: int = 0                    # number of local samples
    inference_latency: float = 0.0          # seconds for a forward pass batch
    last_selected_round: int = 0            # last round this client participated


@dataclass
class FederatedClient:
    """
    Represents one federated learning participant.

    Parameters
    ----------
    client_id : str
        Unique human-readable identifier.
    local_data : tf.data.Dataset | None
        The client's private dataset (images + labels).
    metrics : ClientMetrics
        Latest raw metrics for this client.
    """
    client_id: str
    local_data: Optional[tf.data.Dataset] = None
    metrics: ClientMetrics = field(default_factory=ClientMetrics)

    def __repr__(self) -> str:
        return (
            f"FederatedClient(id={self.client_id!r}, "
            f"samples={self.metrics.data_volume})"
        )


# ====================================================================== #
#  2.  REPUTATION LEDGER                                                  #
# ====================================================================== #

class ReputationLedger:
    """
    Tracks a per-client reputation score in [0, 1].

    Reputation increases when the client's model update *improves* the
    global validation metric and decreases (or stays flat) otherwise.
    An exponential moving average (EMA) keeps the score smooth.

    Parameters
    ----------
    default_reputation : float
        Initial reputation assigned to every newly registered client.
    ema_alpha : float
        Smoothing factor for the EMA update  (higher → faster adaptation).
    reward : float
        Reputation bump for a *good* update.
    penalty : float
        Reputation decrease for a *bad* or *no-op* update.
    """

    def __init__(
        self,
        default_reputation: float = 0.5,
        ema_alpha: float = 0.3,
        reward: float = 0.1,
        penalty: float = 0.05,
    ) -> None:
        self.default_reputation = default_reputation
        self.ema_alpha = ema_alpha
        self.reward = reward
        self.penalty = penalty
        self._scores: Dict[str, float] = {}

    # ---- public API -------------------------------------------------- #

    def register(self, client_id: str) -> None:
        """Register a client with the default reputation."""
        if client_id not in self._scores:
            self._scores[client_id] = self.default_reputation

    def get(self, client_id: str) -> float:
        """Return the current reputation for *client_id*."""
        return self._scores.get(client_id, self.default_reputation)

    def update(self, client_id: str, update_was_beneficial: bool) -> None:
        """
        Update the reputation of *client_id* after a federated round.

        Parameters
        ----------
        client_id : str
        update_was_beneficial : bool
            ``True`` if the client's local update improved (or at least did
            not degrade) the global model on a held-out validation set.
        """
        old = self.get(client_id)
        delta = self.reward if update_was_beneficial else -self.penalty
        new = old + self.ema_alpha * delta
        self._scores[client_id] = float(np.clip(new, 0.0, 1.0))
        logger.debug(
            "Reputation %s: %.4f → %.4f (beneficial=%s)",
            client_id, old, self._scores[client_id], update_was_beneficial,
        )

    def summary(self) -> Dict[str, float]:
        """Return a copy of the full reputation table."""
        return dict(self._scores)


# ====================================================================== #
#  3.  NORMALISATION & SCORING HELPERS                                    #
# ====================================================================== #

def _min_max_normalise(values: np.ndarray) -> np.ndarray:
    """Min-max normalise an array to [0, 1].  Returns zeros when range = 0."""
    lo, hi = values.min(), values.max()
    if hi - lo < 1e-12:
        return np.zeros_like(values, dtype=np.float64)
    return (values - lo) / (hi - lo)


def _log_scale(values: np.ndarray) -> np.ndarray:
    """Log-scale data volumes before normalising (handles zero gracefully)."""
    return np.log1p(values.astype(np.float64))


def staleness_penalty(last_selected_round: int, current_round: int) -> float:
    """
    Compute a staleness penalty in [0, 1].

    Uses an exponential decay:  ``1 - exp(-gap / 5)``  so that a client
    absent for ~15 rounds is nearly fully penalised.
    """
    gap = max(current_round - last_selected_round, 0)
    return 1.0 - math.exp(-gap / 5.0)


# ====================================================================== #
#  4.  ENHANCED CLIENT SELECTION STRATEGY                                 #
# ====================================================================== #

@dataclass
class SelectionWeights:
    """Tuneable weights for the multi-criteria scoring function."""
    w_v: float = 0.30   # local validation performance
    w_d: float = 0.20   # data volume
    w_l: float = 0.15   # latency  (applied to 1 - L_i)
    w_r: float = 0.25   # reputation
    w_s: float = 0.10   # staleness penalty (subtracted)

    def as_tuple(self) -> Tuple[float, ...]:
        return (self.w_v, self.w_d, self.w_l, self.w_r, self.w_s)


class EnhancedClientSelector:
    """
    Implements the *Enhanced Federated Client Selection Strategy*.

    Per round the selector:
      1. Collects raw metrics from every candidate client.
      2. Normalises them across the pool.
      3. Computes a composite score per client.
      4. Returns the top-K (or above-threshold) clients.

    Parameters
    ----------
    clients : list[FederatedClient]
        The full pool of federated clients.
    reputation_ledger : ReputationLedger
        Shared ledger that persists across rounds.
    weights : SelectionWeights
        Relative importance of each scoring dimension.
    target_k : int
        Number of clients to select each round (top-K mode).
    threshold : float | None
        If set, *all* clients with ``score >= threshold`` are selected
        instead of a fixed K.
    """

    def __init__(
        self,
        clients: List[FederatedClient],
        reputation_ledger: ReputationLedger,
        weights: Optional[SelectionWeights] = None,
        target_k: int = 5,
        threshold: Optional[float] = None,
    ) -> None:
        self.clients = {c.client_id: c for c in clients}
        self.ledger = reputation_ledger
        self.weights = weights or SelectionWeights()
        self.target_k = target_k
        self.threshold = threshold

        # Register every client in the ledger
        for cid in self.clients:
            self.ledger.register(cid)

    # ------------------------------------------------------------------ #
    #  Core selection logic                                               #
    # ------------------------------------------------------------------ #

    def score_clients(
        self, current_round: int
    ) -> List[Tuple[str, float]]:
        """
        Compute and return ``(client_id, score)`` for every client,
        sorted descending by score.
        """
        ids = list(self.clients.keys())
        n = len(ids)

        # --- gather raw vectors ---------------------------------------- #
        raw_v = np.array([self.clients[i].metrics.local_validation_metric for i in ids])
        raw_d = np.array([self.clients[i].metrics.data_volume for i in ids])
        raw_l = np.array([self.clients[i].metrics.inference_latency for i in ids])

        # --- normalise ------------------------------------------------- #
        V = _min_max_normalise(raw_v)                    # higher is better
        D = _min_max_normalise(_log_scale(raw_d))        # log-scaled, higher is better
        L = _min_max_normalise(raw_l)                    # 1 = slowest (penalty)
        R = np.array([self.ledger.get(i) for i in ids])  # already in [0,1]
        S = np.array([
            staleness_penalty(self.clients[i].metrics.last_selected_round, current_round)
            for i in ids
        ])

        # --- composite score ------------------------------------------- #
        w = self.weights
        scores = (
            w.w_v * V
            + w.w_d * D
            + w.w_l * (1.0 - L)
            + w.w_r * R
            - w.w_s * S
        )

        ranked = sorted(zip(ids, scores), key=lambda x: x[1], reverse=True)
        return ranked

    def select(
        self, current_round: int
    ) -> List[FederatedClient]:
        """
        Select clients for the current federated round.

        Returns
        -------
        list[FederatedClient]
            The chosen participants (top-K or above-threshold).
        """
        ranked = self.score_clients(current_round)

        if self.threshold is not None:
            selected_ids = [cid for cid, sc in ranked if sc >= self.threshold]
            # Guarantee at least 1 client even when none meets threshold
            if not selected_ids:
                selected_ids = [ranked[0][0]]
                logger.warning(
                    "Round %d — no client met threshold %.3f; "
                    "falling back to best client %s (score %.4f).",
                    current_round, self.threshold,
                    ranked[0][0], ranked[0][1],
                )
        else:
            k = min(self.target_k, len(ranked))
            ids_r = [cid for cid, _ in ranked]
            scores_r = np.array([sc for _, sc in ranked])
            # Shift scores so they are all positive, then normalise to probs
            shifted = scores_r - scores_r.min() + 1e-6
            probs = shifted / shifted.sum()
            rng = np.random.default_rng()
            chosen_idx = rng.choice(len(ids_r), size=k, replace=False, p=probs)
            selected_ids = [ids_r[i] for i in chosen_idx]

        logger.info(
            "Round %d — selected %d / %d clients: %s",
            current_round, len(selected_ids), len(ranked), selected_ids,
        )
        return [self.clients[cid] for cid in selected_ids]


# ====================================================================== #
#  5.  FEDERATED ROUND RUNNER  (skeleton compatible with TFF)             #
# ====================================================================== #

class FederatedRoundRunner:
    """
    Orchestrates the full Enhanced Federated Learning cycle.

    This class ties together:
      * The global model (loaded from the .h5 file)
      * The enhanced client selector
      * Local training on selected clients
      * Federated averaging of model updates
      * Reputation ledger updates after each round

    Parameters
    ----------
    global_model_path : str
        Path to the initial global Keras model (``*.h5``).
    clients : list[FederatedClient]
        All available federated clients.
    selector : EnhancedClientSelector
        The client selection strategy instance.
    local_epochs : int
        Number of local training epochs per client per round.
    local_batch_size : int
        Batch size for local training.
    learning_rate : float
        Learning rate for local SGD.
    """

    def __init__(
        self,
        global_model_path: str,
        clients: List[FederatedClient],
        selector: EnhancedClientSelector,
        local_epochs: int = 1,
        local_batch_size: int = 32,
        learning_rate: float = 1e-4,
    ) -> None:
        # Register EfficientNet preprocessing so Keras can deserialize the .h5
        from tensorflow.keras.applications.efficientnet import (
            preprocess_input as _effnet_preprocess,
        )
        _custom = {"preprocess_input": _effnet_preprocess}
        self.global_model: tf.keras.Model = tf.keras.models.load_model(
            global_model_path, custom_objects=_custom, compile=False,
        )
        self.global_model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        self.clients = clients
        self.selector = selector
        self.local_epochs = local_epochs
        self.local_batch_size = local_batch_size
        self.learning_rate = learning_rate

    # ------------------------------------------------------------------ #
    #  Federated Averaging helpers                                        #
    # ------------------------------------------------------------------ #

    @staticmethod
    def _fedavg(
        global_weights: List[np.ndarray],
        client_weights_list: List[List[np.ndarray]],
        sample_counts: List[int],
    ) -> List[np.ndarray]:
        """
        Weighted Federated Averaging (FedAvg).

        Each client's contribution is proportional to its local sample count.
        """
        total = sum(sample_counts)
        averaged = [
            np.zeros(w.shape, dtype=np.float64) for w in global_weights
        ]
        for cw, n in zip(client_weights_list, sample_counts):
            for idx, w in enumerate(cw):
                averaged[idx] += w * (n / total)
        return averaged

    # ------------------------------------------------------------------ #
    #  Local training on a single client                                  #
    # ------------------------------------------------------------------ #

    def _local_train(
        self,
        client: FederatedClient,
        global_weights: List[np.ndarray],
    ) -> Tuple[List[np.ndarray], int, float]:
        """
        Perform local training on *client* starting from *global_weights*.

        Returns
        -------
        updated_weights : list[np.ndarray]
        num_samples : int
        local_val_metric : float   (e.g. accuracy on local holdout)
        """
        # Build a fresh copy with global weights
        local_model: tf.keras.Model = tf.keras.models.clone_model(self.global_model)
        local_model.build(self.global_model.input_shape)
        local_model.compile(
            optimizer=tf.keras.optimizers.Adam(self.learning_rate),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        local_model.set_weights(global_weights)

        if client.local_data is None:
            logger.warning("Client %s has no local data — skipping.", client.client_id)
            return global_weights, 0, 0.0

        dataset = client.local_data.batch(self.local_batch_size)

        # Measure inference latency (single batch)
        t0 = time.perf_counter()
        for batch in dataset.take(1):
            local_model.predict(batch[0], verbose=0)
        client.metrics.inference_latency = time.perf_counter() - t0

        # Local training
        local_model.fit(dataset, epochs=self.local_epochs, verbose=0)

        # Evaluate on same data as a proxy (in production: use a held-out split)
        results = local_model.evaluate(dataset, verbose=0, return_dict=True)
        local_val = results.get("accuracy", 0.0)

        num_samples = client.metrics.data_volume

        return local_model.get_weights(), num_samples, local_val

    # ------------------------------------------------------------------ #
    #  Validate an individual client update against the global model      #
    # ------------------------------------------------------------------ #

    def _validate_update(
        self,
        updated_weights: List[np.ndarray],
        validation_data: Optional[tf.data.Dataset],
    ) -> Optional[float]:
        """
        Evaluate *updated_weights* on a global validation set.

        Returns the accuracy (or ``None`` if no validation data is provided).
        """
        if validation_data is None:
            return None
        temp_model: tf.keras.Model = tf.keras.models.clone_model(self.global_model)
        temp_model.build(self.global_model.input_shape)
        temp_model.compile(
            optimizer="adam",
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        temp_model.set_weights(updated_weights)
        results = temp_model.evaluate(
            validation_data.batch(self.local_batch_size), verbose=0, return_dict=True,
        )
        return results.get("accuracy", 0.0)

    # ------------------------------------------------------------------ #
    #  Main loop                                                          #
    # ------------------------------------------------------------------ #

    def run(
        self,
        num_rounds: int = 10,
        global_val_data: Optional[tf.data.Dataset] = None,
    ) -> Dict[str, list]:
        """
        Execute the full federated learning cycle.

        Parameters
        ----------
        num_rounds : int
            Total communication rounds ``T``.
        global_val_data : tf.data.Dataset | None
            A small global validation set used to judge update quality
            (for the reputation ledger).

        Returns
        -------
        history : dict
            ``{"round": [...], "global_acc": [...], "selected_clients": [...]}``
        """
        history: Dict[str, list] = {
            "round": [],
            "global_accuracy": [],
            "selected_clients": [],
        }

        # Compute baseline accuracy before federated training
        if global_val_data is not None:
            baseline = self.global_model.evaluate(
                global_val_data.batch(self.local_batch_size),
                verbose=0,
                return_dict=True,
            )
            logger.info("Baseline global accuracy: %.4f", baseline.get("accuracy", 0.0))

        for t in range(1, num_rounds + 1):
            logger.info("=" * 60)
            logger.info("ROUND %d / %d", t, num_rounds)
            logger.info("=" * 60)

            # ---- 1. Select clients ----------------------------------- #
            selected = self.selector.select(current_round=t)

            global_weights = self.global_model.get_weights()
            client_updates: List[List[np.ndarray]] = []
            sample_counts: List[int] = []

            # ---- 2. Local training ----------------------------------- #
            for client in selected:
                updated_w, n_samples, local_val = self._local_train(
                    client, global_weights
                )
                client.metrics.local_validation_metric = local_val
                client.metrics.last_selected_round = t

                client_updates.append(updated_w)
                sample_counts.append(max(n_samples, 1))  # avoid div-by-zero

                # ---- 3. Reputation update ----------------------------- #
                if global_val_data is not None:
                    update_acc = self._validate_update(updated_w, global_val_data)
                    current_acc = self._validate_update(global_weights, global_val_data)
                    beneficial = (
                        update_acc is not None
                        and current_acc is not None
                        and update_acc >= current_acc - 1e-4
                    )
                else:
                    # Optimistic: assume beneficial when we cannot verify
                    beneficial = True

                self.selector.ledger.update(client.client_id, beneficial)

            # ---- 4. Federated averaging ------------------------------ #
            if client_updates:
                new_global = self._fedavg(global_weights, client_updates, sample_counts)
                self.global_model.set_weights(new_global)

            # ---- 5. Global evaluation -------------------------------- #
            round_acc = None
            if global_val_data is not None:
                result = self.global_model.evaluate(
                    global_val_data.batch(self.local_batch_size),
                    verbose=0,
                    return_dict=True,
                )
                round_acc = result.get("accuracy", 0.0)
                logger.info("Round %d global accuracy: %.4f", t, round_acc)

            history["round"].append(t)
            history["global_accuracy"].append(round_acc)
            history["selected_clients"].append([c.client_id for c in selected])

        logger.info("Federated training complete — %d rounds.", num_rounds)
        return history


# ====================================================================== #
#  6.  CONVENIENCE FACTORY                                                #
# ====================================================================== #

def build_default_pipeline(
    model_path: str = "phase3_best.weights.h5",
    num_clients: int = 10,
    target_k: int = 5,
    threshold: Optional[float] = None,
    weights: Optional[SelectionWeights] = None,
) -> Tuple[FederatedRoundRunner, List[FederatedClient]]:
    """
    Quick-start helper that wires up all components.

    In practice you would replace the synthetic client list with real
    data partitions.

    Returns
    -------
    runner : FederatedRoundRunner
    clients : list[FederatedClient]
    """
    # --- Create placeholder clients (replace with real data loaders) --- #
    clients: List[FederatedClient] = []
    for i in range(num_clients):
        c = FederatedClient(
            client_id=f"client_{i:02d}",
            local_data=None,           # <-- plug real tf.data.Dataset here
            metrics=ClientMetrics(
                local_validation_metric=np.random.uniform(0.5, 0.95),
                data_volume=int(np.random.randint(200, 5000)),
                inference_latency=np.random.uniform(0.01, 0.5),
                last_selected_round=0,
            ),
        )
        clients.append(c)

    ledger = ReputationLedger()
    selector = EnhancedClientSelector(
        clients=clients,
        reputation_ledger=ledger,
        weights=weights or SelectionWeights(),
        target_k=target_k,
        threshold=threshold,
    )
    runner = FederatedRoundRunner(
        global_model_path=model_path,
        clients=clients,
        selector=selector,
    )
    return runner, clients


# ====================================================================== #
#  7.  MAIN — demo / smoke test                                           #
# ====================================================================== #

if __name__ == "__main__":
    # -------------------------------------------------------------- #
    #  Standalone demo:  runs selection scoring with synthetic clients #
    #  (no real data or model loading — safe to execute anywhere).    #
    # -------------------------------------------------------------- #

    NUM_CLIENTS = 10
    TARGET_K = 4
    NUM_ROUNDS = 5

    # Create synthetic client pool
    np.random.seed(42)
    demo_clients: List[FederatedClient] = []
    for idx in range(NUM_CLIENTS):
        demo_clients.append(
            FederatedClient(
                client_id=f"client_{idx:02d}",
                local_data=None,
                metrics=ClientMetrics(
                    local_validation_metric=np.random.uniform(0.4, 0.95),
                    data_volume=int(np.random.randint(100, 8000)),
                    inference_latency=np.random.uniform(0.01, 1.0),
                    last_selected_round=0,
                ),
            )
        )

    ledger = ReputationLedger()
    selector = EnhancedClientSelector(
        clients=demo_clients,
        reputation_ledger=ledger,
        weights=SelectionWeights(w_v=0.30, w_d=0.20, w_l=0.15, w_r=0.25, w_s=0.10),
        target_k=TARGET_K,
    )

    print("\n===  Enhanced Client Selection — Demo  ===\n")
    for rnd in range(1, NUM_ROUNDS + 1):
        ranked = selector.score_clients(current_round=rnd)
        selected = selector.select(current_round=rnd)

        print(f"\n--- Round {rnd} ---")
        print(f"{'Client':<14} {'Score':>8}")
        print("-" * 24)
        for cid, score in ranked:
            marker = " ✓" if cid in {c.client_id for c in selected} else ""
            print(f"{cid:<14} {score:>8.4f}{marker}")

        # Simulate reputation changes (random for demo)
        for c in selected:
            beneficial = np.random.random() > 0.3   # 70 % chance of good update
            ledger.update(c.client_id, bool(beneficial))
            c.metrics.last_selected_round = rnd

    print("\n--- Final Reputation Ledger ---")
    for cid, rep in sorted(ledger.summary().items()):
        print(f"  {cid}: {rep:.4f}")


Overwriting enhanced_client_selection.py


### Part 2 — Update Validation & Contribution Weighing


In [7]:
%%writefile update_validation.py
"""
Update Validation and Contribution Weighing
============================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

After each federated round, every client update is individually validated
and assigned a **contribution weight** before aggregation.  The pipeline:

  1. Norm check — flag / clip suspiciously large updates.
  2. Server-side validation gain — apply the update to a temp copy of the
     global model and measure the score delta on a held-out server set.
  3. Similarity check — cosine similarity with the recent global update
     history (catches free-riders that echo old gradients).
  4. Multi-criteria raw contribution score.
  5. Weighted aggregation (contribution-weighted FedAvg).
  6. Reputation ledger feedback from observed gains.

Imports the shared data-structures from ``enhanced_client_selection.py``.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------- shared types from Part 1 ---------------------------------- #
from enhanced_client_selection import (
    FederatedClient,
    ClientMetrics,
    ReputationLedger,
    _min_max_normalise,
    _log_scale,
)

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  CONFIGURATION                                                      #
# ====================================================================== #

@dataclass
class ContributionWeights:
    """
    Tuneable weights for the contribution scoring formula.

    ``raw = α·G_i + β·sim_i + γ·norm(D_i) + δ·R_i``
    """
    alpha: float = 0.40   # server-side validation gain
    beta:  float = 0.15   # cosine similarity to global update history
    gamma: float = 0.20   # normalised data volume
    delta: float = 0.25   # reputation

    def as_tuple(self) -> Tuple[float, ...]:
        return (self.alpha, self.beta, self.gamma, self.delta)


@dataclass
class ClippingConfig:
    """Parameters for the norm-based update clipping / rejection."""
    clip_threshold: float = 10.0     # max allowed L2 norm of a flattened update
    clip_value: Optional[float] = None  # if set, clip *to* this norm instead of rejecting


# ====================================================================== #
#  2.  HELPER UTILITIES                                                   #
# ====================================================================== #

def flatten_weights(weights: List[np.ndarray]) -> np.ndarray:
    """Concatenate a list of weight arrays into a single 1-D vector."""
    return np.concatenate([w.ravel() for w in weights])


def unflatten_weights(
    flat: np.ndarray,
    shapes: List[Tuple[int, ...]],
) -> List[np.ndarray]:
    """Inverse of ``flatten_weights``: split a 1-D vector back into arrays."""
    arrays: List[np.ndarray] = []
    offset = 0
    for shape in shapes:
        size = int(np.prod(shape))
        arrays.append(flat[offset : offset + size].reshape(shape))
        offset += size
    return arrays


def compute_update_delta(
    global_weights: List[np.ndarray],
    updated_weights: List[np.ndarray],
) -> List[np.ndarray]:
    """Return the element-wise difference  ``updated − global``."""
    return [u - g for u, g in zip(updated_weights, global_weights)]


def apply_update(
    base_weights: List[np.ndarray],
    delta: List[np.ndarray],
    scale: float = 1.0,
) -> List[np.ndarray]:
    """Return ``base + scale * delta`` (per-layer)."""
    return [b + scale * d for b, d in zip(base_weights, delta)]


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    """
    Cosine similarity between two 1-D vectors.

    Returns 0.0 when either vector has near-zero norm (avoids NaN).
    """
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a < 1e-12 or norm_b < 1e-12:
        return 0.0
    return float(np.dot(a, b) / (norm_a * norm_b))


def _normalise_scalar_to_01(
    values: np.ndarray,
) -> np.ndarray:
    """Scale an array into [0, 1] via min-max.  Alias for readability."""
    return _min_max_normalise(values)


# ====================================================================== #
#  3.  GLOBAL UPDATE HISTORY                                              #
# ====================================================================== #

class GlobalUpdateHistory:
    """
    Maintains a rolling window of the last *N* aggregated global updates
    (as flattened vectors) so the validator can compute cosine similarity
    between a client update and "what the model has been learning lately".

    Parameters
    ----------
    max_history : int
        Maximum number of past global deltas to keep.
    """

    def __init__(self, max_history: int = 10) -> None:
        self.max_history = max_history
        self._history: List[np.ndarray] = []   # each entry is a 1-D vector

    def push(self, global_delta_flat: np.ndarray) -> None:
        """Append the latest aggregated global delta."""
        self._history.append(global_delta_flat.copy())
        if len(self._history) > self.max_history:
            self._history.pop(0)

    @property
    def mean_direction(self) -> Optional[np.ndarray]:
        """
        Return the mean direction of stored history.

        This single vector captures the *average trend* the global model
        has been moving in.  Returns ``None`` before the first round.
        """
        if not self._history:
            return None
        stacked = np.stack(self._history, axis=0)
        return stacked.mean(axis=0)

    @property
    def size(self) -> int:
        return len(self._history)


# ====================================================================== #
#  4.  UPDATE VALIDATOR & CONTRIBUTION SCORER                             #
# ====================================================================== #

@dataclass
class ClientUpdateRecord:
    """Result of validating a single client's update."""
    client_id: str
    delta: List[np.ndarray]          # raw weight delta  (updated − global)
    norm: float = 0.0                # L2 norm of flattened delta
    is_suspicious: bool = False      # flagged by norm check
    validation_gain: float = 0.0     # G_i = new_score − baseline_score
    similarity: float = 0.0         # cosine similarity with history
    raw_contribution: float = 0.0    # before normalisation
    contribution_weight: float = 0.0 # final c_i in [0, 1]
    rejected: bool = False           # update completely rejected


class UpdateValidator:
    """
    Validates client updates and computes contribution weights.

    This is the **core class** of the second part of the federated cycle.

    Parameters
    ----------
    global_model : tf.keras.Model
        The current global model (used to create temp copies for eval).
    reputation_ledger : ReputationLedger
        Shared reputation tracker (read for R_i, written after scoring).
    weights : ContributionWeights
        α, β, γ, δ for the composite score.
    clipping : ClippingConfig
        Norm-clipping parameters.
    harmful_threshold : float
        ε — if ``G_i < −ε`` the update is rejected outright.
    batch_size : int
        Batch size when evaluating on the server validation set.
    eval_metric : str
        Name of the Keras metric to use as *score* (e.g. ``"accuracy"``
        or ``"f1_score"``).
    """

    def __init__(
        self,
        global_model: tf.keras.Model,
        reputation_ledger: ReputationLedger,
        weights: Optional[ContributionWeights] = None,
        clipping: Optional[ClippingConfig] = None,
        harmful_threshold: float = 0.02,
        batch_size: int = 32,
        eval_metric: str = "accuracy",
    ) -> None:
        self.global_model = global_model
        self.ledger = reputation_ledger
        self.weights = weights or ContributionWeights()
        self.clipping = clipping or ClippingConfig()
        self.harmful_threshold = harmful_threshold
        self.batch_size = batch_size
        self.eval_metric = eval_metric
        self.update_history = GlobalUpdateHistory()
        self._eval_model: Optional[tf.keras.Model] = None   # cached eval model
        self._val_data_batched = None                         # cached batched dataset

    # ------------------------------------------------------------------ #
    #  Evaluation helper                                                  #
    # ------------------------------------------------------------------ #

    def _evaluate(
        self,
        model_weights: List[np.ndarray],
        val_data: tf.data.Dataset,
    ) -> float:
        """
        Evaluate *model_weights* on *val_data* using a cached temporary
        model (avoids clone+build+compile overhead on every call).

        Returns the scalar value of ``self.eval_metric``.
        """
        # Lazily create & cache the temporary evaluation model
        if self._eval_model is None:
            self._eval_model = tf.keras.models.clone_model(self.global_model)
            self._eval_model.build(self.global_model.input_shape)
            self._eval_model.compile(
                optimizer="adam",
                loss="binary_crossentropy",
                metrics=["accuracy"],
            )
        # Cache the batched + prefetched validation dataset
        if self._val_data_batched is None:
            self._val_data_batched = (
                val_data.batch(self.batch_size).prefetch(tf.data.AUTOTUNE)
            )
        self._eval_model.set_weights(model_weights)
        results = self._eval_model.evaluate(
            self._val_data_batched, verbose=0, return_dict=True,
        )
        return float(results.get(self.eval_metric, 0.0))

    # ------------------------------------------------------------------ #
    #  Norm check                                                         #
    # ------------------------------------------------------------------ #

    def _norm_check(
        self,
        delta_flat: np.ndarray,
    ) -> Tuple[bool, np.ndarray]:
        """
        Check if the update norm exceeds the clip threshold.

        Returns
        -------
        is_suspicious : bool
        clipped_flat : np.ndarray
            The (possibly clipped) flat delta.
        """
        norm = float(np.linalg.norm(delta_flat))
        if norm <= self.clipping.clip_threshold:
            return False, delta_flat

        logger.warning(
            "Update norm %.4f exceeds threshold %.4f",
            norm, self.clipping.clip_threshold,
        )
        if self.clipping.clip_value is not None:
            # Clip to allowed magnitude instead of outright rejection
            scale = self.clipping.clip_value / (norm + 1e-12)
            return True, delta_flat * scale
        # No clip_value ⇒ hard rejection (caller sets weight = 0)
        return True, delta_flat

    # ------------------------------------------------------------------ #
    #  Main validation pipeline                                           #
    # ------------------------------------------------------------------ #

    def validate_updates(
        self,
        client_updates: Dict[str, List[np.ndarray]],
        data_volumes: Dict[str, int],
        server_val_data: tf.data.Dataset,
    ) -> List[ClientUpdateRecord]:
        """
        Validate every client update and assign contribution weights.

        Parameters
        ----------
        client_updates : dict[str, list[np.ndarray]]
            Mapping ``client_id → updated model weights`` (full weights,
            **not** deltas — deltas are computed internally).
        data_volumes : dict[str, int]
            Mapping ``client_id → number of local training samples``.
        server_val_data : tf.data.Dataset
            The server-side held-out validation set.

        Returns
        -------
        records : list[ClientUpdateRecord]
            One record per client, with ``contribution_weight`` set.
        """
        global_weights = self.global_model.get_weights()
        shapes = [w.shape for w in global_weights]
        global_flat = flatten_weights(global_weights)

        # ---- 0. Baseline score on server val set ---------------------- #
        baseline_score = self._evaluate(global_weights, server_val_data)
        logger.info("Baseline server score (%s): %.4f", self.eval_metric, baseline_score)

        records: List[ClientUpdateRecord] = []
        gains: List[float] = []       # for min-max normalisation later
        sims: List[float] = []
        raw_data_vols: List[int] = []
        reps: List[float] = []

        # ---- Per-client loop ----------------------------------------- #
        for cid, updated_weights in client_updates.items():
            delta = compute_update_delta(global_weights, updated_weights)
            delta_flat = flatten_weights(delta)
            norm = float(np.linalg.norm(delta_flat))

            rec = ClientUpdateRecord(client_id=cid, delta=delta, norm=norm)

            # 1.  Norm check
            is_suspicious, clipped_flat = self._norm_check(delta_flat)
            rec.is_suspicious = is_suspicious

            if is_suspicious and self.clipping.clip_value is None:
                # Hard rejection — set weight = 0, skip evaluation
                rec.rejected = True
                rec.contribution_weight = 0.0
                records.append(rec)
                gains.append(0.0)
                sims.append(0.0)
                raw_data_vols.append(data_volumes.get(cid, 0))
                reps.append(self.ledger.get(cid))
                logger.debug(
                    "Client %s REJECTED (norm %.4f > %.4f, no clip_value).",
                    cid, norm, self.clipping.clip_threshold,
                )
                continue

            # Possibly overwrite delta with clipped version
            if is_suspicious:
                delta = unflatten_weights(clipped_flat, shapes)
                rec.delta = delta

            # 2.  Server-side validation gain
            temp_weights = apply_update(global_weights, delta, scale=1.0)
            new_score = self._evaluate(temp_weights, server_val_data)
            G_i = new_score - baseline_score
            rec.validation_gain = G_i

            # 3.  Similarity check
            hist_dir = self.update_history.mean_direction
            if hist_dir is not None:
                sim_i = cosine_similarity(flatten_weights(delta), hist_dir)
            else:
                sim_i = 0.5   # neutral when no history yet
            rec.similarity = sim_i

            gains.append(G_i)
            sims.append(sim_i)
            raw_data_vols.append(data_volumes.get(cid, 0))
            reps.append(self.ledger.get(cid))
            records.append(rec)

        # ---- 4. Combine into normalised contribution weights ---------- #
        n = len(records)
        if n == 0:
            return records

        arr_G = np.array(gains, dtype=np.float64)
        arr_sim = np.array(sims, dtype=np.float64)
        arr_D = _normalise_scalar_to_01(_log_scale(np.array(raw_data_vols, dtype=np.float64)))
        arr_R = np.array(reps, dtype=np.float64)          # already [0, 1]

        w = self.weights
        raw_scores = (
            w.alpha * arr_G
            + w.beta  * arr_sim
            + w.gamma * arr_D
            + w.delta * arr_R
        )

        # Normalise raw scores into [0, 1]
        c = _normalise_scalar_to_01(raw_scores)

        # Reject strongly harmful updates  (G_i < −ε)
        for idx, rec in enumerate(records):
            if rec.rejected:
                c[idx] = 0.0
                continue
            rec.raw_contribution = float(raw_scores[idx])
            rec.contribution_weight = float(c[idx])

            if rec.validation_gain < -self.harmful_threshold:
                rec.contribution_weight = 0.0
                rec.rejected = True
                logger.debug(
                    "Client %s rejected — G_i=%.4f < −ε (%.4f).",
                    rec.client_id, rec.validation_gain, self.harmful_threshold,
                )

        return records

    # ------------------------------------------------------------------ #
    #  5. Weighted aggregation                                            #
    # ------------------------------------------------------------------ #

    def aggregate_weighted(
        self,
        records: List[ClientUpdateRecord],
        global_weights: Optional[List[np.ndarray]] = None,
    ) -> List[np.ndarray]:
        """
        Contribution-weighted aggregation of client deltas.

        ``new_global = global + Σ_i  (c_i / Σ c_j) · delta_i``

        Parameters
        ----------
        records : list[ClientUpdateRecord]
            Output of ``validate_updates``.
        global_weights : list[np.ndarray] | None
            If ``None``, reads from ``self.global_model``.

        Returns
        -------
        new_global_weights : list[np.ndarray]
        """
        if global_weights is None:
            global_weights = self.global_model.get_weights()

        # Filter to accepted updates with positive weight
        active = [(r.delta, r.contribution_weight) for r in records
                  if not r.rejected and r.contribution_weight > 0]

        if not active:
            logger.warning("No valid updates this round — global model unchanged.")
            return global_weights

        total_c = sum(c for _, c in active)
        aggregated_delta = [np.zeros(w.shape, dtype=np.float64) for w in global_weights]

        for delta, c_i in active:
            weight = c_i / total_c
            for idx, d in enumerate(delta):
                aggregated_delta[idx] += weight * d

        new_weights = apply_update(global_weights, aggregated_delta)

        # Push this aggregated delta into the history for future similarity
        self.update_history.push(flatten_weights(aggregated_delta))

        return new_weights

    # ------------------------------------------------------------------ #
    #  6. Reputation feedback                                             #
    # ------------------------------------------------------------------ #

    def update_reputations(
        self,
        records: List[ClientUpdateRecord],
    ) -> None:
        """
        Feed observed validation gains and contribution weights back into
        the reputation ledger.

        - Clients with ``G_i > 0`` and ``c_i > 0`` are rewarded.
        - Clients with ``G_i < 0`` or no contribution are penalised.
        - Rejected clients receive a stronger penalty.
        """
        for rec in records:
            if rec.rejected:
                # Stronger penalty for rejected / harmful updates
                self.ledger.update(rec.client_id, update_was_beneficial=False)
                self.ledger.update(rec.client_id, update_was_beneficial=False)
                logger.debug("Reputation double-penalty for %s (rejected).", rec.client_id)
            elif rec.validation_gain > 0 and rec.contribution_weight > 0:
                self.ledger.update(rec.client_id, update_was_beneficial=True)
            else:
                self.ledger.update(rec.client_id, update_was_beneficial=False)

Overwriting update_validation.py


### Part 3 — Server-Side Knowledge Distillation


In [8]:
%%writefile knowledge_distillation.py
"""
Server-Side Knowledge Distillation
===================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

After federated aggregation (Parts 1 & 2), the server refines the global
model by distilling knowledge from the contribution-weighted ensemble of
client models into the global student model, using unlabelled proxy data
from FF++ c23.

Pipeline
--------
1.  **Build teacher logits** — weighted average of per-client logits on
    every proxy input, where the weights are the contribution scores
    ``{c_i}`` from Part 2.
2.  **Distillation loop** — minimise ``λ · T² · KL(p_teach ‖ p_stud)``
    (soft targets) plus an optional ``(1−λ) · CE`` supervised term when
    labelled data is available.
3.  Return the distilled global model.

Imports shared types from ``enhanced_client_selection.py`` and helpers
from ``update_validation.py``.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------- shared types from Parts 1 & 2 ----------------------------- #
from enhanced_client_selection import FederatedClient, ReputationLedger

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  CONFIGURATION                                                      #
# ====================================================================== #

@dataclass
class DistillationConfig:
    """
    Hyper-parameters for server-side knowledge distillation.

    Parameters
    ----------
    temperature : float
        Softmax temperature ``T`` — higher values produce softer
        probability distributions, transferring more "dark knowledge".
    lam : float
        Interpolation weight ``λ`` between the distillation loss and the
        optional supervised cross-entropy loss:
        ``L_total = λ · L_KD  +  (1 − λ) · L_sup``
    epochs : int
        Number of distillation training epochs.
    batch_size : int
        Batch size for iterating over proxy / supervised data.
    learning_rate : float
        Learning rate for the distillation optimiser.
    """
    temperature: float = 3.0
    lam: float = 0.7
    epochs: int = 5
    batch_size: int = 32
    learning_rate: float = 1e-4


# ====================================================================== #
#  2.  TEACHER LOGIT BUILDER                                              #
# ====================================================================== #

class TeacherEnsemble:
    """
    Builds a *virtual* teacher by computing the contribution-weighted
    average of per-client logits for every proxy input.

    The teacher is never materialised as a single model; instead its
    output logits are pre-computed and cached so the distillation loop
    can iterate over them efficiently.

    Parameters
    ----------
    global_model : tf.keras.Model
        Used as the architecture template for loading client weights.
    client_weights : dict[str, list[np.ndarray]]
        ``{client_id: model_weights}`` for every participating client.
    contribution_weights : dict[str, float]
        ``{client_id: c_i}`` from Part 2 — only clients with ``c_i > 0``
        are included.
    """

    def __init__(
        self,
        global_model: tf.keras.Model,
        client_weights: Dict[str, List[np.ndarray]],
        contribution_weights: Dict[str, float],
    ) -> None:
        self.global_model = global_model
        # Filter to clients that actually contribute
        self.client_weights = {
            cid: w for cid, w in client_weights.items()
            if contribution_weights.get(cid, 0.0) > 0
        }
        self.contribution_weights = {
            cid: contribution_weights[cid]
            for cid in self.client_weights
        }
        total_c = sum(self.contribution_weights.values())
        # Normalise so weights sum to 1
        self._norm_weights = {
            cid: c / total_c for cid, c in self.contribution_weights.items()
        }
        # Pre-build and cache logit models for all clients
        self._logit_models: Dict[str, tf.keras.Model] = {}
        logit_arch = self._rebuild_with_linear_output(self.global_model)
        for cid, w in self.client_weights.items():
            model_copy = tf.keras.models.clone_model(logit_arch)
            model_copy.build(logit_arch.input_shape)
            model_copy.set_weights(w)
            self._logit_models[cid] = model_copy
        logger.debug(
            "TeacherEnsemble: %d client(s) cached",
            len(self._logit_models),
        )

    # ------------------------------------------------------------------ #
    #  Logit-model builder (creates a model that outputs pre-softmax)     #
    # ------------------------------------------------------------------ #

    def _build_logit_model(
        self,
        weights: List[np.ndarray],
    ) -> tf.keras.Model:
        """
        Clone the global model, load *weights*, and strip the final
        activation so the output is raw **logits**.

        Strategy: rebuild the architecture layer-by-layer. For the last
        Dense layer, replace its activation with ``linear``.  This avoids
        fragile graph surgery and works across Sequential / Functional
        models in all TF 2.x versions.
        """
        # --- Rebuild architecture with linear final activation --------- #
        logit_model = self._rebuild_with_linear_output(self.global_model)
        logit_model.set_weights(weights)
        return logit_model

    @staticmethod
    def _rebuild_with_linear_output(
        ref_model: tf.keras.Model,
    ) -> tf.keras.Model:
        """
        Rebuild *ref_model* identically, except the last ``Dense`` layer
        uses ``activation='linear'`` so the output is raw logits.

        Uses ``clone_model`` to preserve the full graph topology (skip
        connections in Functional models like EfficientNet).

        Falls back to the original architecture when no Dense layer is
        found (e.g. the model already produces logits).
        """
        cloned = tf.keras.models.clone_model(ref_model)
        cloned.build(ref_model.input_shape)
        cloned.set_weights(ref_model.get_weights())

        # Override the last Dense layer's activation to linear
        for layer in reversed(cloned.layers):
            if isinstance(layer, tf.keras.layers.Dense):
                layer.activation = tf.keras.activations.linear
                break

        return cloned

    # ------------------------------------------------------------------ #
    #  Compute teacher logits for a batch                                 #
    # ------------------------------------------------------------------ #

    def compute_teacher_logits_batch(
        self,
        x_batch: tf.Tensor,
    ) -> tf.Tensor:
        """
        Return the contribution-weighted average teacher logits for
        *x_batch*.  Uses pre-cached logit models.

        ``z_teach(x) = Σ_i  w_i · logits(M_i, x)``
        """
        weighted_logits = None
        for cid in self.client_weights:
            logit_model = self._logit_models[cid]
            logits = logit_model(x_batch, training=False)
            if logits.shape[-1] == 1:
                probs = tf.sigmoid(logits)
            else:
                probs = tf.nn.softmax(logits)
            scaled = tf.cast(probs, tf.float32) * self._norm_weights[cid]
            if weighted_logits is None:
                weighted_logits = scaled
            else:
                weighted_logits = weighted_logits + scaled
        return weighted_logits

    # ------------------------------------------------------------------ #
    #  Pre-compute teacher logits for entire proxy dataset                #
    # ------------------------------------------------------------------ #

    def precompute_teacher_logits(
        self,
        proxy_data: tf.data.Dataset,
        batch_size: int = 32,
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Pre-compute teacher logits for every sample in *proxy_data*.

        Parameters
        ----------
        proxy_data : tf.data.Dataset
            Must yield ``(images,)`` or ``(images, _)`` — labels are
            ignored.
        batch_size : int

        Returns
        -------
        all_inputs : np.ndarray   — shape ``(N, ...)``
        all_logits : np.ndarray   — shape ``(N, num_classes)`` or ``(N, 1)``
        """
        all_inputs: List[np.ndarray] = []
        all_logits: List[np.ndarray] = []

        batched = proxy_data.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        for batch in batched:
            if isinstance(batch, (list, tuple)):
                x_batch = batch[0]
            else:
                x_batch = batch

            teacher_logits = self.compute_teacher_logits_batch(x_batch)
            all_inputs.append(x_batch.numpy())
            all_logits.append(teacher_logits.numpy())

        return np.concatenate(all_inputs), np.concatenate(all_logits)


# ====================================================================== #
#  3.  DISTILLATION LOSSES                                                #
# ====================================================================== #

def distillation_loss(
    teacher_logits: tf.Tensor,
    student_logits: tf.Tensor,
    temperature: float,
) -> tf.Tensor:
    """
    Compute the knowledge-distillation loss:

        ``L_KD = T² · KL( softmax(z_teach / T)  ‖  softmax(z_stud / T) )``

    Works for both multi-class (last dim > 1) and binary (last dim = 1).
    """
    # Determine if binary or multi-class
    num_outputs = teacher_logits.shape[-1]

    if num_outputs == 1:
        # Binary case: use sigmoid + binary KL
        p_teach = teacher_logits
        p_stud  = tf.sigmoid(student_logits / temperature)
        # Binary KL divergence  KL(p || q) = p·log(p/q) + (1-p)·log((1-p)/(1-q))
        eps = 1e-7
        p_teach = tf.clip_by_value(p_teach, eps, 1.0 - eps)
        p_stud  = tf.clip_by_value(p_stud,  eps, 1.0 - eps)
        kl = (
            p_teach * tf.math.log(p_teach / p_stud)
            + (1.0 - p_teach) * tf.math.log((1.0 - p_teach) / (1.0 - p_stud))
        )
        return temperature ** 2 * tf.reduce_mean(kl)
    else:
        # Multi-class case: standard KL on softmax outputs
        p_teach = tf.nn.softmax(teacher_logits / temperature)
        log_p_stud = tf.nn.log_softmax(student_logits / temperature)
        # KL(p_teach || p_stud) = Σ p_teach · (log p_teach − log p_stud)
        kl = tf.reduce_sum(
            p_teach * (tf.math.log(p_teach + 1e-12) - log_p_stud),
            axis=-1,
        )
        return temperature ** 2 * tf.reduce_mean(kl)


def supervised_loss(
    student_logits: tf.Tensor,
    labels: tf.Tensor,
) -> tf.Tensor:
    """
    Standard supervised cross-entropy loss.

    Handles binary (1-unit output, BCE) and multi-class (softmax CE)
    automatically.
    """
    num_outputs = student_logits.shape[-1]
    if num_outputs == 1:
        # Ensure labels have the same rank as logits: (B,) → (B, 1)
        labels = tf.cast(labels, tf.float32)
        if len(labels.shape) < len(student_logits.shape):
            labels = tf.expand_dims(labels, -1)
        return tf.reduce_mean(
            tf.keras.losses.binary_crossentropy(
                labels, student_logits, from_logits=True,
            )
        )
    else:
        return tf.reduce_mean(
            tf.keras.losses.sparse_categorical_crossentropy(
                labels, student_logits, from_logits=True,
            )
        )


# ====================================================================== #
#  4.  KNOWLEDGE DISTILLATION ENGINE                                      #
# ====================================================================== #

class KnowledgeDistiller:
    """
    Performs server-side knowledge distillation from a weighted ensemble
    of client models (teacher) into the aggregated global model (student).

    Parameters
    ----------
    global_model : tf.keras.Model
        The global/student model — **modified in-place**.
    teacher : TeacherEnsemble
        The virtual teacher (built from client logits).
    config : DistillationConfig
        Temperature, λ, epochs, batch size, learning rate.
    """

    def __init__(
        self,
        global_model: tf.keras.Model,
        teacher: TeacherEnsemble,
        config: Optional[DistillationConfig] = None,
    ) -> None:
        self.global_model = global_model
        self.teacher = teacher
        self.config = config or DistillationConfig()
        self.optimizer = tf.keras.optimizers.Adam(self.config.learning_rate)

    # ------------------------------------------------------------------ #
    #  Build a "logit model" view of the global student                   #
    # ------------------------------------------------------------------ #

    def _build_student_logit_model(self) -> tf.keras.Model:
        """
        Return a version of the global model that outputs raw logits.

        Uses the same stripping logic as ``TeacherEnsemble``.
        """
        return self.teacher._build_logit_model(self.global_model.get_weights())

    # ------------------------------------------------------------------ #
    #  Single training step                                               #
    # ------------------------------------------------------------------ #

    @tf.function
    def _train_step_kd_only(
        self,
        x_batch: tf.Tensor,
        teacher_logits: tf.Tensor,
        student_model: tf.keras.Model,
        temperature: float,
    ) -> tf.Tensor:
        """Pure distillation step (no supervised term)."""
        with tf.GradientTape() as tape:
            student_logits = student_model(x_batch, training=True)
            loss = distillation_loss(teacher_logits, student_logits, temperature)
        grads = tape.gradient(loss, student_model.trainable_variables)
        self.optimizer.apply_gradients(
            zip(grads, student_model.trainable_variables)
        )
        return loss

    @tf.function
    def _train_step_combined(
        self,
        x_proxy: tf.Tensor,
        teacher_logits: tf.Tensor,
        x_sup: tf.Tensor,
        y_sup: tf.Tensor,
        student_model: tf.keras.Model,
        temperature: float,
        lam: float,
    ) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
        """Combined distillation + supervised step."""
        with tf.GradientTape() as tape:
            # KD part
            stud_logits_proxy = student_model(x_proxy, training=True)
            l_kd = distillation_loss(teacher_logits, stud_logits_proxy, temperature)

            # Supervised part
            stud_logits_sup = student_model(x_sup, training=True)
            l_sup = supervised_loss(stud_logits_sup, y_sup)

            l_total = lam * l_kd + (1.0 - lam) * l_sup

        grads = tape.gradient(l_total, student_model.trainable_variables)
        self.optimizer.apply_gradients(
            zip(grads, student_model.trainable_variables)
        )
        return l_total, l_kd, l_sup

    # ------------------------------------------------------------------ #
    #  Full distillation loop                                             #
    # ------------------------------------------------------------------ #

    def distill(
        self,
        proxy_data: tf.data.Dataset,
        supervised_data: Optional[tf.data.Dataset] = None,
    ) -> Dict[str, List[float]]:
        """
        Run the full distillation loop.

        Parameters
        ----------
        proxy_data : tf.data.Dataset
            Unlabelled proxy inputs from FF++ c23.
            Yields ``(images,)`` or ``(images, _)`` — labels ignored.
        supervised_data : tf.data.Dataset | None
            Optional labelled data yielding ``(images, labels)``.
            When provided, the combined loss
            ``λ · L_KD + (1−λ) · L_sup`` is used.

        Returns
        -------
        history : dict
            ``{"epoch": [...], "loss_total": [...], "loss_kd": [...],
               "loss_sup": [...]}``
        """
        cfg = self.config
        T = cfg.temperature
        lam = cfg.lam

        history: Dict[str, List[float]] = {
            "epoch": [],
            "loss_total": [],
            "loss_kd": [],
            "loss_sup": [],
        }

        # --- Pre-compute teacher logits ------------------------------- #
        proxy_inputs, teacher_logits_all = self.teacher.precompute_teacher_logits(
            proxy_data, batch_size=cfg.batch_size,
        )

        # Wrap into a tf.data.Dataset for batching
        proxy_ds = (
            tf.data.Dataset.from_tensor_slices((proxy_inputs, teacher_logits_all))
            .shuffle(buffer_size=len(proxy_inputs))
            .batch(cfg.batch_size)
            .prefetch(tf.data.AUTOTUNE)
        )

        # Prepare supervised data iterator (if available)
        if supervised_data is not None:
            sup_ds = (
                supervised_data
                .shuffle(buffer_size=1000)
                .batch(cfg.batch_size)
                .repeat()                       # cycle forever — we zip with proxy
                .prefetch(tf.data.AUTOTUNE)
            )
            sup_iter = iter(sup_ds)
        else:
            sup_iter = None

        # --- Build student logit model -------------------------------- #
        # We train a clone that shares the same architecture & initial
        # weights, then copy weights back to self.global_model at the end.
        student = self._build_student_logit_model()

        # --- Distillation epochs -------------------------------------- #
        for epoch in range(1, cfg.epochs + 1):
            epoch_loss_total = []
            epoch_loss_kd = []
            epoch_loss_sup = []

            for batch in proxy_ds:
                x_proxy_batch, teach_logits_batch = batch

                if sup_iter is not None:
                    # Combined loss
                    try:
                        x_sup_batch, y_sup_batch = next(sup_iter)
                    except StopIteration:
                        sup_iter = iter(sup_ds)
                        x_sup_batch, y_sup_batch = next(sup_iter)

                    l_total, l_kd, l_sup = self._train_step_combined(
                        x_proxy_batch, teach_logits_batch,
                        x_sup_batch, y_sup_batch,
                        student, T, lam,
                    )
                    epoch_loss_total.append(float(l_total))
                    epoch_loss_kd.append(float(l_kd))
                    epoch_loss_sup.append(float(l_sup))
                else:
                    # KD only
                    l_kd = self._train_step_kd_only(
                        x_proxy_batch, teach_logits_batch,
                        student, T,
                    )
                    epoch_loss_total.append(float(l_kd))
                    epoch_loss_kd.append(float(l_kd))
                    epoch_loss_sup.append(0.0)

            mean_total = float(np.mean(epoch_loss_total))
            mean_kd    = float(np.mean(epoch_loss_kd))
            mean_sup   = float(np.mean(epoch_loss_sup))

            history["epoch"].append(epoch)
            history["loss_total"].append(mean_total)
            history["loss_kd"].append(mean_kd)
            history["loss_sup"].append(mean_sup)

            logger.debug(
                "Distillation epoch %d/%d — L_total=%.5f  L_KD=%.5f  L_sup=%.5f",
                epoch, cfg.epochs, mean_total, mean_kd, mean_sup,
            )

        # --- Copy distilled weights back to global model -------------- #
        self.global_model.set_weights(student.get_weights())

        return history


# ====================================================================== #
#  5.  CONVENIENCE: one-call distillation after a federated round         #
# ====================================================================== #

def run_distillation_round(
    global_model: tf.keras.Model,
    client_weights: Dict[str, List[np.ndarray]],
    contribution_weights: Dict[str, float],
    proxy_data: tf.data.Dataset,
    supervised_data: Optional[tf.data.Dataset] = None,
    config: Optional[DistillationConfig] = None,
) -> Dict[str, List[float]]:
    """
    One-liner helper that creates the teacher ensemble, distiller,
    and runs the distillation loop.

    Parameters
    ----------
    global_model : tf.keras.Model
        Aggregated global model (modified in-place).
    client_weights : dict[str, list[np.ndarray]]
        Per-client model weights.
    contribution_weights : dict[str, float]
        Per-client contribution scores ``c_i`` from Part 2.
    proxy_data : tf.data.Dataset
        Unlabelled proxy inputs (FF++ c23).
    supervised_data : tf.data.Dataset | None
        Optional labelled data for the combined loss.
    config : DistillationConfig | None
        Hyper-parameters (defaults are sensible).

    Returns
    -------
    history : dict
    """
    config = config or DistillationConfig()

    teacher = TeacherEnsemble(
        global_model=global_model,
        client_weights=client_weights,
        contribution_weights=contribution_weights,
    )
    distiller = KnowledgeDistiller(
        global_model=global_model,
        teacher=teacher,
        config=config,
    )
    return distiller.distill(proxy_data, supervised_data)


# ====================================================================== #
#  DEMO / SMOKE-TEST  (synthetic data — no real model needed)             #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  Server-Side Knowledge Distillation — Demo  ===\n")

    np.random.seed(42)
    tf.random.set_seed(42)

    # ---- 1. Build a tiny model to act as global / student ------------ #
    INPUT_DIM = 16
    NUM_CLASSES = 1          # binary (DeepFake vs Real)
    NUM_PROXY = 200
    NUM_SUP = 60
    NUM_CLIENTS = 4

    global_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(INPUT_DIM,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(NUM_CLASSES, activation="sigmoid"),
    ])
    global_model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    print(f"Global model params: {global_model.count_params()}")

    # ---- 2. Synthetic client models (slight perturbations) ----------- #
    base_weights = global_model.get_weights()
    client_weights: Dict[str, List[np.ndarray]] = {}
    contribution_weights: Dict[str, float] = {}

    for i in range(NUM_CLIENTS):
        cid = f"client_{i:02d}"
        noise = [np.random.randn(*w.shape).astype(np.float32) * 0.05
                 for w in base_weights]
        client_weights[cid] = [w + n for w, n in zip(base_weights, noise)]
        contribution_weights[cid] = np.random.uniform(0.3, 1.0)

    print(f"Contribution weights: {contribution_weights}")

    # ---- 3. Synthetic proxy & supervised data ------------------------ #
    proxy_x = np.random.randn(NUM_PROXY, INPUT_DIM).astype(np.float32)
    proxy_data = tf.data.Dataset.from_tensor_slices(proxy_x)

    sup_x = np.random.randn(NUM_SUP, INPUT_DIM).astype(np.float32)
    sup_y = np.random.randint(0, 2, size=(NUM_SUP,)).astype(np.float32)
    supervised_data = tf.data.Dataset.from_tensor_slices((sup_x, sup_y))

    # ---- 4. Run distillation (KD only) ------------------------------- #
    print("\n--- KD-only distillation ---")
    history_kd = run_distillation_round(
        global_model=global_model,
        client_weights=client_weights,
        contribution_weights=contribution_weights,
        proxy_data=proxy_data,
        supervised_data=None,              # no supervised term
        config=DistillationConfig(
            temperature=2.0,
            lam=0.5,
            epochs=3,
            batch_size=32,
            learning_rate=1e-3,
        ),
    )
    for ep, lt, lk in zip(history_kd["epoch"], history_kd["loss_total"],
                          history_kd["loss_kd"]):
        print(f"  Epoch {ep}: L_total={lt:.5f}  L_KD={lk:.5f}")

    # ---- 5. Run distillation (combined KD + supervised) -------------- #
    print("\n--- Combined KD + supervised distillation ---")
    # Reset model to base weights for a fair comparison
    global_model.set_weights(base_weights)

    history_comb = run_distillation_round(
        global_model=global_model,
        client_weights=client_weights,
        contribution_weights=contribution_weights,
        proxy_data=proxy_data,
        supervised_data=supervised_data,
        config=DistillationConfig(
            temperature=2.0,
            lam=0.5,
            epochs=3,
            batch_size=32,
            learning_rate=1e-3,
        ),
    )
    for ep, lt, lk, ls in zip(
        history_comb["epoch"], history_comb["loss_total"],
        history_comb["loss_kd"], history_comb["loss_sup"],
    ):
        print(f"  Epoch {ep}: L_total={lt:.5f}  L_KD={lk:.5f}  L_sup={ls:.5f}")

    print("\nDone.")

Overwriting knowledge_distillation.py


### Part 4 — Client Reputation & Persistent Ledger


In [9]:
%%writefile client_reputation_ledger.py
"""
Client Reputation and Client Ledger
====================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Extends the basic ``ReputationLedger`` from Part 1 with:

* **Validation-gain-based reputation update** — reputation increases
  proportionally to the measured validation gain ``G_i`` when it exceeds
  a threshold ``θ``, and holds steady (or decays) otherwise.
* **Persistent ledger** with full round-by-round history, JSON
  serialisation, and audit trail.
* **Decay & floor mechanics** — inactive or consistently harmful clients
  slowly lose reputation; a configurable floor prevents permanent
  exclusion.
* **Integration helpers** that plug directly into Parts 1–3.

Pseudo-code from the thesis proposal is faithfully reproduced.
"""

from __future__ import annotations

import json
import logging
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np

# ---------- shared types from Part 1 ---------------------------------- #
from enhanced_client_selection import ReputationLedger

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  CONFIGURATION                                                      #
# ====================================================================== #

@dataclass
class ReputationConfig:
    """
    Hyper-parameters for the gain-based reputation update rule.

    Parameters
    ----------
    theta : float
        Validation-gain threshold ``θ``.  An update is considered *valid*
        only when ``G_i > θ``.
    gamma : float
        Reputation update factor ``γ``.  Controls how strongly a positive
        gain boosts reputation.
    decay_rate : float
        Per-round multiplicative decay applied to clients that did **not**
        participate (models staleness into reputation).  Set to ``1.0`` to
        disable decay.
    floor : float
        Minimum reputation — prevents permanent exclusion so every client
        retains a chance to be re-selected.
    ceiling : float
        Maximum reputation (typically ``1.0``).
    initial_reputation : float
        Reputation assigned to newly registered clients.
    penalty_factor : float
        Fraction of ``|G_i|`` subtracted from reputation when the update
        is actively harmful (``G_i < -θ``).  Set to ``0.0`` for the
        "no-change-on-invalid" variant from the thesis pseudo-code.
    """
    theta: float = 0.0               # validation-gain threshold
    gamma: float = 0.10              # reputation update factor
    decay_rate: float = 0.995        # per-round decay for non-participants
    floor: float = 0.05              # minimum reputation
    ceiling: float = 1.0             # maximum reputation
    initial_reputation: float = 0.5  # starting score
    penalty_factor: float = 0.0      # penalty multiplier for harmful updates


# ====================================================================== #
#  2.  LEDGER ENTRY — per-client record                                   #
# ====================================================================== #

@dataclass
class ClientLedgerEntry:
    """
    Full reputation record for a single client, including audit history.
    """
    client_id: str
    reputation: float = 0.5
    total_rounds_participated: int = 0
    total_valid_updates: int = 0
    total_invalid_updates: int = 0
    cumulative_gain: float = 0.0
    last_participated_round: int = 0
    history: List[Dict[str, Any]] = field(default_factory=list)

    # ---- serialisation ----------------------------------------------- #

    def to_dict(self) -> Dict[str, Any]:
        return {
            "client_id": self.client_id,
            "reputation": self.reputation,
            "total_rounds_participated": self.total_rounds_participated,
            "total_valid_updates": self.total_valid_updates,
            "total_invalid_updates": self.total_invalid_updates,
            "cumulative_gain": self.cumulative_gain,
            "last_participated_round": self.last_participated_round,
            "history": self.history,
        }

    @classmethod
    def from_dict(cls, d: Dict[str, Any]) -> "ClientLedgerEntry":
        return cls(**d)


# ====================================================================== #
#  3.  CLIENT REPUTATION LEDGER (extended)                                #
# ====================================================================== #

class ClientReputationLedger:
    """
    Persistent, auditable reputation ledger for all federated clients.

    Implements the thesis pseudo-code:

    .. code-block:: text

        For each client i:
            G_i = validation_gain(u_i)
            if G_i > θ:  is_valid, delta_i = True,  G_i
            else:         is_valid, delta_i = False, 0
            if is_valid:  R_i = min(1, R_i + γ · delta_i)
            else:         R_i = R_i          # no change
            store(i, R_i)

    Additionally supports:

    * Optional **penalty** for harmful updates (``G_i < -θ``).
    * **Decay** for non-participating clients each round.
    * Full **history** / audit trail per client.
    * **JSON persistence** (save / load).
    * Backward-compatible bridge to the basic ``ReputationLedger`` from
      Part 1 via :meth:`as_basic_ledger`.

    Parameters
    ----------
    config : ReputationConfig
        All tuneable knobs.
    """

    def __init__(self, config: Optional[ReputationConfig] = None) -> None:
        self.config = config or ReputationConfig()
        self._entries: Dict[str, ClientLedgerEntry] = {}

    # ------------------------------------------------------------------ #
    #  Registration                                                       #
    # ------------------------------------------------------------------ #

    def register(self, client_id: str) -> None:
        """Register a new client with the default initial reputation."""
        if client_id not in self._entries:
            self._entries[client_id] = ClientLedgerEntry(
                client_id=client_id,
                reputation=self.config.initial_reputation,
            )
            logger.debug("Registered client %s (R=%.4f).",
                         client_id, self.config.initial_reputation)

    def register_many(self, client_ids: List[str]) -> None:
        for cid in client_ids:
            self.register(cid)

    # ------------------------------------------------------------------ #
    #  Read                                                                #
    # ------------------------------------------------------------------ #

    def get(self, client_id: str) -> float:
        """Return the current reputation for *client_id*."""
        entry = self._entries.get(client_id)
        if entry is None:
            return self.config.initial_reputation
        return entry.reputation

    def get_entry(self, client_id: str) -> Optional[ClientLedgerEntry]:
        """Return the full ledger entry (or ``None``)."""
        return self._entries.get(client_id)

    def all_reputations(self) -> Dict[str, float]:
        """Return ``{client_id: reputation}`` for every registered client."""
        return {cid: e.reputation for cid, e in self._entries.items()}

    # ------------------------------------------------------------------ #
    #  Core update — implements the thesis pseudo-code                    #
    # ------------------------------------------------------------------ #

    def update_reputation(
        self,
        client_id: str,
        validation_gain: float,
        current_round: int,
        contribution_weight: Optional[float] = None,
    ) -> float:
        """
        Update a single client's reputation based on validation gain.

        Follows the pseudo-code:

        .. code-block:: text

            if G_i > θ:  R_i = min(ceiling, R_i + γ · G_i)
            else:        R_i = R_i                       # no change
            (optional)   if G_i < -θ:  R_i = max(floor, R_i - penalty · |G_i|)

        Parameters
        ----------
        client_id : str
        validation_gain : float
            ``G_i`` — the measured improvement on the server validation set.
        current_round : int
            Current federated round number (for the audit trail).
        contribution_weight : float | None
            ``c_i`` from Part 2 (stored in the audit trail, not used in
            the reputation formula itself).

        Returns
        -------
        new_reputation : float
        """
        self.register(client_id)                 # idempotent
        entry = self._entries[client_id]
        cfg = self.config
        old_rep = entry.reputation

        # --- 1. Determine validity ------------------------------------ #
        is_valid = validation_gain > cfg.theta
        delta = validation_gain if is_valid else 0.0

        # --- 2. Compute new reputation -------------------------------- #
        if is_valid:
            new_rep = min(cfg.ceiling, old_rep + cfg.gamma * delta)
        else:
            new_rep = old_rep                    # no change on invalid

        # --- 2b. Optional penalty for actively harmful updates -------- #
        if cfg.penalty_factor > 0 and validation_gain < -cfg.theta:
            penalty = cfg.penalty_factor * abs(validation_gain)
            new_rep = max(cfg.floor, new_rep - penalty)

        # --- 3. Clamp to [floor, ceiling] ----------------------------- #
        new_rep = float(np.clip(new_rep, cfg.floor, cfg.ceiling))

        # --- 4. Store ------------------------------------------------- #
        entry.reputation = new_rep
        entry.total_rounds_participated += 1
        entry.last_participated_round = current_round
        entry.cumulative_gain += validation_gain
        if is_valid:
            entry.total_valid_updates += 1
        else:
            entry.total_invalid_updates += 1

        # Audit trail
        entry.history.append({
            "round": current_round,
            "G_i": round(validation_gain, 6),
            "is_valid": is_valid,
            "delta": round(delta, 6),
            "R_old": round(old_rep, 6),
            "R_new": round(new_rep, 6),
            "c_i": round(contribution_weight, 6) if contribution_weight is not None else None,
            "ts": time.time(),
        })

        logger.debug(
            "Client %s: G_i=%.4f, valid=%s, R %.4f → %.4f",
            client_id, validation_gain, is_valid, old_rep, new_rep,
        )
        return new_rep

    # ------------------------------------------------------------------ #
    #  Batch update — process all clients in a round                      #
    # ------------------------------------------------------------------ #

    def update_round(
        self,
        gains: Dict[str, float],
        current_round: int,
        contribution_weights: Optional[Dict[str, float]] = None,
        participant_ids: Optional[List[str]] = None,
    ) -> Dict[str, float]:
        """
        Update reputations for an entire federated round.

        Parameters
        ----------
        gains : dict[str, float]
            ``{client_id: G_i}`` for each participating client.
        current_round : int
        contribution_weights : dict[str, float] | None
            ``{client_id: c_i}`` from Part 2 (for the audit trail).
        participant_ids : list[str] | None
            Explicit list of participants.  If ``None``, derived from
            ``gains.keys()``.

        Returns
        -------
        updated : dict[str, float]
            ``{client_id: new_reputation}`` for every registered client
            (including decayed non-participants).
        """
        participants = set(participant_ids or gains.keys())
        cw = contribution_weights or {}

        # Update participants
        for cid in participants:
            g = gains.get(cid, 0.0)
            c = cw.get(cid)
            self.update_reputation(cid, g, current_round, c)

        # Decay non-participants
        self._decay_non_participants(participants, current_round)

        return self.all_reputations()

    # ------------------------------------------------------------------ #
    #  Decay for idle clients                                             #
    # ------------------------------------------------------------------ #

    def _decay_non_participants(
        self,
        participants: set,
        current_round: int,
    ) -> None:
        """
        Apply multiplicative decay to clients that did **not** participate
        this round.  Keeps reputation ≥ floor.
        """
        cfg = self.config
        if cfg.decay_rate >= 1.0:
            return                               # decay disabled

        for cid, entry in self._entries.items():
            if cid in participants:
                continue
            old = entry.reputation
            new = max(cfg.floor, old * cfg.decay_rate)
            if abs(new - old) > 1e-9:
                entry.reputation = new
                entry.history.append({
                    "round": current_round,
                    "event": "decay",
                    "R_old": round(old, 6),
                    "R_new": round(new, 6),
                    "ts": time.time(),
                })
                logger.debug(
                    "Client %s decayed: R %.4f → %.4f (non-participant).",
                    cid, old, new,
                )

    # ------------------------------------------------------------------ #
    #  Querying & analytics                                               #
    # ------------------------------------------------------------------ #

    def ranked(self, descending: bool = True) -> List[Tuple[str, float]]:
        """Return ``(client_id, reputation)`` sorted by reputation."""
        items = list(self.all_reputations().items())
        items.sort(key=lambda x: x[1], reverse=descending)
        return items

    def statistics(self) -> Dict[str, Any]:
        """Aggregate statistics across all registered clients."""
        reps = [e.reputation for e in self._entries.values()]
        if not reps:
            return {}
        arr = np.array(reps)
        return {
            "num_clients": len(reps),
            "mean_reputation": float(arr.mean()),
            "std_reputation": float(arr.std()),
            "min_reputation": float(arr.min()),
            "max_reputation": float(arr.max()),
            "median_reputation": float(np.median(arr)),
        }

    def client_summary(self, client_id: str) -> Optional[Dict[str, Any]]:
        """Return a human-readable summary dict for one client."""
        entry = self._entries.get(client_id)
        if entry is None:
            return None
        return {
            "client_id": entry.client_id,
            "reputation": entry.reputation,
            "rounds_participated": entry.total_rounds_participated,
            "valid_updates": entry.total_valid_updates,
            "invalid_updates": entry.total_invalid_updates,
            "cumulative_gain": entry.cumulative_gain,
            "last_participated_round": entry.last_participated_round,
            "history_length": len(entry.history),
        }

    # ------------------------------------------------------------------ #
    #  Bridge to Part 1 basic ReputationLedger                            #
    # ------------------------------------------------------------------ #

    def as_basic_ledger(self) -> ReputationLedger:
        """
        Export current reputations into a new ``ReputationLedger``
        (Part 1 format) so the client selector can consume them directly.
        """
        basic = ReputationLedger(
            default_reputation=self.config.initial_reputation,
        )
        for cid, entry in self._entries.items():
            basic.register(cid)
            # Overwrite internal score to match our ledger
            basic._scores[cid] = entry.reputation
        return basic

    def sync_from_basic_ledger(self, basic: ReputationLedger) -> None:
        """
        Import scores from an existing ``ReputationLedger`` (Part 1)
        into this extended ledger — useful when migrating mid-experiment.
        """
        for cid, score in basic.summary().items():
            self.register(cid)
            self._entries[cid].reputation = score

    # ------------------------------------------------------------------ #
    #  Persistence  (JSON)                                                #
    # ------------------------------------------------------------------ #

    def save(self, path: str | Path) -> None:
        """Serialise the full ledger (including history) to a JSON file."""
        data = {
            "config": {
                "theta": self.config.theta,
                "gamma": self.config.gamma,
                "decay_rate": self.config.decay_rate,
                "floor": self.config.floor,
                "ceiling": self.config.ceiling,
                "initial_reputation": self.config.initial_reputation,
                "penalty_factor": self.config.penalty_factor,
            },
            "entries": {cid: e.to_dict() for cid, e in self._entries.items()},
        }
        Path(path).write_text(json.dumps(data, indent=2), encoding="utf-8")
        logger.info("Ledger saved to %s (%d clients).", path, len(self._entries))

    @classmethod
    def load(cls, path: str | Path) -> "ClientReputationLedger":
        """Load a ledger from a previously saved JSON file."""
        raw = json.loads(Path(path).read_text(encoding="utf-8"))
        cfg = ReputationConfig(**raw["config"])
        ledger = cls(config=cfg)
        for cid, edict in raw["entries"].items():
            ledger._entries[cid] = ClientLedgerEntry.from_dict(edict)
        logger.info("Ledger loaded from %s (%d clients).", path, len(ledger._entries))
        return ledger


# ====================================================================== #
#  4.  INTEGRATION HELPER — plug into Parts 2 & 3                        #
# ====================================================================== #

def update_ledger_from_records(
    ledger: ClientReputationLedger,
    records,                        # List[ClientUpdateRecord] from Part 2
    current_round: int,
) -> Dict[str, float]:
    """
    Convenience function: extract ``G_i`` and ``c_i`` from Part 2's
    ``ClientUpdateRecord`` list and feed them into the extended ledger.

    Parameters
    ----------
    ledger : ClientReputationLedger
    records : list[ClientUpdateRecord]
        Output of ``UpdateValidator.validate_updates()``.
    current_round : int

    Returns
    -------
    updated : dict[str, float]
    """
    gains: Dict[str, float] = {}
    cws: Dict[str, float] = {}
    for rec in records:
        gains[rec.client_id] = rec.validation_gain
        cws[rec.client_id] = rec.contribution_weight
    return ledger.update_round(gains, current_round, contribution_weights=cws)


# ====================================================================== #
#  DEMO / SMOKE-TEST                                                      #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  Client Reputation & Ledger — Demo  ===\n")

    np.random.seed(42)

    NUM_CLIENTS = 8
    NUM_ROUNDS = 10

    # ---- Configuration ----------------------------------------------- #
    cfg = ReputationConfig(
        theta=0.0,                    # any positive gain counts
        gamma=0.10,                   # reputation update factor
        decay_rate=0.99,              # 1 % decay per idle round
        floor=0.05,                   # minimum reputation
        ceiling=1.0,
        initial_reputation=0.50,
        penalty_factor=0.05,          # small penalty for harmful updates
    )

    ledger = ClientReputationLedger(config=cfg)
    client_ids = [f"client_{i:02d}" for i in range(NUM_CLIENTS)]
    ledger.register_many(client_ids)

    print(f"Config: θ={cfg.theta}, γ={cfg.gamma}, decay={cfg.decay_rate}, "
          f"floor={cfg.floor}, penalty={cfg.penalty_factor}")
    print(f"Registered {NUM_CLIENTS} clients, initial R={cfg.initial_reputation}\n")

    # ---- Simulate rounds --------------------------------------------- #
    for rnd in range(1, NUM_ROUNDS + 1):
        # Each round, randomly select ~60 % of clients
        num_selected = max(2, int(0.6 * NUM_CLIENTS))
        participants = list(np.random.choice(client_ids, size=num_selected, replace=False))

        # Simulate validation gains — mostly small positive, some negative
        gains: Dict[str, float] = {}
        for cid in participants:
            g = np.random.normal(loc=0.02, scale=0.04)   # mean 2 %, σ 4 %
            gains[cid] = float(g)

        # Simulate contribution weights (from Part 2)
        cws = {cid: float(np.random.uniform(0.3, 1.0)) for cid in participants}

        updated = ledger.update_round(
            gains=gains,
            current_round=rnd,
            contribution_weights=cws,
            participant_ids=participants,
        )

        # Print round summary
        print(f"--- Round {rnd} ---  participants: {participants}")
        print(f"  Gains:  { {k: f'{v:+.4f}' for k, v in gains.items()} }")
        ranked = ledger.ranked()
        print(f"  Reputations: { {k: f'{v:.4f}' for k, v in ranked} }\n")

    # ---- Final statistics -------------------------------------------- #
    print("=" * 50)
    print("Final Ledger Statistics:")
    stats = ledger.statistics()
    for k, v in stats.items():
        print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

    print("\nPer-client summaries:")
    for cid in client_ids:
        s = ledger.client_summary(cid)
        print(f"  {cid}: R={s['reputation']:.4f}, "
              f"participated={s['rounds_participated']}, "
              f"valid={s['valid_updates']}, invalid={s['invalid_updates']}, "
              f"cumG={s['cumulative_gain']:+.4f}")

    # ---- Persistence round-trip -------------------------------------- #
    save_path = "demo_ledger.json"
    ledger.save(save_path)
    loaded = ClientReputationLedger.load(save_path)
    assert loaded.all_reputations() == ledger.all_reputations(), "Round-trip mismatch!"
    print(f"\nJSON round-trip OK ({save_path})")

    # ---- Bridge to Part 1 -------------------------------------------- #
    basic = ledger.as_basic_ledger()
    print(f"\nBasic ReputationLedger bridge: {basic.summary()}")

    # Clean up demo file
    Path(save_path).unlink(missing_ok=True)

    print("\nDone.")

Overwriting client_reputation_ledger.py


### Part 5 — Evaluation Metrics & Report Generation


In [10]:
%%writefile evaluation_metrics.py
"""
Evaluation Metrics & Report Generation
=======================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Evaluates the global model after the federated learning cycle on:

* **Accuracy**
* **F1 Score** (macro, weighted, and per-class)
* **ROC-AUC**
* **Inference Latency** (mean, std, p95 over repeated batches)
* **Model Size** (parameter count + on-disk file size)

Generates structured reports (JSON + human-readable text) into a
dedicated ``reports/`` folder to keep the workspace organised.
"""

from __future__ import annotations

import json
import logging
import os
import tempfile
import time
from dataclasses import dataclass, field, asdict
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)

# ---------------------------------------------------------------------------
# Default report output directory (sibling of this file)
# ---------------------------------------------------------------------------
_MODULE_DIR = Path(__file__).resolve().parent
DEFAULT_REPORTS_DIR = _MODULE_DIR / "reports"


# ====================================================================== #
#  1.  DATA STRUCTURES                                                    #
# ====================================================================== #

@dataclass
class ClassificationMetrics:
    """Container for all classification-related evaluation metrics."""
    accuracy: float = 0.0
    f1_macro: float = 0.0
    f1_weighted: float = 0.0
    f1_per_class: Dict[str, float] = field(default_factory=dict)
    precision_macro: float = 0.0
    recall_macro: float = 0.0
    roc_auc: float = 0.0
    confusion_matrix: Optional[List[List[int]]] = None
    num_samples: int = 0

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class LatencyMetrics:
    """Container for inference-latency measurements."""
    mean_ms: float = 0.0
    std_ms: float = 0.0
    min_ms: float = 0.0
    max_ms: float = 0.0
    p50_ms: float = 0.0
    p95_ms: float = 0.0
    p99_ms: float = 0.0
    num_batches: int = 0
    batch_size: int = 0
    total_samples: int = 0

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class ModelSizeMetrics:
    """Container for model-size information."""
    total_params: int = 0
    trainable_params: int = 0
    non_trainable_params: int = 0
    file_size_bytes: int = 0
    file_size_mb: float = 0.0

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


@dataclass
class EvaluationReport:
    """Full evaluation report aggregating all metric categories."""
    model_name: str = ""
    timestamp: str = ""
    federated_round: Optional[int] = None
    classification: ClassificationMetrics = field(default_factory=ClassificationMetrics)
    latency: LatencyMetrics = field(default_factory=LatencyMetrics)
    model_size: ModelSizeMetrics = field(default_factory=ModelSizeMetrics)
    extra: Dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> Dict[str, Any]:
        return {
            "model_name": self.model_name,
            "timestamp": self.timestamp,
            "federated_round": self.federated_round,
            "classification": self.classification.to_dict(),
            "latency": self.latency.to_dict(),
            "model_size": self.model_size.to_dict(),
            "extra": self.extra,
        }


# ====================================================================== #
#  2.  METRIC COMPUTATIONS                                                #
# ====================================================================== #

def compute_classification_metrics(
    y_true: np.ndarray,
    y_pred_probs: np.ndarray,
    threshold: float = 0.5,
    class_names: Optional[List[str]] = None,
) -> ClassificationMetrics:
    """
    Compute accuracy, F1 (macro/weighted/per-class), precision, recall,
    ROC-AUC, and confusion matrix from ground-truth labels and predicted
    probabilities.

    Parameters
    ----------
    y_true : np.ndarray, shape ``(N,)``
        Ground-truth binary labels (0 or 1).
    y_pred_probs : np.ndarray, shape ``(N,)`` or ``(N, 1)``
        Predicted probabilities for the positive class.
    threshold : float
        Decision threshold for converting probabilities to hard labels.
    class_names : list[str] | None
        Human-readable names for class 0 and class 1.
        Defaults to ``["Real", "Fake"]``.

    Returns
    -------
    ClassificationMetrics
    """
    if class_names is None:
        class_names = ["Real", "Fake"]

    y_true = np.asarray(y_true).ravel().astype(int)
    y_probs = np.asarray(y_pred_probs).ravel().astype(float)
    y_pred = (y_probs >= threshold).astype(int)
    n = len(y_true)

    # --- Confusion matrix --------------------------------------------- #
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    cm = [[tn, fp], [fn, tp]]

    # --- Accuracy ----------------------------------------------------- #
    accuracy = (tp + tn) / max(n, 1)

    # --- Per-class precision / recall / F1 ---------------------------- #
    def _prf(tp_c: int, fp_c: int, fn_c: int):
        prec = tp_c / max(tp_c + fp_c, 1)
        rec = tp_c / max(tp_c + fn_c, 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-12)
        return prec, rec, f1

    prec_0, rec_0, f1_0 = _prf(tn, fn, fp)   # class 0 = Real
    prec_1, rec_1, f1_1 = _prf(tp, fp, fn)   # class 1 = Fake

    support_0 = int(np.sum(y_true == 0))
    support_1 = int(np.sum(y_true == 1))

    f1_macro = (f1_0 + f1_1) / 2.0
    f1_weighted = (f1_0 * support_0 + f1_1 * support_1) / max(n, 1)
    precision_macro = (prec_0 + prec_1) / 2.0
    recall_macro = (rec_0 + rec_1) / 2.0

    f1_per_class = {
        class_names[0]: round(f1_0, 6),
        class_names[1]: round(f1_1, 6),
    }

    # --- ROC-AUC ------------------------------------------------------ #
    roc_auc = _compute_roc_auc(y_true, y_probs)

    return ClassificationMetrics(
        accuracy=round(accuracy, 6),
        f1_macro=round(f1_macro, 6),
        f1_weighted=round(f1_weighted, 6),
        f1_per_class=f1_per_class,
        precision_macro=round(precision_macro, 6),
        recall_macro=round(recall_macro, 6),
        roc_auc=round(roc_auc, 6),
        confusion_matrix=cm,
        num_samples=n,
    )


def _compute_roc_auc(y_true: np.ndarray, y_scores: np.ndarray) -> float:
    """
    Compute ROC-AUC using the trapezoidal rule (no sklearn dependency).

    Returns 0.0 when only one class is present.
    """
    if len(np.unique(y_true)) < 2:
        return 0.0

    # Sort by descending score
    desc = np.argsort(-y_scores)
    y_sorted = y_true[desc]
    scores_sorted = y_scores[desc]

    num_pos = np.sum(y_true == 1)
    num_neg = np.sum(y_true == 0)

    tp = 0
    fp = 0
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0

    # Walk through thresholds (distinct score values)
    for i in range(len(y_sorted)):
        if y_sorted[i] == 1:
            tp += 1
        else:
            fp += 1

        # Only compute a point when the score changes or at the end
        if i == len(y_sorted) - 1 or scores_sorted[i] != scores_sorted[i + 1]:
            tpr = tp / max(num_pos, 1)
            fpr = fp / max(num_neg, 1)
            auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2.0
            prev_fpr = fpr
            prev_tpr = tpr

    return float(auc)


# ====================================================================== #
#  3.  INFERENCE LATENCY                                                  #
# ====================================================================== #

def measure_inference_latency(
    model: tf.keras.Model,
    test_data: tf.data.Dataset,
    batch_size: int = 32,
    warmup_batches: int = 3,
    max_batches: Optional[int] = None,
) -> LatencyMetrics:
    """
    Measure per-batch inference latency.

    Parameters
    ----------
    model : tf.keras.Model
    test_data : tf.data.Dataset
        Yields ``(images,)`` or ``(images, labels)``.
    batch_size : int
    warmup_batches : int
        Number of initial batches to discard (JIT warm-up).
    max_batches : int | None
        Cap the number of measured batches (``None`` = all).

    Returns
    -------
    LatencyMetrics
    """
    batched = test_data.batch(batch_size).prefetch(tf.data.AUTOTUNE)

    times_ms: List[float] = []
    total_samples = 0
    batch_count = 0

    for batch in batched:
        x = batch[0] if isinstance(batch, (list, tuple)) else batch
        batch_count += 1

        if batch_count <= warmup_batches:
            model(x, training=False)   # warm-up, discard timing
            continue

        t0 = time.perf_counter()
        model(x, training=False)
        t1 = time.perf_counter()

        times_ms.append((t1 - t0) * 1000.0)
        total_samples += int(x.shape[0])

        if max_batches is not None and len(times_ms) >= max_batches:
            break

    if not times_ms:
        logger.warning("No batches measured — dataset may be too small for warm-up.")
        return LatencyMetrics(batch_size=batch_size)

    arr = np.array(times_ms)
    return LatencyMetrics(
        mean_ms=round(float(arr.mean()), 4),
        std_ms=round(float(arr.std()), 4),
        min_ms=round(float(arr.min()), 4),
        max_ms=round(float(arr.max()), 4),
        p50_ms=round(float(np.percentile(arr, 50)), 4),
        p95_ms=round(float(np.percentile(arr, 95)), 4),
        p99_ms=round(float(np.percentile(arr, 99)), 4),
        num_batches=len(times_ms),
        batch_size=batch_size,
        total_samples=total_samples,
    )


# ====================================================================== #
#  4.  MODEL SIZE                                                         #
# ====================================================================== #

def measure_model_size(
    model: tf.keras.Model,
    save_path: Optional[str] = None,
) -> ModelSizeMetrics:
    """
    Count parameters and measure on-disk file size of the model.

    Parameters
    ----------
    model : tf.keras.Model
    save_path : str | None
        If ``None``, a temporary file is used and deleted afterwards.

    Returns
    -------
    ModelSizeMetrics
    """
    total = int(model.count_params())
    trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    non_trainable = total - trainable

    # Estimate on-disk size directly from weight arrays to avoid
    # deepcopy/pickle errors in Keras 3.x when the model config
    # contains non-serialisable objects (e.g. module references).
    file_bytes = sum(w.numpy().nbytes for w in model.weights)

    return ModelSizeMetrics(
        total_params=total,
        trainable_params=trainable,
        non_trainable_params=non_trainable,
        file_size_bytes=file_bytes,
        file_size_mb=round(file_bytes / (1024 * 1024), 4),
    )


# ====================================================================== #
#  5.  FULL EVALUATOR                                                     #
# ====================================================================== #

class FederatedModelEvaluator:
    """
    One-stop evaluator that runs all metrics and produces reports.

    Parameters
    ----------
    model : tf.keras.Model
        The global model to evaluate.
    model_name : str
        Human-readable label (used in report filenames).
    reports_dir : str | Path
        Directory where reports are saved.
    class_names : list[str]
        Names for class 0 / class 1.  Defaults to ``["Real", "Fake"]``.
    """

    def __init__(
        self,
        model: tf.keras.Model,
        model_name: str = "effnet_global",
        reports_dir: str | Path = DEFAULT_REPORTS_DIR,
        class_names: Optional[List[str]] = None,
    ) -> None:
        self.model = model
        self.model_name = model_name
        self.reports_dir = Path(reports_dir)
        self.class_names = class_names or ["Real", "Fake"]

        # Ensure the reports directory exists
        self.reports_dir.mkdir(parents=True, exist_ok=True)

    # ------------------------------------------------------------------ #
    #  Run full evaluation                                                #
    # ------------------------------------------------------------------ #

    def evaluate(
        self,
        test_data: tf.data.Dataset,
        batch_size: int = 32,
        threshold: float = 0.5,
        federated_round: Optional[int] = None,
        warmup_batches: int = 3,
        latency_max_batches: Optional[int] = None,
        extra_info: Optional[Dict[str, Any]] = None,
    ) -> EvaluationReport:
        """
        Run **all** evaluations and return a structured report.

        Parameters
        ----------
        test_data : tf.data.Dataset
            Yields ``(images, labels)``.
        batch_size : int
        threshold : float
            Classification decision threshold.
        federated_round : int | None
            If provided, included in the report metadata.
        warmup_batches : int
            Warm-up batches for latency measurement.
        latency_max_batches : int | None
            Cap measured batches for latency.
        extra_info : dict | None
            Arbitrary metadata to attach to the report.

        Returns
        -------
        EvaluationReport
        """
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        logger.info("Starting evaluation for '%s' …", self.model_name)

        # --- 1. Predictions ------------------------------------------- #
        y_true_list: List[np.ndarray] = []
        y_prob_list: List[np.ndarray] = []

        batched = test_data.batch(batch_size).prefetch(tf.data.AUTOTUNE)
        for batch in batched:
            x, y = batch[0], batch[1]
            preds = self.model(x, training=False)
            y_true_list.append(y.numpy())
            y_prob_list.append(preds.numpy())

        y_true = np.concatenate(y_true_list).ravel()
        y_probs = np.concatenate(y_prob_list).ravel()

        # --- 2. Classification metrics -------------------------------- #
        cls_metrics = compute_classification_metrics(
            y_true, y_probs,
            threshold=threshold,
            class_names=self.class_names,
        )
        logger.info(
            "Classification — Acc: %.4f | F1-macro: %.4f | ROC-AUC: %.4f",
            cls_metrics.accuracy, cls_metrics.f1_macro, cls_metrics.roc_auc,
        )

        # --- 3. Inference latency ------------------------------------- #
        lat_metrics = measure_inference_latency(
            self.model, test_data,
            batch_size=batch_size,
            warmup_batches=warmup_batches,
            max_batches=latency_max_batches,
        )
        logger.info(
            "Latency — mean: %.2f ms | p95: %.2f ms | p99: %.2f ms",
            lat_metrics.mean_ms, lat_metrics.p95_ms, lat_metrics.p99_ms,
        )

        # --- 4. Model size -------------------------------------------- #
        size_metrics = measure_model_size(self.model)
        logger.info(
            "Model size — params: %s | disk: %.2f MB",
            f"{size_metrics.total_params:,}", size_metrics.file_size_mb,
        )

        report = EvaluationReport(
            model_name=self.model_name,
            timestamp=ts,
            federated_round=federated_round,
            classification=cls_metrics,
            latency=lat_metrics,
            model_size=size_metrics,
            extra=extra_info or {},
        )

        return report

    # ------------------------------------------------------------------ #
    #  Report persistence                                                 #
    # ------------------------------------------------------------------ #

    def save_report(
        self,
        report: EvaluationReport,
        tag: Optional[str] = None,
    ) -> Tuple[Path, Path]:
        """
        Save the report as both JSON and a human-readable text file.

        Parameters
        ----------
        report : EvaluationReport
        tag : str | None
            Optional suffix for the filename (e.g. ``"round_05"``).

        Returns
        -------
        (json_path, txt_path) : tuple[Path, Path]
        """
        ts_slug = datetime.now().strftime("%Y%m%d_%H%M%S")
        base = f"{self.model_name}_{ts_slug}"
        if tag:
            base += f"_{tag}"

        json_path = self.reports_dir / f"{base}.json"
        txt_path = self.reports_dir / f"{base}.txt"

        # --- JSON ----------------------------------------------------- #
        json_path.write_text(
            json.dumps(report.to_dict(), indent=2, default=str),
            encoding="utf-8",
        )

        # --- Human-readable text -------------------------------------- #
        txt_path.write_text(
            self._format_text_report(report),
            encoding="utf-8",
        )

        logger.info("Reports saved → %s  &  %s", json_path.name, txt_path.name)
        return json_path, txt_path

    # ------------------------------------------------------------------ #
    #  Text report formatter                                              #
    # ------------------------------------------------------------------ #

    @staticmethod
    def _format_text_report(r: EvaluationReport) -> str:
        """Render a pretty-printed text report."""
        sep = "=" * 62
        cls = r.classification
        lat = r.latency
        sz = r.model_size

        lines = [
            sep,
            "  EVALUATION REPORT",
            sep,
            f"  Model:            {r.model_name}",
            f"  Timestamp:        {r.timestamp}",
        ]
        if r.federated_round is not None:
            lines.append(f"  Federated Round:  {r.federated_round}")
        lines.append("")

        # Classification
        lines += [
            "-" * 62,
            "  CLASSIFICATION METRICS",
            "-" * 62,
            f"  Samples evaluated:  {cls.num_samples:,}",
            f"  Accuracy:           {cls.accuracy:.4f}",
            f"  F1 (macro):         {cls.f1_macro:.4f}",
            f"  F1 (weighted):      {cls.f1_weighted:.4f}",
            f"  Precision (macro):  {cls.precision_macro:.4f}",
            f"  Recall (macro):     {cls.recall_macro:.4f}",
            f"  ROC-AUC:            {cls.roc_auc:.4f}",
            "",
            "  Per-class F1:",
        ]
        for name, f1 in cls.f1_per_class.items():
            lines.append(f"    {name:<16} {f1:.4f}")

        if cls.confusion_matrix is not None:
            cm = cls.confusion_matrix
            lines += [
                "",
                "  Confusion Matrix:     Pred=0   Pred=1",
                f"    Actual=0 (Real)    {cm[0][0]:>7,}  {cm[0][1]:>7,}",
                f"    Actual=1 (Fake)    {cm[1][0]:>7,}  {cm[1][1]:>7,}",
            ]

        # Latency
        lines += [
            "",
            "-" * 62,
            "  INFERENCE LATENCY",
            "-" * 62,
            f"  Batches measured:   {lat.num_batches}  (batch_size={lat.batch_size})",
            f"  Total samples:      {lat.total_samples:,}",
            f"  Mean:               {lat.mean_ms:.2f} ms",
            f"  Std:                {lat.std_ms:.2f} ms",
            f"  Min:                {lat.min_ms:.2f} ms",
            f"  Max:                {lat.max_ms:.2f} ms",
            f"  P50 (median):       {lat.p50_ms:.2f} ms",
            f"  P95:                {lat.p95_ms:.2f} ms",
            f"  P99:                {lat.p99_ms:.2f} ms",
        ]

        # Model size
        lines += [
            "",
            "-" * 62,
            "  MODEL SIZE",
            "-" * 62,
            f"  Total params:       {sz.total_params:,}",
            f"  Trainable params:   {sz.trainable_params:,}",
            f"  Non-trainable:      {sz.non_trainable_params:,}",
            f"  File size:          {sz.file_size_mb:.2f} MB  ({sz.file_size_bytes:,} bytes)",
        ]

        # Extra
        if r.extra:
            lines += [
                "",
                "-" * 62,
                "  ADDITIONAL INFO",
                "-" * 62,
            ]
            for k, v in r.extra.items():
                lines.append(f"  {k}: {v}")

        lines += ["", sep, "  END OF REPORT", sep, ""]
        return "\n".join(lines)

    # ------------------------------------------------------------------ #
    #  Comparative report across rounds                                   #
    # ------------------------------------------------------------------ #

    def save_comparison_report(
        self,
        reports: List[EvaluationReport],
        filename: str = "comparison",
    ) -> Tuple[Path, Path]:
        """
        Generate a comparison table across multiple evaluation reports
        (e.g. one per federated round).

        Returns ``(json_path, txt_path)``.
        """
        ts_slug = datetime.now().strftime("%Y%m%d_%H%M%S")
        base = f"{filename}_{ts_slug}"
        json_path = self.reports_dir / f"{base}.json"
        txt_path = self.reports_dir / f"{base}.txt"

        # JSON
        json_data = [r.to_dict() for r in reports]
        json_path.write_text(
            json.dumps(json_data, indent=2, default=str),
            encoding="utf-8",
        )

        # Text table
        header = (
            f"{'Round':>6} | {'Acc':>8} | {'F1-mac':>8} | {'ROC-AUC':>8} | "
            f"{'Lat(ms)':>8} | {'P95(ms)':>8} | {'Size(MB)':>9}"
        )
        sep = "-" * len(header)
        lines = [
            "=" * len(header),
            "  COMPARISON REPORT",
            "=" * len(header),
            "",
            header,
            sep,
        ]
        for r in reports:
            rnd = r.federated_round if r.federated_round is not None else "—"
            lines.append(
                f"{rnd:>6} | {r.classification.accuracy:>8.4f} | "
                f"{r.classification.f1_macro:>8.4f} | "
                f"{r.classification.roc_auc:>8.4f} | "
                f"{r.latency.mean_ms:>8.2f} | "
                f"{r.latency.p95_ms:>8.2f} | "
                f"{r.model_size.file_size_mb:>9.2f}"
            )
        lines += [sep, ""]

        txt_path.write_text("\n".join(lines), encoding="utf-8")
        logger.info("Comparison report saved → %s  &  %s", json_path.name, txt_path.name)
        return json_path, txt_path


# ====================================================================== #
#  6.  CONVENIENCE: evaluate + save in one call                           #
# ====================================================================== #

def evaluate_and_report(
    model: tf.keras.Model,
    test_data: tf.data.Dataset,
    model_name: str = "effnet_global",
    reports_dir: str | Path = DEFAULT_REPORTS_DIR,
    batch_size: int = 32,
    threshold: float = 0.5,
    federated_round: Optional[int] = None,
    extra_info: Optional[Dict[str, Any]] = None,
    tag: Optional[str] = None,
) -> EvaluationReport:
    """
    One-liner: evaluate the model and write JSON + text reports.

    Returns the ``EvaluationReport`` dataclass for programmatic use.
    """
    evaluator = FederatedModelEvaluator(
        model=model,
        model_name=model_name,
        reports_dir=reports_dir,
    )
    report = evaluator.evaluate(
        test_data,
        batch_size=batch_size,
        threshold=threshold,
        federated_round=federated_round,
        extra_info=extra_info,
    )
    evaluator.save_report(report, tag=tag)
    return report


# ====================================================================== #
#  DEMO / SMOKE-TEST                                                      #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  Evaluation Metrics & Report Generation — Demo  ===\n")

    np.random.seed(42)
    tf.random.set_seed(42)

    # ---- 1. Build a tiny model --------------------------------------- #
    INPUT_DIM = 16
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(INPUT_DIM,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])

    # ---- 2. Synthetic test data -------------------------------------- #
    NUM_SAMPLES = 500
    x_test = np.random.randn(NUM_SAMPLES, INPUT_DIM).astype(np.float32)
    y_test = np.random.randint(0, 2, size=(NUM_SAMPLES,)).astype(np.float32)
    test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))

    # ---- 3. Run evaluation ------------------------------------------- #
    evaluator = FederatedModelEvaluator(
        model=model,
        model_name="demo_effnet",
        reports_dir=DEFAULT_REPORTS_DIR,
    )

    report = evaluator.evaluate(
        test_data=test_ds,
        batch_size=64,
        federated_round=5,
        extra_info={"dataset": "synthetic", "notes": "smoke-test"},
    )

    # ---- 4. Print text report to console ----------------------------- #
    print(evaluator._format_text_report(report))

    # ---- 5. Save reports to disk ------------------------------------- #
    json_path, txt_path = evaluator.save_report(report, tag="round_05")
    print(f"JSON report: {json_path}")
    print(f"Text report: {txt_path}")

    # ---- 6. Simulate multi-round comparison -------------------------- #
    reports: List[EvaluationReport] = []
    for rnd in range(1, 4):
        # Slightly perturb weights to simulate different rounds
        for w in model.trainable_weights:
            w.assign_add(tf.random.normal(w.shape, stddev=0.01))
        r = evaluator.evaluate(test_ds, batch_size=64, federated_round=rnd)
        reports.append(r)

    cmp_json, cmp_txt = evaluator.save_comparison_report(reports)
    print(f"\nComparison JSON: {cmp_json}")
    print(f"Comparison text: {cmp_txt}")

    print("\nDone.")

Overwriting evaluation_metrics.py


### Part 6 — Main FL Cycle Orchestrator (non-TFF, pure Keras)


In [11]:
%%writefile federated_learning_cycle.py
"""
Federated Learning Cycle — Main Orchestrator
==============================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Integrates **all five** modules into one end-to-end pipeline:

 1. **Enhanced Client Selection**  (``enhanced_client_selection.py``)
 2. **Update Validation & Contribution Weighing** (``update_validation.py``)
 3. **Server-Side Knowledge Distillation**  (``knowledge_distillation.py``)
 4. **Client Reputation & Ledger**  (``client_reputation_ledger.py``)
 5. **Evaluation Metrics & Reporting**  (``evaluation_metrics.py``)

Cycle summary (per round)
-------------------------
1.  Select clients from the pool via multi-criteria scoring  (Part 1).
2.  Distribute global weights → clients train locally for 5 epochs.
3.  Validate updates, assign contribution weights, reject harmful ones,
    and perform weighted aggregation  (Part 2).
4.  Optionally refine the aggregated model with server-side knowledge
    distillation from the contribution-weighted client ensemble  (Part 3).
5.  Update the persistent reputation ledger based on gains & c_i  (Part 4).
6.  Every ``eval_every`` rounds, run full evaluation & save reports  (Part 5).
7.  After all rounds, convert the final model to TensorFlow Lite.

Configuration
-------------
- **Model**: ``phase3_best.weights.h5``   (EfficientNet‑based binary
  classifier — Real vs Fake)
- **Devices**: 100 simulated federated clients
- **Local epochs per round**: 5
- **Global aggregation rounds**: 50
- **Frameworks**: TensorFlow / TF Lite / TF Federated concepts
"""

from __future__ import annotations

import logging
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ====================================================================== #
#  Module imports  (Parts 1–5)                                            #
# ====================================================================== #

from enhanced_client_selection import (       # Part 1
    ClientMetrics,
    FederatedClient,
    ReputationLedger,
    SelectionWeights,
    EnhancedClientSelector,
)
from update_validation import (               # Part 2
    ContributionWeights,
    ClippingConfig,
    ClientUpdateRecord,
    UpdateValidator,
    flatten_weights,
    unflatten_weights,
    compute_update_delta,
    apply_update,
)
from knowledge_distillation import (          # Part 3
    DistillationConfig,
    run_distillation_round,
)
from client_reputation_ledger import (        # Part 4
    ReputationConfig,
    ClientReputationLedger,
    update_ledger_from_records,
)
from evaluation_metrics import (              # Part 5
    FederatedModelEvaluator,
    evaluate_and_report,
)

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  CONFIGURATION                                                      #
# ====================================================================== #

@dataclass
class FLCycleConfig:
    """
    Central configuration for the entire Federated Learning cycle.

    Parameters
    ----------
    model_path : str
        Path to the pre-trained EfficientNet HDF5 model.
    num_devices : int
        Total number of simulated federated client devices.
    local_epochs : int
        Number of local training epochs per client per round.
    global_rounds : int
        Total number of federated aggregation rounds.
    clients_per_round : int
        Target number of clients selected each round (``k``).
    local_batch_size : int
        Batch size for client-side local training.
    local_lr : float
        Client-side optimiser learning rate.
    eval_every : int
        Run full evaluation every N rounds (also round 1 & last).
    enable_distillation : bool
        Whether to run server-side knowledge distillation each round.
    distillation_config : DistillationConfig
        Hyper-parameters for the distillation loop.
    selection_weights : SelectionWeights
        Weights for the multi-criteria client scoring  (Part 1).
    contribution_weights : ContributionWeights
        α, β, γ, δ for the contribution scoring  (Part 2).
    clipping_config : ClippingConfig
        Norm-clipping parameters for update validation.
    harmful_threshold : float
        ε — reject updates with ``G_i < −ε``.
    reputation_config : ReputationConfig
        Parameters for the persistent reputation ledger  (Part 4).
    reports_dir : str
        Directory for saving evaluation reports  (Part 5).
    tflite_output_path : str
        Path for the final TF Lite model export.
    input_shape : tuple
        Input shape expected by the model  (H, W, C).
    """
    # -- Core FL settings ---------------------------------------------- #
    model_path: str = "phase3_best.weights.h5"
    num_devices: int = 100
    local_epochs: int = 5
    global_rounds: int = 50
    clients_per_round: int = 15
    local_batch_size: int = 32
    local_lr: float = 1e-4
    eval_every: int = 5

    # -- Distillation (Part 3) ---------------------------------------- #
    enable_distillation: bool = True
    distillation_config: DistillationConfig = field(
        default_factory=lambda: DistillationConfig(
            temperature=2.0,
            lam=0.5,
            epochs=3,
            batch_size=32,
            learning_rate=1e-4,
        )
    )

    # -- Client selection (Part 1) ------------------------------------- #
    selection_weights: SelectionWeights = field(
        default_factory=lambda: SelectionWeights(
            w_v=0.30,
            w_d=0.20,
            w_l=0.10,
            w_r=0.25,
            w_s=0.15,
        )
    )

    # -- Update validation (Part 2) ----------------------------------- #
    contribution_weights: ContributionWeights = field(
        default_factory=lambda: ContributionWeights(
            alpha=0.35,   # validation gain
            beta=0.20,    # similarity
            gamma=0.20,   # data volume
            delta=0.25,   # reputation
        )
    )
    clipping_config: ClippingConfig = field(
        default_factory=lambda: ClippingConfig(
            clip_threshold=10.0,
            clip_value=5.0,
        )
    )
    harmful_threshold: float = 0.02

    # -- Reputation ledger (Part 4) ------------------------------------ #
    reputation_config: ReputationConfig = field(
        default_factory=lambda: ReputationConfig(
            theta=0.0,
            gamma=0.10,
            decay_rate=0.99,
            floor=0.05,
            ceiling=1.0,
            initial_reputation=0.50,
            penalty_factor=0.05,
        )
    )

    # -- Evaluation & output (Part 5) --------------------------------- #
    reports_dir: str = "reports"
    tflite_output_path: str = "effnet_global_final.tflite"
    input_shape: Tuple[int, ...] = (224, 224, 3)


# ====================================================================== #
#  2.  DATA HELPERS  (simulation — replace with real FF++ loaders)        #
# ====================================================================== #

def _generate_synthetic_data(
    num_samples: int,
    input_shape: Tuple[int, ...],
    seed: Optional[int] = None,
) -> tf.data.Dataset:
    """
    Generate a synthetic (random) labelled dataset for smoke-testing.

    In production, replace this with a real FF++ c23 data loader that
    returns ``(image, label)`` pairs as ``tf.data.Dataset``.
    """
    rng = np.random.RandomState(seed)
    x = rng.randn(num_samples, *input_shape).astype(np.float32) * 0.1
    y = rng.randint(0, 2, size=(num_samples,)).astype(np.float32)
    return tf.data.Dataset.from_tensor_slices((x, y))


def _generate_proxy_data(
    num_samples: int,
    input_shape: Tuple[int, ...],
    seed: Optional[int] = None,
) -> tf.data.Dataset:
    """
    Generate unlabelled proxy data (images only) for knowledge
    distillation.  Replace with real FF++ c23 unlabelled images.
    """
    rng = np.random.RandomState(seed)
    x = rng.randn(num_samples, *input_shape).astype(np.float32) * 0.1
    return tf.data.Dataset.from_tensor_slices(x)


def partition_data_iid(
    full_dataset: tf.data.Dataset,
    num_clients: int,
    seed: int = 42,
) -> Dict[str, tf.data.Dataset]:
    """
    IID partition: shuffle the dataset and split evenly across clients.

    Returns
    -------
    dict[str, tf.data.Dataset]
        ``{client_id: local_dataset}``
    """
    all_data = list(full_dataset.shuffle(buffer_size=10_000, seed=seed))
    total = len(all_data)
    shard_size = max(1, total // num_clients)

    partitions: Dict[str, tf.data.Dataset] = {}
    for i in range(num_clients):
        cid = f"client_{i:03d}"
        start = i * shard_size
        end = min(start + shard_size, total)
        if start >= total:
            # Wrap around for the last few clients
            start = start % total
            end = start + shard_size

        shard_x = [elem[0].numpy() for elem in all_data[start:end]]
        shard_y = [elem[1].numpy() for elem in all_data[start:end]]

        if len(shard_x) == 0:
            shard_x = [all_data[0][0].numpy()]
            shard_y = [all_data[0][1].numpy()]

        partitions[cid] = tf.data.Dataset.from_tensor_slices(
            (np.stack(shard_x), np.array(shard_y))
        )
    return partitions


# ====================================================================== #
#  3.  TF LITE CONVERSION                                                 #
# ====================================================================== #

def convert_to_tflite(
    model: tf.keras.Model,
    output_path: str,
    quantise: bool = False,
) -> str:
    """
    Convert a Keras model to TensorFlow Lite format.

    Parameters
    ----------
    model : tf.keras.Model
    output_path : str
    quantise : bool
        If ``True``, apply dynamic-range (post-training) quantisation.

    Returns
    -------
    output_path : str
    """
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    if quantise:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    Path(output_path).write_bytes(tflite_model)
    size_mb = len(tflite_model) / (1024 * 1024)
    logger.info(
        "TF Lite model saved → %s  (%.2f MB, quantised=%s)",
        output_path, size_mb, quantise,
    )
    return output_path


# ====================================================================== #
#  4.  FEDERATED LEARNING CYCLE  (main orchestrator)                      #
# ====================================================================== #

class FederatedLearningCycle:
    """
    End-to-end federated learning cycle orchestrating Parts 1–5.

    Parameters
    ----------
    config : FLCycleConfig
        The full cycle configuration.
    """

    def __init__(self, config: Optional[FLCycleConfig] = None) -> None:
        self.config = config or FLCycleConfig()
        self.global_model: Optional[tf.keras.Model] = None
        self.clients: List[FederatedClient] = []
        self.reputation_ledger: Optional[ClientReputationLedger] = None
        self.basic_ledger: Optional[ReputationLedger] = None
        self.selector: Optional[EnhancedClientSelector] = None
        self.validator: Optional[UpdateValidator] = None
        self.evaluator: Optional[FederatedModelEvaluator] = None

        # Round history
        self.history: Dict[str, list] = {
            "round": [],
            "global_accuracy": [],
            "selected_clients": [],
            "num_accepted": [],
            "num_rejected": [],
            "distillation_loss": [],
        }

    # ------------------------------------------------------------------ #
    #  Initialisation helpers                                              #
    # ------------------------------------------------------------------ #

    def load_global_model(self) -> tf.keras.Model:
        """Load the pre-trained EfficientNet model from disk."""
        cfg = self.config
        logger.info("Loading global model from %s …", cfg.model_path)
        # Register EfficientNet preprocessing so Keras can deserialize the .h5
        from tensorflow.keras.applications.efficientnet import (
            preprocess_input as _effnet_preprocess,
        )
        _custom = {"preprocess_input": _effnet_preprocess}
        model = tf.keras.models.load_model(
            cfg.model_path, custom_objects=_custom, compile=False,
        )
        model.compile(
            optimizer=tf.keras.optimizers.Adam(cfg.local_lr),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        logger.info(
            "Global model loaded — %s params, input shape %s",
            f"{model.count_params():,}", model.input_shape,
        )
        self.global_model = model
        return model

    def create_clients(
        self,
        client_data: Dict[str, tf.data.Dataset],
    ) -> List[FederatedClient]:
        """
        Instantiate ``FederatedClient`` objects for every device.

        Parameters
        ----------
        client_data : dict[str, tf.data.Dataset]
            Pre-partitioned local data per client.
        """
        rng = np.random.RandomState(42)
        clients: List[FederatedClient] = []

        for cid, local_ds in client_data.items():
            n_samples = sum(1 for _ in local_ds)
            metrics = ClientMetrics(
                local_validation_metric=float(rng.uniform(0.4, 0.9)),
                data_volume=n_samples,
                inference_latency=float(rng.uniform(0.01, 0.15)),
                last_selected_round=0,
            )
            clients.append(FederatedClient(
                client_id=cid,
                local_data=local_ds,
                metrics=metrics,
            ))

        logger.info("Created %d federated clients.", len(clients))
        self.clients = clients
        return clients

    def setup_components(self) -> None:
        """
        Wire all the modules together: reputation ledger, client
        selector, update validator, and evaluator.
        """
        cfg = self.config
        assert self.global_model is not None, "Call load_global_model() first."
        assert len(self.clients) > 0, "Call create_clients() first."

        # -- Part 4: Reputation ledger --------------------------------- #
        self.reputation_ledger = ClientReputationLedger(config=cfg.reputation_config)
        for c in self.clients:
            self.reputation_ledger.register(c.client_id)
        # Create a basic ledger view for Part 1
        self.basic_ledger = self.reputation_ledger.as_basic_ledger()

        # -- Part 1: Enhanced client selector -------------------------- #
        self.selector = EnhancedClientSelector(
            clients=self.clients,
            reputation_ledger=self.basic_ledger,
            weights=cfg.selection_weights,
            target_k=cfg.clients_per_round,
        )

        # -- Part 2: Update validator ---------------------------------- #
        self.validator = UpdateValidator(
            global_model=self.global_model,
            reputation_ledger=self.basic_ledger,
            weights=cfg.contribution_weights,
            clipping=cfg.clipping_config,
            harmful_threshold=cfg.harmful_threshold,
            batch_size=cfg.local_batch_size,
        )

        # -- Part 5: Evaluator ---------------------------------------- #
        self.evaluator = FederatedModelEvaluator(
            model=self.global_model,
            model_name="effnet_global",
            reports_dir=cfg.reports_dir,
        )

        logger.info("All FL-cycle components initialised.")

    # ------------------------------------------------------------------ #
    #  Local training                                                     #
    # ------------------------------------------------------------------ #

    def _local_train(
        self,
        client: FederatedClient,
        global_weights: List[np.ndarray],
    ) -> Tuple[List[np.ndarray], int]:
        """
        Train a local model replica for ``local_epochs`` on the client's
        private data, starting from the current global weights.

        Returns
        -------
        updated_weights : list[np.ndarray]
        num_samples : int
        """
        cfg = self.config

        local_model = tf.keras.models.clone_model(self.global_model)
        local_model.build(self.global_model.input_shape)
        local_model.compile(
            optimizer=tf.keras.optimizers.Adam(cfg.local_lr),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        local_model.set_weights(global_weights)

        if client.local_data is None:
            logger.warning(
                "Client %s has no local data — returning global weights.",
                client.client_id,
            )
            return global_weights, 0

        dataset = client.local_data.batch(cfg.local_batch_size)
        local_model.fit(dataset, epochs=cfg.local_epochs, verbose=0)
        return local_model.get_weights(), client.metrics.data_volume

    # ------------------------------------------------------------------ #
    #  Sync reputation bridge                                             #
    # ------------------------------------------------------------------ #

    def _sync_reputation_to_basic_ledger(self) -> None:
        """
        Copy the extended ledger scores into the basic ``ReputationLedger``
        view used by Parts 1 & 2.
        """
        updated_basic = self.reputation_ledger.as_basic_ledger()
        # Update the internal _scores dict of the existing basic ledger
        # so the selector and validator see fresh reputations.
        self.basic_ledger._scores = updated_basic._scores

    # ------------------------------------------------------------------ #
    #  Single round                                                       #
    # ------------------------------------------------------------------ #

    def execute_round(
        self,
        current_round: int,
        server_val_data: tf.data.Dataset,
        proxy_data: Optional[tf.data.Dataset] = None,
        supervised_data: Optional[tf.data.Dataset] = None,
    ) -> Dict[str, Any]:
        """
        Execute one complete federated round.

        Pipeline
        --------
        1.  **Client selection**   (Part 1)
        2.  **Local training**     — each selected client trains for
            ``local_epochs`` on its private data.
        3.  **Update validation & weighted aggregation**  (Part 2)
        4.  **Knowledge distillation**  (Part 3, optional)
        5.  **Reputation ledger update**  (Part 4)
        6.  Quick accuracy check on the server validation set.

        Parameters
        ----------
        current_round : int
        server_val_data : tf.data.Dataset
            Held-out server validation set  ``(x, y)``  for baseline
            comparison and post-round evaluation.
        proxy_data : tf.data.Dataset | None
            Unlabelled proxy inputs for distillation  (Part 3).
        supervised_data : tf.data.Dataset | None
            Labelled data for the combined distillation loss  (Part 3).

        Returns
        -------
        round_info : dict
        """
        cfg = self.config

        logger.info("=" * 70)
        logger.info("  ROUND %d / %d", current_round, cfg.global_rounds)
        logger.info("=" * 70)

        # ── 1. Client selection  (Part 1) ──────────────────────────── #
        selected: List[FederatedClient] = self.selector.select(
            current_round=current_round,
        )
        selected_ids = [c.client_id for c in selected]
        logger.info(
            "Selected %d / %d clients: %s",
            len(selected), len(self.clients), selected_ids,
        )

        global_weights = self.global_model.get_weights()

        # ── 2. Local training ──────────────────────────────────────── #
        client_updates: Dict[str, List[np.ndarray]] = {}
        data_volumes: Dict[str, int] = {}

        for client in selected:
            t0 = time.time()
            updated_w, n_samples = self._local_train(client, global_weights)
            elapsed = time.time() - t0

            client_updates[client.client_id] = updated_w
            data_volumes[client.client_id] = max(n_samples, 1)

            # Refresh client metrics
            client.metrics.last_selected_round = current_round
            client.metrics.inference_latency = elapsed / max(n_samples, 1)

            logger.debug(
                "Client %s trained — %d samples, %.2fs",
                client.client_id, n_samples, elapsed,
            )

        # ── 3. Update validation & aggregation  (Part 2) ──────────── #
        records: List[ClientUpdateRecord] = self.validator.validate_updates(
            client_updates=client_updates,
            data_volumes=data_volumes,
            server_val_data=server_val_data,
        )

        new_weights = self.validator.aggregate_weighted(
            records, global_weights,
        )
        self.global_model.set_weights(new_weights)
        self.validator.global_model.set_weights(new_weights)

        # Rejection statistics
        num_accepted = sum(1 for r in records if not r.rejected)
        num_rejected = sum(1 for r in records if r.rejected)
        logger.info(
            "Updates: %d accepted, %d rejected, out of %d total.",
            num_accepted, num_rejected, len(records),
        )

        # ── 4. Knowledge distillation  (Part 3, optional) ─────────── #
        distill_loss = None
        if cfg.enable_distillation and proxy_data is not None:
            # Only distill from clients with positive contribution
            contribution_weights = {
                r.client_id: r.contribution_weight
                for r in records
                if not r.rejected and r.contribution_weight > 0
            }

            if len(contribution_weights) >= 1:
                logger.info(
                    "Running knowledge distillation with %d teacher(s) …",
                    len(contribution_weights),
                )
                kd_history = run_distillation_round(
                    global_model=self.global_model,
                    client_weights={
                        cid: client_updates[cid]
                        for cid in contribution_weights
                    },
                    contribution_weights=contribution_weights,
                    proxy_data=proxy_data,
                    supervised_data=supervised_data,
                    config=cfg.distillation_config,
                )
                distill_loss = kd_history["loss_total"][-1]
                logger.info(
                    "Distillation complete — final loss %.5f", distill_loss,
                )
                # Update validator reference after distillation
                self.validator.global_model.set_weights(
                    self.global_model.get_weights()
                )
            else:
                logger.info(
                    "Skipping distillation — no contributing "
                    "clients (%d).", len(contribution_weights),
                )

        # ── 5. Reputation ledger update  (Part 4) ─────────────────── #
        updated_reps = update_ledger_from_records(
            ledger=self.reputation_ledger,
            records=records,
            current_round=current_round,
        )
        # Also feed reputation changes back via the basic (Part 1/2)
        # validator update_reputations for the simple ledger
        self.validator.update_reputations(records)

        # Sync extended ledger → basic ledger used by selector/validator
        self._sync_reputation_to_basic_ledger()

        logger.info(
            "Reputation update complete — top 5: %s",
            dict(list(self.reputation_ledger.ranked()[:5])),
        )

        # ── 6. Quick evaluation on val set ────────────────────────── #
        val_result = self.global_model.evaluate(
            server_val_data.batch(cfg.local_batch_size),
            verbose=0,
            return_dict=True,
        )
        acc = val_result.get("accuracy", 0.0)
        logger.info("Round %d — post-round accuracy: %.4f", current_round, acc)

        # ── Collect summary ────────────────────────────────────────── #
        round_info = {
            "round": current_round,
            "selected": selected_ids,
            "num_accepted": num_accepted,
            "num_rejected": num_rejected,
            "records": records,
            "global_accuracy": acc,
            "distillation_loss": distill_loss,
        }
        return round_info

    # ------------------------------------------------------------------ #
    #  Full FL cycle                                                      #
    # ------------------------------------------------------------------ #

    def run(
        self,
        server_val_data: tf.data.Dataset,
        test_data: tf.data.Dataset,
        proxy_data: Optional[tf.data.Dataset] = None,
        supervised_data: Optional[tf.data.Dataset] = None,
    ) -> Dict[str, list]:
        """
        Run the full federated learning cycle for ``global_rounds``
        rounds.

        Parameters
        ----------
        server_val_data : tf.data.Dataset
            Held-out validation set used for update scoring & quick eval.
        test_data : tf.data.Dataset
            Independent test set for full evaluation (Part 5).
        proxy_data : tf.data.Dataset | None
            Unlabelled proxy data for knowledge distillation (Part 3).
        supervised_data : tf.data.Dataset | None
            Labelled data for the combined distillation loss (Part 3).

        Returns
        -------
        history : dict
        """
        cfg = self.config

        logger.info("╔══════════════════════════════════════════════════════╗")
        logger.info("║  FEDERATED LEARNING CYCLE — START                   ║")
        logger.info("║  Devices: %3d  |  Rounds: %3d  |  Local epochs: %d  ║",
                     cfg.num_devices, cfg.global_rounds, cfg.local_epochs)
        logger.info("╚══════════════════════════════════════════════════════╝")

        # -- Pre-cycle evaluation (baseline) --------------------------- #
        logger.info("Evaluating baseline model before federated training …")
        baseline_report = self.evaluator.evaluate(
            test_data=test_data,
            batch_size=cfg.local_batch_size,
            federated_round=0,
            extra_info={"stage": "baseline"},
        )
        self.evaluator.save_report(baseline_report, tag="round_000_baseline")
        logger.info(
            "Baseline — Acc: %.4f, F1: %.4f, AUC: %.4f",
            baseline_report.classification.accuracy,
            baseline_report.classification.f1_macro,
            baseline_report.classification.roc_auc,
        )

        all_reports = [baseline_report]
        cycle_start = time.time()

        # ============================================================== #
        #  MAIN LOOP                                                      #
        # ============================================================== #

        for t in range(1, cfg.global_rounds + 1):
            round_start = time.time()

            info = self.execute_round(
                current_round=t,
                server_val_data=server_val_data,
                proxy_data=proxy_data,
                supervised_data=supervised_data,
            )

            # Record history
            self.history["round"].append(t)
            self.history["global_accuracy"].append(info["global_accuracy"])
            self.history["selected_clients"].append(info["selected"])
            self.history["num_accepted"].append(info["num_accepted"])
            self.history["num_rejected"].append(info["num_rejected"])
            self.history["distillation_loss"].append(info["distillation_loss"])

            round_elapsed = time.time() - round_start
            logger.info("Round %d completed in %.1fs.", t, round_elapsed)

            # -- Periodic full evaluation (Part 5) --------------------- #
            is_eval_round = (
                t % cfg.eval_every == 0
                or t == 1
                or t == cfg.global_rounds
            )
            if is_eval_round:
                logger.info("Running full evaluation (round %d) …", t)
                report = self.evaluator.evaluate(
                    test_data=test_data,
                    batch_size=cfg.local_batch_size,
                    federated_round=t,
                    extra_info={
                        "accepted": info["num_accepted"],
                        "rejected": info["num_rejected"],
                        "distillation_loss": info["distillation_loss"],
                    },
                )
                self.evaluator.save_report(report, tag=f"round_{t:03d}")
                all_reports.append(report)
                logger.info(
                    "Round %d eval — Acc: %.4f, F1: %.4f, AUC: %.4f",
                    t,
                    report.classification.accuracy,
                    report.classification.f1_macro,
                    report.classification.roc_auc,
                )

        total_elapsed = time.time() - cycle_start

        # ============================================================== #
        #  POST-CYCLE                                                     #
        # ============================================================== #

        logger.info("╔══════════════════════════════════════════════════════╗")
        logger.info("║  FEDERATED LEARNING CYCLE — COMPLETE                ║")
        logger.info("║  Total time: %.1fs                                  ║", total_elapsed)
        logger.info("╚══════════════════════════════════════════════════════╝")

        # -- Save comparison report ------------------------------------ #
        if len(all_reports) > 1:
            self.evaluator.save_comparison_report(all_reports)

        # -- Save reputation ledger ------------------------------------ #
        ledger_path = Path(cfg.reports_dir) / "reputation_ledger_final.json"
        self.reputation_ledger.save(str(ledger_path))

        # -- Convert to TF Lite ---------------------------------------- #
        convert_to_tflite(
            self.global_model,
            cfg.tflite_output_path,
            quantise=False,
        )
        # Also save a quantised version
        convert_to_tflite(
            self.global_model,
            cfg.tflite_output_path.replace(".tflite", "_quantised.tflite"),
            quantise=True,
        )

        # -- Print final summary --------------------------------------- #
        self._print_summary()

        return self.history

    # ------------------------------------------------------------------ #
    #  Summary                                                            #
    # ------------------------------------------------------------------ #

    def _print_summary(self) -> None:
        """Pretty-print a final training summary."""
        h = self.history
        if not h["round"]:
            return

        best_idx = int(np.argmax(h["global_accuracy"]))
        best_round = h["round"][best_idx]
        best_acc = h["global_accuracy"][best_idx]
        final_acc = h["global_accuracy"][-1]

        # Reputation statistics
        stats = self.reputation_ledger.statistics()

        print("\n" + "=" * 60)
        print("  FEDERATED LEARNING CYCLE — FINAL SUMMARY")
        print("=" * 60)
        print(f"  Total rounds:          {len(h['round'])}")
        print(f"  Best accuracy:         {best_acc:.4f}  (round {best_round})")
        print(f"  Final accuracy:        {final_acc:.4f}")
        print(f"  Mean accuracy:         {np.mean(h['global_accuracy']):.4f}")
        print(f"  Total accepted:        {sum(h['num_accepted'])}")
        print(f"  Total rejected:        {sum(h['num_rejected'])}")
        print(f"  Reputation — mean:     {stats.get('mean_reputation', 0):.4f}")
        print(f"  Reputation — std:      {stats.get('std_reputation', 0):.4f}")

        kd_losses = [l for l in h["distillation_loss"] if l is not None]
        if kd_losses:
            print(f"  Avg distillation loss: {np.mean(kd_losses):.5f}")
        print(f"  TF Lite model saved:   {self.config.tflite_output_path}")
        print("=" * 60 + "\n")


# ====================================================================== #
#  5.  ENTRY POINT                                                        #
# ====================================================================== #

def main() -> None:
    """
    Main entry point — runs the full Enhanced Federated Learning Cycle
    for DeepFake detection.
    """
    np.random.seed(42)
    tf.random.set_seed(42)

    # ------------------------------------------------------------------ #
    #  Configuration                                                      #
    # ------------------------------------------------------------------ #
    config = FLCycleConfig(
        model_path="phase3_best.weights.h5",
        num_devices=100,
        local_epochs=5,
        global_rounds=50,
        clients_per_round=15,
        local_batch_size=32,
        local_lr=1e-4,
        eval_every=10,
        enable_distillation=True,
    )

    cycle = FederatedLearningCycle(config)

    # ------------------------------------------------------------------ #
    #  Load global model                                                  #
    # ------------------------------------------------------------------ #
    model = cycle.load_global_model()
    input_shape = model.input_shape[1:]        # strip batch dim
    config.input_shape = input_shape
    logger.info("Model input shape: %s", input_shape)

    # ------------------------------------------------------------------ #
    #  Prepare data  (synthetic — replace with real FF++ c23 loaders)     #
    # ------------------------------------------------------------------ #
    #
    #  In production, replace these synthetic generators with actual
    #  FF++ c23 data loaders:
    #
    #    train_data = load_ffpp_c23_train(...)      # for client partitions
    #    val_data   = load_ffpp_c23_val(...)        # server validation
    #    test_data  = load_ffpp_c23_test(...)       # independent test set
    #    proxy_data = load_ffpp_c23_unlabelled(...) # for distillation
    #
    logger.info("Generating synthetic data for %d clients …", config.num_devices)

    TOTAL_TRAIN_SAMPLES = config.num_devices * 10   # 10 samples/client
    VAL_SAMPLES   = 200
    TEST_SAMPLES  = 300
    PROXY_SAMPLES = 150

    train_data = _generate_synthetic_data(
        TOTAL_TRAIN_SAMPLES, input_shape, seed=1,
    )
    server_val_data = _generate_synthetic_data(
        VAL_SAMPLES, input_shape, seed=2,
    )
    test_data = _generate_synthetic_data(
        TEST_SAMPLES, input_shape, seed=3,
    )
    proxy_data = _generate_proxy_data(
        PROXY_SAMPLES, input_shape, seed=4,
    )
    supervised_data = _generate_synthetic_data(
        100, input_shape, seed=5,
    )

    # Partition training data across clients (IID)
    client_data = partition_data_iid(
        train_data, config.num_devices, seed=42,
    )

    # ------------------------------------------------------------------ #
    #  Create clients & wire components                                   #
    # ------------------------------------------------------------------ #
    cycle.create_clients(client_data)
    cycle.setup_components()

    # ------------------------------------------------------------------ #
    #  Run the Federated Learning Cycle                                   #
    # ------------------------------------------------------------------ #
    history = cycle.run(
        server_val_data=server_val_data,
        test_data=test_data,
        proxy_data=proxy_data,
        supervised_data=supervised_data,
    )

    logger.info("Federated Learning Cycle finished. History keys: %s",
                list(history.keys()))


# ====================================================================== #
#  DEMO / SMOKE-TEST  (lightweight — reduced params for quick run)        #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  Enhanced Federated Learning Cycle — Demo  ===\n")

    np.random.seed(42)
    tf.random.set_seed(42)

    # ---- Use a small synthetic model for smoke-testing --------------- #
    # (The real cycle uses phase3_best.weights.h5 via main())
    INPUT_DIM = 16
    demo_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(INPUT_DIM,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    demo_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    print(f"Demo model — {demo_model.count_params()} params, "
          f"input shape {demo_model.input_shape}")

    # ---- Configuration (reduced for speed) --------------------------- #
    demo_config = FLCycleConfig(
        model_path="(in-memory demo)",
        num_devices=8,
        local_epochs=1,
        global_rounds=3,
        clients_per_round=4,
        local_batch_size=16,
        local_lr=1e-3,
        eval_every=1,
        enable_distillation=True,
        distillation_config=DistillationConfig(
            temperature=2.0,
            lam=0.5,
            epochs=2,
            batch_size=16,
            learning_rate=1e-3,
        ),
    )

    # ---- Synthetic data ---------------------------------------------- #
    N_CLIENT = demo_config.num_devices
    SAMPLES_PER_CLIENT = 30
    TOTAL = N_CLIENT * SAMPLES_PER_CLIENT
    input_shape = (INPUT_DIM,)

    train_ds = _generate_synthetic_data(TOTAL, input_shape, seed=10)
    val_ds   = _generate_synthetic_data(100, input_shape, seed=20)
    test_ds  = _generate_synthetic_data(120, input_shape, seed=30)
    proxy_ds = _generate_proxy_data(80, input_shape, seed=40)
    sup_ds   = _generate_synthetic_data(60, input_shape, seed=50)

    client_data = partition_data_iid(train_ds, N_CLIENT, seed=42)

    # ---- Build cycle ------------------------------------------------- #
    cycle = FederatedLearningCycle(demo_config)
    cycle.global_model = demo_model         # skip load_model for demo
    cycle.create_clients(client_data)
    cycle.setup_components()

    # ---- Run --------------------------------------------------------- #
    history = cycle.run(
        server_val_data=val_ds,
        test_data=test_ds,
        proxy_data=proxy_ds,
        supervised_data=sup_ds,
    )

    # ---- Show history ------------------------------------------------ #
    print("\nRound-by-round accuracy:")
    for rnd, acc in zip(history["round"], history["global_accuracy"]):
        print(f"  Round {rnd}: {acc:.4f}")

    # ---- TF Lite check ----------------------------------------------- #
    tflite_path = demo_config.tflite_output_path
    if Path(tflite_path).exists():
        size_kb = Path(tflite_path).stat().st_size / 1024
        print(f"\nTF Lite model: {tflite_path} ({size_kb:.1f} KB)")
        # Clean up demo artifacts
        Path(tflite_path).unlink(missing_ok=True)
        q_path = tflite_path.replace(".tflite", "_quantised.tflite")
        Path(q_path).unlink(missing_ok=True)

    print("\nDone.")

Overwriting federated_learning_cycle.py


### TFF Wrapper — Federated Dataset Management


In [12]:
%%writefile tff_data_utils.py
"""
TFF Data Utilities — Federated Dataset Management
===================================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Bridges existing ``tf.data.Dataset`` client partitions into
TensorFlow Federated (TFF) compatible data structures.

Provides
--------
* ``TFFDataManager``  — element specs, federated-data creation,
  optional wrapping into ``tff.simulation.datasets.ClientData``.
* ``partition_data_iid_tff``  — IID partition helper.
* ``generate_synthetic_data`` / ``generate_proxy_data``  — quick
  generators for smoke-testing.

Environment
-----------
Requires ``tensorflow-federated >= 0.48.0``.
See ``requirements_tff.txt`` for the exact compatible stack.
Recommended runtime: **Google Colab** (TFF pre-installed).
"""

from __future__ import annotations

import logging
from typing import Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------- Conditional TFF import ------------------------------------- #
# Primary: use the Flower-backed adapter (works on Python 3.12 + TF 2.19).
# Fallback: real TFF if the adapter is unavailable.
try:
    from flwr_adapter import tff_compat as tff  # type: ignore[assignment]
    TFF_AVAILABLE = True
except ImportError:
    try:
        import tensorflow_federated as tff
        TFF_AVAILABLE = True
    except ImportError:
        tff = None  # type: ignore[assignment]
        TFF_AVAILABLE = False

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  Guard                                                                  #
# ====================================================================== #

def _require_tff() -> None:
    """Raise a clear error if neither the Flower adapter nor TFF is available."""
    if not TFF_AVAILABLE:
        raise RuntimeError(
            "Neither the Flower adapter (flwr_adapter) nor TensorFlow Federated\n"
            "is available in this environment.\n"
            "Install Flower with:  pip install flwr\n"
            "The flwr_adapter module provides a drop-in replacement for TFF.\n"
            "Alternatively, install TFF:  pip install tensorflow-federated==0.48.0"
        )


# ====================================================================== #
#  1.  TFF DATA MANAGER                                                   #
# ====================================================================== #

class TFFDataManager:
    """
    Manages federated data for TFF integration.

    Converts per-client ``tf.data.Dataset`` partitions into the formats
    expected by ``tff.learning.algorithms`` and provides the
    ``element_spec`` / ``input_spec`` required by
    ``tff.learning.models.from_keras_model()``.

    Parameters
    ----------
    input_shape : tuple[int, ...]
        Spatial input shape *without* the batch dimension,
        e.g. ``(224, 224, 3)`` for the EfficientNet model.
    """

    def __init__(self, input_shape: Tuple[int, ...]) -> None:
        self.input_shape = input_shape

    # ------------------------------------------------------------------ #
    #  Spec helpers                                                       #
    # ------------------------------------------------------------------ #

    def get_element_spec(self) -> Tuple[tf.TensorSpec, tf.TensorSpec]:
        """
        Return the **batched** element spec for ``from_keras_model()``.

        Matches datasets that yield ``(images, labels)`` with a leading
        ``None`` batch dimension.
        """
        return (
            tf.TensorSpec(shape=(None, *self.input_shape), dtype=tf.float32),
            tf.TensorSpec(shape=(None,), dtype=tf.float32),
        )

    def get_unbatched_spec(self) -> Tuple[tf.TensorSpec, tf.TensorSpec]:
        """Per-example spec (no batch dim) — useful for type annotations."""
        return (
            tf.TensorSpec(shape=self.input_shape, dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.float32),
        )

    # ------------------------------------------------------------------ #
    #  Federated data creation (for process.next())                       #
    # ------------------------------------------------------------------ #

    def make_federated_data(
        self,
        client_datasets: Dict[str, tf.data.Dataset],
        selected_ids: List[str],
        batch_size: int = 32,
        local_epochs: int = 1,
        shuffle_buffer: int = 1000,
    ) -> List[tf.data.Dataset]:
        """
        Create a **list of batched datasets** for TFF's
        ``process.next(state, federated_data)``.

        Parameters
        ----------
        client_datasets : dict[str, tf.data.Dataset]
            Pre-partitioned datasets keyed by client ID.
        selected_ids : list[str]
            Client IDs chosen for this round (from Part 1).
        batch_size : int
            Batch size for local training.
        local_epochs : int
            Number of local training epochs — implemented via
            ``dataset.repeat(local_epochs)`` which is the standard
            TFF pattern.
        shuffle_buffer : int
            Buffer size for per-epoch shuffling.

        Returns
        -------
        list[tf.data.Dataset]
            One **batched** dataset per selected client.
        """
        federated: List[tf.data.Dataset] = []
        for cid in selected_ids:
            if cid not in client_datasets:
                logger.warning("Client %s has no dataset — skipping.", cid)
                continue
            ds = (
                client_datasets[cid]
                .repeat(local_epochs)
                .shuffle(buffer_size=shuffle_buffer)
                .batch(batch_size)
                .prefetch(tf.data.AUTOTUNE)
            )
            federated.append(ds)
        return federated

    # ------------------------------------------------------------------ #
    #  TFF ClientData wrapper (optional — for TFF simulation tools)       #
    # ------------------------------------------------------------------ #

    def create_tff_client_data(
        self,
        client_datasets: Dict[str, tf.data.Dataset],
    ):
        """
        Wrap per-client datasets into ``tff.simulation.datasets.ClientData``
        for advanced TFF simulation tools (e.g. ``ClientData.preprocess``,
        sampling utilities, etc.).

        Returns
        -------
        tff.simulation.datasets.ClientData
        """
        _require_tff()

        client_ids = sorted(client_datasets.keys())
        local_ref = client_datasets  # captured by closure

        def create_dataset_fn(client_id):
            cid = (
                client_id.numpy().decode("utf-8")
                if isinstance(client_id, tf.Tensor)
                else client_id
            )
            return local_ref[cid]

        return tff.simulation.datasets.ClientData.from_clients_and_tf_fn(
            client_ids=client_ids,
            serializable_dataset_fn=create_dataset_fn,
        )

    # ------------------------------------------------------------------ #
    #  Preprocessing pipeline                                             #
    # ------------------------------------------------------------------ #

    @staticmethod
    def preprocess_dataset(
        dataset: tf.data.Dataset,
        batch_size: int = 32,
        local_epochs: int = 1,
        shuffle_buffer: int = 1000,
    ) -> tf.data.Dataset:
        """
        Standard preprocessing pipeline applied to each client dataset
        before passing to TFF.
        """
        return (
            dataset
            .repeat(local_epochs)
            .shuffle(buffer_size=shuffle_buffer)
            .batch(batch_size)
            .prefetch(tf.data.AUTOTUNE)
        )


# ====================================================================== #
#  2.  PARTITIONING HELPERS                                               #
# ====================================================================== #

def partition_data_iid_tff(
    full_dataset: tf.data.Dataset,
    num_clients: int,
    seed: int = 42,
) -> Dict[str, tf.data.Dataset]:
    """
    IID partition: shuffle the dataset and split evenly across clients.

    Each shard is a ``tf.data.Dataset`` yielding ``(image, label)`` pairs.

    Parameters
    ----------
    full_dataset : tf.data.Dataset
        The complete labelled dataset.
    num_clients : int
        Number of federated client partitions.
    seed : int
        Random seed for reproducible shuffling.

    Returns
    -------
    dict[str, tf.data.Dataset]
        ``{client_id: local_dataset}``
    """
    all_data = list(full_dataset.shuffle(buffer_size=10_000, seed=seed))
    total = len(all_data)
    shard_size = max(1, total // num_clients)

    partitions: Dict[str, tf.data.Dataset] = {}
    for i in range(num_clients):
        cid = f"client_{i:03d}"
        start = i * shard_size
        end = min(start + shard_size, total)
        if start >= total:
            start = start % total
            end = start + shard_size

        shard_x = [elem[0].numpy() for elem in all_data[start:end]]
        shard_y = [elem[1].numpy() for elem in all_data[start:end]]

        if not shard_x:
            shard_x = [all_data[0][0].numpy()]
            shard_y = [all_data[0][1].numpy()]

        partitions[cid] = tf.data.Dataset.from_tensor_slices(
            (np.stack(shard_x), np.array(shard_y))
        )
    return partitions


# ====================================================================== #
#  3.  SYNTHETIC DATA GENERATORS  (for smoke-testing)                     #
# ====================================================================== #

def generate_synthetic_data(
    num_samples: int,
    input_shape: Tuple[int, ...],
    seed: Optional[int] = None,
) -> tf.data.Dataset:
    """
    Synthetic labelled dataset ``(image, label)`` for smoke-testing.

    Replace with real FF++ c23 data loaders in production.
    """
    rng = np.random.RandomState(seed)
    x = rng.randn(num_samples, *input_shape).astype(np.float32) * 0.1
    y = rng.randint(0, 2, size=(num_samples,)).astype(np.float32)
    return tf.data.Dataset.from_tensor_slices((x, y))


def generate_proxy_data(
    num_samples: int,
    input_shape: Tuple[int, ...],
    seed: Optional[int] = None,
) -> tf.data.Dataset:
    """Unlabelled proxy data ``(image,)`` for knowledge distillation."""
    rng = np.random.RandomState(seed)
    x = rng.randn(num_samples, *input_shape).astype(np.float32) * 0.1
    return tf.data.Dataset.from_tensor_slices(x)


# ====================================================================== #
#  DEMO / SMOKE-TEST                                                      #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  TFF Data Utilities — Demo  ===\n")

    INPUT_SHAPE = (16,)          # tiny for fast demo
    NUM_CLIENTS = 8
    SAMPLES = NUM_CLIENTS * 30   # 30 per client

    # ---- 1. Synthetic dataset ---------------------------------------- #
    full_ds = generate_synthetic_data(SAMPLES, INPUT_SHAPE, seed=10)
    print(f"Full dataset: {SAMPLES} samples, shape {INPUT_SHAPE}")

    # ---- 2. Partition ------------------------------------------------ #
    client_data = partition_data_iid_tff(full_ds, NUM_CLIENTS)
    for cid, ds in sorted(client_data.items()):
        n = sum(1 for _ in ds)
        print(f"  {cid}: {n} samples")

    # ---- 3. TFF data manager ---------------------------------------- #
    dm = TFFDataManager(input_shape=INPUT_SHAPE)

    print(f"\nElement spec (batched): {dm.get_element_spec()}")
    print(f"Unbatched spec:        {dm.get_unbatched_spec()}")

    # ---- 4. Make federated data for a round -------------------------- #
    selected = ["client_001", "client_003", "client_005"]
    fed_data = dm.make_federated_data(
        client_data, selected, batch_size=16, local_epochs=2,
    )
    print(f"\nFederated data for {selected}: {len(fed_data)} datasets")
    for i, ds in enumerate(fed_data):
        n_batches = sum(1 for _ in ds)
        print(f"  Client {selected[i]}: {n_batches} batches (2 epochs)")

    # ---- 5. TFF ClientData (requires TFF) ---------------------------- #
    if TFF_AVAILABLE:
        tff_cd = dm.create_tff_client_data(client_data)
        print(f"\nTFF ClientData: {len(tff_cd.client_ids)} clients")
        print(f"  Client IDs: {tff_cd.client_ids[:5]} ...")
    else:
        print(
            "\n⚠  TFF not installed — skipping ClientData creation.\n"
            "   Install: pip install tensorflow-federated==0.48.0\n"
            "   See requirements_tff.txt for full compatible stack.\n"
            "   Recommended runtime: Google Colab."
        )



    # ---- 6. Proxy data ----------------------------------------------- #    print("\nDone.")

    proxy = generate_proxy_data(50, INPUT_SHAPE, seed=20)
    print(f"\nProxy data: 50 unlabelled samples, shape {INPUT_SHAPE}")

Overwriting tff_data_utils.py


### TFF Wrapper — Model Wrapping & Learning Process


In [13]:
%%writefile tff_learning_process.py
"""
TFF Learning Process — Model Wrapping & Federated Training
===========================================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Wraps the EfficientNet Keras model into TFF's learning framework and
builds a customised Federated Averaging learning process.

Provides
--------
* ``TFFModelFactory``    — creates the ``model_fn`` callable required by
  ``tff.learning.algorithms.build_weighted_fed_avg()``.
* ``build_tff_learning_process``  — builds the TFF ``LearningProcess``.
* ``TFFRoundExecutor``   — executes TFF rounds and converts weights
  between TFF ``ModelWeights`` and Keras ``model.get_weights()`` format.
* Weight-conversion utilities: ``tff_weights_to_keras``,
  ``keras_weights_to_tff``.

Architecture
------------
TFF handles the federated computation (model broadcast → client-side
local training → data-weighted aggregation a.k.a. FedAvg).  Our custom
enhancements (contribution weighting, distillation, reputation) operate
as a **post-aggregation refinement** in the outer Python loop.

Environment
-----------
Requires ``tensorflow-federated >= 0.48.0``.
See ``requirements_tff.txt`` and ``tff_data_utils.py`` for details.
"""

from __future__ import annotations

import logging
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------- Conditional TFF import ------------------------------------- #
# Primary: use the Flower-backed adapter (works on Python 3.12 + TF 2.19).
# Fallback: real TFF if the adapter is unavailable.
try:
    from flwr_adapter import tff_compat as tff  # type: ignore[assignment]
    TFF_AVAILABLE = True
except ImportError:
    try:
        import tensorflow_federated as tff
        TFF_AVAILABLE = True
    except ImportError:
        tff = None  # type: ignore[assignment]
        TFF_AVAILABLE = False

from tff_data_utils import _require_tff

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  MODEL FACTORY                                                      #
# ====================================================================== #

class TFFModelFactory:
    """
    Creates the ``model_fn`` callable that TFF's learning algorithms
    require.

    Each call to ``model_fn()`` must return a **fresh** TFF-wrapped
    Keras model instance with the same architecture.  TFF traces the
    computation graph on first call and reuses it, so deterministic
    architecture is critical.

    Parameters
    ----------
    keras_model : tf.keras.Model
        Reference Keras model whose architecture will be cloned.
        Must **not** be compiled — TFF handles compilation internally.
    input_spec : tuple[tf.TensorSpec, tf.TensorSpec]
        Batched element spec ``(x_spec, y_spec)`` matching the dataset
        structure.  Obtain via ``TFFDataManager.get_element_spec()``.
    loss : tf.keras.losses.Loss | None
        Loss function; defaults to ``BinaryCrossentropy(from_logits=False)``.
    metrics : list[tf.keras.metrics.Metric] | None
        Metrics; defaults to ``[BinaryAccuracy()]``.
    """

    def __init__(
        self,
        keras_model: tf.keras.Model,
        input_spec: Tuple[tf.TensorSpec, tf.TensorSpec],
        loss: Optional[tf.keras.losses.Loss] = None,
        metrics: Optional[list] = None,
    ) -> None:
        self._ref_model = keras_model
        self._input_spec = input_spec
        self._loss = loss or tf.keras.losses.BinaryCrossentropy()
        self._metrics = metrics or [tf.keras.metrics.BinaryAccuracy()]

    # ------------------------------------------------------------------ #
    #  model_fn builder                                                   #
    # ------------------------------------------------------------------ #

    def create_model_fn(self) -> Callable[[], Any]:
        """
        Return a **no-args callable** compatible with
        ``tff.learning.algorithms.build_weighted_fed_avg(model_fn=...)``.

        The closure captures the reference model, spec, loss, and metrics
        so that every invocation produces an architecturally identical
        (but freshly initialised) TFF model.
        """
        _require_tff()

        ref = self._ref_model
        spec = self._input_spec
        loss = self._loss
        metrics_list = self._metrics

        def model_fn():
            # Clone architecture (fresh random weights — TFF will inject
            # the server weights before each client round).
            keras_clone = tf.keras.models.clone_model(ref)
            keras_clone.build(ref.input_shape)
            return tff.learning.models.from_keras_model(
                keras_model=keras_clone,
                input_spec=spec,
                loss=loss,
                metrics=metrics_list,
            )

        return model_fn


# ====================================================================== #
#  2.  WEIGHT CONVERSION  (TFF ↔ Keras)                                  #
# ====================================================================== #

def tff_weights_to_keras(
    model_weights,
    keras_model: tf.keras.Model,
) -> None:
    """
    Copy TFF ``ModelWeights`` into a Keras model's variables.

    Uses TFF's built-in ``assign_weights_to`` when available; falls back
    to manual assignment otherwise.

    Parameters
    ----------
    model_weights : tff.learning.models.ModelWeights
        ``ModelWeights(trainable=[...], non_trainable=[...])``.
    keras_model : tf.keras.Model
        Target Keras model — must share the same architecture.
    """
    _require_tff()

    try:
        # Preferred: TFF's built-in helper
        model_weights.assign_weights_to(keras_model)
    except AttributeError:
        # Fallback: manual assignment
        trainable = list(model_weights.trainable)
        non_trainable = list(model_weights.non_trainable)
        for var, val in zip(keras_model.trainable_variables, trainable):
            var.assign(val)
        for var, val in zip(keras_model.non_trainable_variables, non_trainable):
            var.assign(val)
    logger.debug("TFF → Keras weight transfer complete.")


def keras_weights_to_tff(
    keras_model: tf.keras.Model,
):
    """
    Convert Keras model weights to TFF ``ModelWeights``.

    Returns
    -------
    tff.learning.models.ModelWeights
    """
    _require_tff()

    trainable = [v.numpy() for v in keras_model.trainable_variables]
    non_trainable = [v.numpy() for v in keras_model.non_trainable_variables]
    return tff.learning.models.ModelWeights(
        trainable=trainable,
        non_trainable=non_trainable,
    )


# ====================================================================== #
#  3.  LEARNING-PROCESS BUILDER                                           #
# ====================================================================== #

@dataclass
class TFFProcessConfig:
    """Hyper-parameters for the TFF weighted-FedAvg process."""
    client_lr: float = 1e-4
    server_lr: float = 0.1
    client_optimizer: str = "adam"     # "adam" | "sgd"
    server_optimizer: str = "sgd"     # typically SGD with lr=1.0


def build_tff_learning_process(
    model_fn: Callable[[], Any],
    config: Optional[TFFProcessConfig] = None,
):
    """
    Build a TFF ``LearningProcess`` using weighted Federated Averaging.

    This wraps ``tff.learning.algorithms.build_weighted_fed_avg`` and
    returns a process with the following methods:

    * ``initialize()`` → initial server state
    * ``next(state, federated_data)`` → ``LearningProcessOutput``
    * ``get_model_weights(state)`` → ``ModelWeights``
    * ``set_model_weights(state, weights)`` → updated state

    Parameters
    ----------
    model_fn : callable
        No-args callable that returns a ``tff.learning.models.VariableModel``.
        Obtain from ``TFFModelFactory.create_model_fn()``.
    config : TFFProcessConfig | None
        Optimiser and learning-rate settings.

    Returns
    -------
    tff.learning.templates.LearningProcess
    """
    _require_tff()
    config = config or TFFProcessConfig()

    def client_optimizer_fn():
        if config.client_optimizer == "adam":
            return tf.keras.optimizers.Adam(config.client_lr)
        return tf.keras.optimizers.SGD(config.client_lr)

    def server_optimizer_fn():
        if config.server_optimizer == "adam":
            return tf.keras.optimizers.Adam(config.server_lr)
        return tf.keras.optimizers.SGD(config.server_lr)

    process = tff.learning.algorithms.build_weighted_fed_avg(
        model_fn=model_fn,
        client_optimizer_fn=client_optimizer_fn,
        server_optimizer_fn=server_optimizer_fn,
    )

    logger.info(
        "TFF LearningProcess built — client_opt=%s(lr=%g), "
        "server_opt=%s(lr=%g).",
        config.client_optimizer, config.client_lr,
        config.server_optimizer, config.server_lr,
    )
    return process


# ====================================================================== #
#  4.  ROUND EXECUTOR                                                     #
# ====================================================================== #

class TFFRoundExecutor:
    """
    Thin wrapper around a TFF ``LearningProcess`` that handles state
    management and weight conversion.

    Typical workflow
    ----------------
    >>> executor = TFFRoundExecutor(process, keras_model)
    >>> executor.initialize()
    >>> executor.inject_pretrained_weights()     # start from .h5 model
    >>> for t in range(num_rounds):
    ...     metrics = executor.execute_round(federated_data)
    ...     keras_weights = executor.get_keras_weights()
    ...     # ... apply enhancements ...
    ...     executor.set_keras_weights(enhanced_weights)

    Parameters
    ----------
    process : tff.learning.templates.LearningProcess
        TFF learning process (from ``build_tff_learning_process``).
    keras_model : tf.keras.Model
        Reference Keras model — used for weight conversion only.
    """

    def __init__(
        self,
        process,                        # tff.learning.templates.LearningProcess
        keras_model: tf.keras.Model,
    ) -> None:
        _require_tff()
        self.process = process
        self.keras_model = keras_model
        self.state = None
        self._round_count = 0

    # ------------------------------------------------------------------ #
    #  Initialisation                                                     #
    # ------------------------------------------------------------------ #

    def initialize(self) -> None:
        """
        Call TFF's ``process.initialize()`` to create the initial server
        state (with random model weights).
        """
        self.state = self.process.initialize()
        self._round_count = 0
        logger.info("TFF process initialised (random server weights).")

    def inject_pretrained_weights(self) -> None:
        """
        Inject the weights from ``self.keras_model`` into the TFF server
        state.  Call this after ``initialize()`` to start federated
        training from a pre-trained checkpoint (e.g. EfficientNet).
        """
        assert self.state is not None, "Call initialize() first."
        tff_weights = keras_weights_to_tff(self.keras_model)
        self.state = self.process.set_model_weights(self.state, tff_weights)
        logger.info("Pre-trained Keras weights injected into TFF state.")

    # ------------------------------------------------------------------ #
    #  Round execution                                                    #
    # ------------------------------------------------------------------ #

    def execute_round(
        self,
        federated_data: List[tf.data.Dataset],
    ) -> Dict[str, Any]:
        """
        Execute one TFF federated round (broadcast → local training →
        FedAvg aggregation).

        Parameters
        ----------
        federated_data : list[tf.data.Dataset]
            Batched datasets for the selected clients.  Obtain from
            ``TFFDataManager.make_federated_data()``.

        Returns
        -------
        metrics : dict
            TFF round metrics (client loss, accuracy, num_examples, …).
        """
        assert self.state is not None, "Call initialize() first."

        result = self.process.next(self.state, federated_data)
        self.state = result.state
        self._round_count += 1

        # Extract metrics — TFF returns nested OrderedDicts
        metrics = _flatten_tff_metrics(result.metrics)
        logger.info(
            "TFF round %d complete — %s",
            self._round_count, _format_metrics(metrics),
        )
        return metrics

    # ------------------------------------------------------------------ #
    #  Weight access                                                      #
    # ------------------------------------------------------------------ #

    def get_keras_weights(self) -> List[np.ndarray]:
        """
        Extract model weights from the TFF state and set them on the
        Keras reference model.

        Returns the weights as a list of numpy arrays (same format
        as ``keras_model.get_weights()``).
        """
        model_weights = self.process.get_model_weights(self.state)
        tff_weights_to_keras(model_weights, self.keras_model)
        return self.keras_model.get_weights()

    def set_keras_weights(self, weights: List[np.ndarray]) -> None:
        """
        Inject (possibly enhanced) Keras weights back into the TFF
        server state so the next round broadcasts the updated model.
        """
        self.keras_model.set_weights(weights)
        tff_weights = keras_weights_to_tff(self.keras_model)
        self.state = self.process.set_model_weights(self.state, tff_weights)
        logger.debug("Enhanced weights injected into TFF state.")

    def get_tff_model_weights(self):
        """Return the raw TFF ``ModelWeights`` from the current state."""
        return self.process.get_model_weights(self.state)


# ====================================================================== #
#  5.  METRIC HELPERS                                                     #
# ====================================================================== #

def _flatten_tff_metrics(metrics_struct) -> Dict[str, float]:
    """
    Flatten TFF's nested ``OrderedDict`` metrics into a simple dict.

    TFF returns metrics like::

        OrderedDict([
            ('distributor', OrderedDict()),
            ('client_work', OrderedDict([
                ('train', OrderedDict([('loss', 0.693), ...])),
            ])),
            ...
        ])

    This flattens all leaf values into ``{'client_work/train/loss': 0.693}``.
    """
    flat: Dict[str, float] = {}

    def _walk(obj, prefix: str = "") -> None:
        if isinstance(obj, dict):
            for k, v in obj.items():
                _walk(v, f"{prefix}{k}/")
        elif hasattr(obj, "_asdict"):
            _walk(obj._asdict(), prefix)
        elif hasattr(obj, "items"):
            for k, v in obj.items():
                _walk(v, f"{prefix}{k}/")
        else:
            key = prefix.rstrip("/")
            try:
                flat[key] = float(obj)
            except (TypeError, ValueError):
                flat[key] = str(obj)

    try:
        _walk(metrics_struct)
    except Exception:
        flat["raw"] = str(metrics_struct)
    return flat


def _format_metrics(metrics: Dict[str, float], max_items: int = 5) -> str:
    """Compact single-line metrics string for logging."""
    items = list(metrics.items())[:max_items]
    return ", ".join(f"{k}={v:.4f}" if isinstance(v, float) else f"{k}={v}"
                     for k, v in items)


# ====================================================================== #
#  DEMO / SMOKE-TEST                                                      #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  TFF Learning Process — Demo  ===\n")

    np.random.seed(42)
    tf.random.set_seed(42)

    # ---- 1. Build a tiny reference model ----------------------------- #
    INPUT_DIM = 16
    ref_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(INPUT_DIM,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    # Do NOT compile — TFF handles compilation internally
    print(f"Reference model: {ref_model.count_params()} params, "
          f"input shape {ref_model.input_shape}")

    # ---- 2. Element spec --------------------------------------------- #
    from tff_data_utils import TFFDataManager, generate_synthetic_data, partition_data_iid_tff

    dm = TFFDataManager(input_shape=(INPUT_DIM,))
    input_spec = dm.get_element_spec()
    print(f"Input spec: {input_spec}")

    # ---- 3. Model factory -------------------------------------------- #
    factory = TFFModelFactory(
        keras_model=ref_model,
        input_spec=input_spec,
    )
    print("TFFModelFactory created.")

    # ---- 4. TFF-specific tests --------------------------------------- #
    if TFF_AVAILABLE:
        print("\n--- TFF available — running full demo ---\n")

        model_fn = factory.create_model_fn()
        print(f"model_fn created: {model_fn}")

        # Build learning process
        process = build_tff_learning_process(
            model_fn=model_fn,
            config=TFFProcessConfig(client_lr=1e-3, server_lr=0.1),
        )
        print("Learning process built.")

        # Create round executor
        executor = TFFRoundExecutor(process, ref_model)
        executor.initialize()
        executor.inject_pretrained_weights()
        print("Executor initialised with pre-trained weights.")

        # Synthesise federated data
        full_ds = generate_synthetic_data(240, (INPUT_DIM,), seed=10)
        client_data = partition_data_iid_tff(full_ds, 8)
        selected = ["client_001", "client_003", "client_005"]
        fed_data = dm.make_federated_data(
            client_data, selected, batch_size=16, local_epochs=2,
        )

        # Run 3 TFF rounds
        for rnd in range(1, 4):
            metrics = executor.execute_round(fed_data)
            keras_w = executor.get_keras_weights()
            print(f"  Round {rnd}: {len(keras_w)} weight arrays, "
                  f"metrics keys: {list(metrics.keys())[:4]}")

            # Simulate enhancement: perturb weights slightly
            enhanced = [w + np.random.randn(*w.shape).astype(np.float32) * 0.001
                        for w in keras_w]
            executor.set_keras_weights(enhanced)

        print("\nTFF demo complete.")

    else:
        print(
            "\n⚠  TFF not installed — running structural checks only.\n"
            "   Install: pip install tensorflow-federated==0.48.0\n"
            "   See requirements_tff.txt for the full stack.\n"
        )

        # Structural checks
        print("✓ TFFModelFactory instantiable (model_fn creation requires TFF)")
        print("✓ TFFProcessConfig:", TFFProcessConfig())

        try:
            _ = factory.create_model_fn()
        except RuntimeError as e:
            print(f"✓ model_fn correctly raises: {e.__class__.__name__}")

        # Weight conversion type check (without TFF, just show shapes)
        w = ref_model.get_weights()
        tr = [v.numpy() for v in ref_model.trainable_variables]
        ntr = [v.numpy() for v in ref_model.non_trainable_variables]
        print(f"✓ Keras → TFF mapping: {len(tr)} trainable, "

              f"{len(ntr)} non-trainable arrays")

    print("\nDone.")

    print("\nStructural checks passed.")

Overwriting tff_learning_process.py


### TFF Wrapper — Main TFF-Based Orchestrator


In [14]:
%%writefile tff_federated_cycle.py
"""
TFF Federated Learning Cycle — Main Orchestrator
==================================================
Part of: Enhanced Federated Learning Cycle for DeepFake Detection (Thesis)

Integrates **TensorFlow Federated** with all five enhancement modules
into one end-to-end pipeline:

 1. **Enhanced Client Selection**   (``enhanced_client_selection.py``)
 2. **Update Validation & Weighing** (``update_validation.py``)
 3. **Knowledge Distillation**      (``knowledge_distillation.py``)
 4. **Client Reputation Ledger**    (``client_reputation_ledger.py``)
 5. **Evaluation Metrics**          (``evaluation_metrics.py``)

Architecture
------------
TFF handles the **core federated computation**: model broadcasting,
client-side local training, and data-weighted Federated Averaging.
Our thesis enhancements operate as a **post-aggregation refinement
layer** that runs in the outer Python loop after each TFF round.

Per-round pipeline
~~~~~~~~~~~~~~~~~~
1. **Client selection** (Part 1) — multi-criteria scoring to choose
   which clients participate.  Selected client IDs determine which
   data is passed to TFF's ``process.next()``.
2. **TFF round** — ``process.next(state, federated_data)`` performs
   broadcasting, local training (``local_epochs`` via dataset repeat),
   and weighted FedAvg aggregation.
3. **Per-client analysis** — local training is re-run *outside TFF* on
   the selected clients so that per-client model weights are available
   for contribution scoring, distillation, and validation.
4. **Update validation & contribution-weighted aggregation** (Part 2) —
   re-aggregate using contribution weights ``c_i`` (which account for
   validation gain, similarity, data volume, and reputation).  This
   replaces TFF's data-volume-only FedAvg result.
5. **Knowledge distillation** (Part 3) — refine the server model by
   distilling knowledge from the contribution-weighted client ensemble.
6. **Reputation ledger update** (Part 4) — feed validation gains and
   contribution scores into the persistent ledger.
7. **Evaluation & reporting** (Part 5) — periodic full evaluation with
   JSON + text reports.
8. **Inject enhanced weights** back into the TFF server state for the
   next round.

Comparison mode
~~~~~~~~~~~~~~~
When ``enable_comparison=True`` the cycle logs **both** TFF's standard
FedAvg accuracy and our enhanced accuracy each round, providing a
direct side-by-side comparison for the thesis.

Configuration
~~~~~~~~~~~~~
- **Model**: ``phase3_best.weights.h5`` (EfficientNet, binary)
- **Devices**: 100 simulated clients
- **Local epochs**: 5 per round
- **Global rounds**: 50
- **Frameworks**: TensorFlow Federated + TF Lite

Environment
-----------
Requires ``tensorflow-federated >= 0.48.0``.
See ``requirements_tff.txt`` for the exact compatible stack.
Recommended runtime: **Google Colab**.
"""

from __future__ import annotations

import logging
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import tensorflow as tf

# ---------- Conditional TFF import ------------------------------------- #
# Primary: use the Flower-backed adapter (works on Python 3.12 + TF 2.19).
# Fallback: real TFF if the adapter is unavailable.
try:
    from flwr_adapter import tff_compat as tff  # type: ignore[assignment]
    TFF_AVAILABLE = True
except ImportError:
    try:
        import tensorflow_federated as tff
        TFF_AVAILABLE = True
    except ImportError:
        tff = None  # type: ignore[assignment]
        TFF_AVAILABLE = False

# ---------- Existing modules  (Parts 1–5) ----------------------------- #
from enhanced_client_selection import (          # Part 1
    ClientMetrics,
    FederatedClient,
    ReputationLedger,
    SelectionWeights,
    EnhancedClientSelector,
)
from update_validation import (                  # Part 2
    ContributionWeights,
    ClippingConfig,
    ClientUpdateRecord,
    UpdateValidator,
)
from knowledge_distillation import (             # Part 3
    DistillationConfig,
    run_distillation_round,
)
from client_reputation_ledger import (           # Part 4
    ReputationConfig,
    ClientReputationLedger,
    update_ledger_from_records,
)
from evaluation_metrics import (                 # Part 5
    FederatedModelEvaluator,
    evaluate_and_report,
)

# ---------- TFF wrappers ---------------------------------------------- #
from tff_data_utils import (
    TFFDataManager,
    _require_tff,
    partition_data_iid_tff,
    generate_synthetic_data,
    generate_proxy_data,
)
from tff_learning_process import (
    TFFModelFactory,
    TFFProcessConfig,
    build_tff_learning_process,
    TFFRoundExecutor,
    keras_weights_to_tff,
    tff_weights_to_keras,
)

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
)
logger = logging.getLogger(__name__)


# ====================================================================== #
#  1.  CONFIGURATION                                                      #
# ====================================================================== #

@dataclass
class TFFCycleConfig:
    """
    Central configuration for the TFF-based Federated Learning cycle.
    """
    # -- Core FL settings ---------------------------------------------- #
    model_path: str = "phase3_best.weights.h5"
    num_devices: int = 100
    local_epochs: int = 5
    global_rounds: int = 50
    clients_per_round: int = 15
    local_batch_size: int = 32
    local_lr: float = 1e-4
    server_lr: float = 0.1
    eval_every: int = 5

    # -- TFF process settings ------------------------------------------ #
    client_optimizer: str = "adam"
    server_optimizer: str = "sgd"

    # -- Comparison mode ----------------------------------------------- #
    enable_comparison: bool = True     # Log TFF FedAvg vs enhanced

    # -- Distillation (Part 3) ---------------------------------------- #
    enable_distillation: bool = True
    distillation_config: DistillationConfig = field(
        default_factory=lambda: DistillationConfig(
            temperature=2.0,
            lam=0.5,
            epochs=3,
            batch_size=32,
            learning_rate=1e-4,
        )
    )

    # -- Client selection (Part 1) ------------------------------------- #
    selection_weights: SelectionWeights = field(
        default_factory=lambda: SelectionWeights(
            w_v=0.30,
            w_d=0.20,
            w_l=0.10,
            w_r=0.25,
            w_s=0.15,
        )
    )

    # -- Update validation (Part 2) ----------------------------------- #
    contribution_weights: ContributionWeights = field(
        default_factory=lambda: ContributionWeights(
            alpha=0.35,
            beta=0.20,
            gamma=0.20,
            delta=0.25,
        )
    )
    clipping_config: ClippingConfig = field(
        default_factory=lambda: ClippingConfig(
            clip_threshold=10.0,
            clip_value=5.0,
        )
    )
    harmful_threshold: float = 0.02

    # -- Reputation (Part 4) ------------------------------------------ #
    reputation_config: ReputationConfig = field(
        default_factory=lambda: ReputationConfig(
            theta=0.0,
            gamma=0.10,
            decay_rate=0.99,
            floor=0.05,
            ceiling=1.0,
            initial_reputation=0.50,
            penalty_factor=0.05,
        )
    )

    # -- Evaluation & output (Part 5) --------------------------------- #
    reports_dir: str = "reports"
    tflite_output_path: str = "effnet_global_tff_final.tflite"
    input_shape: Tuple[int, ...] = (224, 224, 3)


# ====================================================================== #
#  2.  TF LITE CONVERSION                                                 #
# ====================================================================== #


def convert_to_tflite(
    model: tf.keras.Model,
    output_path: str,
    quantise: bool = False,
) -> str:
    """Convert a Keras model to TF Lite format."""
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    if quantise:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_bytes = converter.convert()
    Path(output_path).write_bytes(tflite_bytes)
    size_mb = len(tflite_bytes) / (1024 * 1024)
    logger.info(
        "TF Lite model saved → %s  (%.2f MB, quantised=%s)",
        output_path, size_mb, quantise,
    )
    return output_path


# ====================================================================== #
#  3.  TFF FEDERATED LEARNING CYCLE                                       #
# ====================================================================== #


class TFFFederatedLearningCycle:
    """
    End-to-end Federated Learning cycle using TFF,
    integrating all five enhancement modules.

    See module docstring for the detailed per-round pipeline.
    """

    def __init__(self, config: Optional[TFFCycleConfig] = None) -> None:
        self.config = config or TFFCycleConfig()
        self.global_model: Optional[tf.keras.Model] = None
        self.clients: List[FederatedClient] = []
        self.client_datasets: Dict[str, tf.data.Dataset] = {}

        # TFF components
        self.data_manager: Optional[TFFDataManager] = None
        self.tff_executor: Optional[TFFRoundExecutor] = None

        # Cached reusable models (avoid clone+build+compile per call)
        self._local_model: Optional[tf.keras.Model] = None
        self._comparison_model: Optional[tf.keras.Model] = None

        # Enhancement components (Parts 1–5)
        self.reputation_ledger: Optional[ClientReputationLedger] = None
        self.basic_ledger: Optional[ReputationLedger] = None
        self.selector: Optional[EnhancedClientSelector] = None
        self.validator: Optional[UpdateValidator] = None
        self.evaluator: Optional[FederatedModelEvaluator] = None

        # History
        self.history: Dict[str, list] = {
            "round": [],
            "tff_fedavg_accuracy": [],
            "enhanced_accuracy": [],
            "selected_clients": [],
            "num_accepted": [],
            "num_rejected": [],
            "distillation_loss": [],
        }

    # ------------------------------------------------------------------ #
    #  Initialisation                                                     #
    # ------------------------------------------------------------------ #

    def load_global_model(self) -> tf.keras.Model:
        """Load the pre-trained EfficientNet model."""
        import shutil, h5py
        cfg = self.config
        logger.info("Loading global model from %s …", cfg.model_path)
        from tensorflow.keras.applications.efficientnet import (
            preprocess_input as _effnet_preprocess,
        )
        from tensorflow.keras.applications import EfficientNetB2

        def _build_model(input_shape=(260, 260, 3)):
            base = EfficientNetB2(include_top=False, weights=None, input_shape=input_shape)
            x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
            x = tf.keras.layers.Dropout(0.3)(x)
            x = tf.keras.layers.Dense(1, activation="sigmoid")(x)
            return tf.keras.Model(inputs=base.input, outputs=x)

        model = None
        model_path = cfg.model_path

        # Inspect HDF5 to determine if it is a full model or weights-only
        with h5py.File(model_path, "r") as f:
            top_keys = list(f.keys())
        is_full_model = any(k in top_keys for k in ("model_weights", "model_config", "keras_version"))

        # Strategy 1: full saved model (rename away from .weights.h5 if needed)
        if is_full_model:
            load_path = model_path
            tmp_path = None
            if model_path.endswith(".weights.h5"):
                tmp_path = model_path.replace(".weights.h5", "_full_tmp.h5")
                shutil.copy(model_path, tmp_path)
                load_path = tmp_path
            try:
                _custom = {"preprocess_input": _effnet_preprocess}
                model = tf.keras.models.load_model(load_path, custom_objects=_custom, compile=False)
                logger.info("Loaded as full saved model.")
            except Exception as e:
                logger.warning("load_model() failed: %s", e)
            finally:
                if tmp_path and os.path.exists(tmp_path):
                    os.remove(tmp_path)

        # Strategy 2: weights-only (Keras 3 .weights.h5 format)
        if model is None:
            model = _build_model()
            try:
                model.load_weights(model_path)
                logger.info("Weights loaded (Keras 3 format).")
            except Exception:
                try:
                    model.load_weights(model_path, skip_mismatch=True)
                    logger.info("Weights loaded with skip_mismatch=True.")
                except Exception:
                    model = None

        # Strategy 3: legacy HDF5 weights (rename to .h5)
        if model is None and model_path.endswith(".weights.h5"):
            model = _build_model()
            legacy_path = model_path.replace(".weights.h5", "_legacy_tmp.h5")
            shutil.copy(model_path, legacy_path)
            try:
                model.load_weights(legacy_path)
                logger.info("Weights loaded via legacy HDF5.")
            except Exception:
                try:
                    model.load_weights(legacy_path, skip_mismatch=True)
                    logger.info("Legacy weights loaded with skip_mismatch=True.")
                except Exception:
                    model = None
            finally:
                if os.path.exists(legacy_path):
                    os.remove(legacy_path)

        if model is None:
            raise RuntimeError(
                f"Could not load model from {model_path}. "
                f"HDF5 keys: {top_keys}. Check architecture and Keras version match."
            )

        model.compile(
            optimizer=tf.keras.optimizers.Adam(cfg.local_lr),
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )
        logger.info(
            "Global model loaded — %s params, input shape %s",
            f"{model.count_params():,}", model.input_shape,
        )
        self.global_model = model
        return model

    def create_clients(
        self,
        client_data: Dict[str, tf.data.Dataset],
    ) -> List[FederatedClient]:
        """Create ``FederatedClient`` objects and store the dataset dict."""
        rng = np.random.RandomState(42)
        clients: List[FederatedClient] = []

        for cid, local_ds in client_data.items():
            card = tf.data.experimental.cardinality(local_ds).numpy()
            n_samples = int(card) if card > 0 else sum(1 for _ in local_ds)
            metrics = ClientMetrics(
                local_validation_metric=float(rng.uniform(0.4, 0.9)),
                data_volume=n_samples,
                inference_latency=float(rng.uniform(0.01, 0.15)),
                last_selected_round=0,
            )
            clients.append(FederatedClient(
                client_id=cid,
                local_data=local_ds,
                metrics=metrics,
            ))

        logger.info("Created %d federated clients.", len(clients))
        self.clients = clients
        self.client_datasets = client_data
        return clients

    def setup_tff_process(self) -> None:
        """
        Build the TFF learning process and initialise the round executor.
        """
        _require_tff()
        cfg = self.config
        assert self.global_model is not None, "Call load_global_model() first."

        # Data manager
        input_shape = self.global_model.input_shape[1:]
        cfg.input_shape = input_shape
        self.data_manager = TFFDataManager(input_shape=input_shape)
        element_spec = self.data_manager.get_element_spec()

        # Model factory → model_fn
        factory = TFFModelFactory(
            keras_model=self.global_model,
            input_spec=element_spec,
        )
        model_fn = factory.create_model_fn()

        # Build TFF learning process
        process = build_tff_learning_process(
            model_fn=model_fn,
            config=TFFProcessConfig(
                client_lr=cfg.local_lr,
                server_lr=cfg.server_lr,
                client_optimizer=cfg.client_optimizer,
                server_optimizer=cfg.server_optimizer,
            ),
        )

        # Round executor
        self.tff_executor = TFFRoundExecutor(process, self.global_model)
        self.tff_executor.initialize()
        self.tff_executor.inject_pretrained_weights()

        logger.info("TFF process initialised with pre-trained weights.")

    def setup_enhancement_modules(self) -> None:
        """Wire Parts 1–5 (same logic as federated_learning_cycle.py)."""
        cfg = self.config
        assert self.global_model is not None
        assert len(self.clients) > 0

        # Part 4: Reputation ledger
        self.reputation_ledger = ClientReputationLedger(
            config=cfg.reputation_config,
        )
        for c in self.clients:
            self.reputation_ledger.register(c.client_id)
        self.basic_ledger = self.reputation_ledger.as_basic_ledger()

        # Part 1: Client selector
        self.selector = EnhancedClientSelector(
            clients=self.clients,
            reputation_ledger=self.basic_ledger,
            weights=cfg.selection_weights,
            target_k=cfg.clients_per_round,
        )

        # Part 2: Update validator
        self.validator = UpdateValidator(
            global_model=self.global_model,
            reputation_ledger=self.basic_ledger,
            weights=cfg.contribution_weights,
            clipping=cfg.clipping_config,
            harmful_threshold=cfg.harmful_threshold,
            batch_size=cfg.local_batch_size,
        )

        # Part 5: Evaluator
        self.evaluator = FederatedModelEvaluator(
            model=self.global_model,
            model_name="effnet_global_tff",
            reports_dir=cfg.reports_dir,
        )

        logger.info("Enhancement modules (Parts 1–5) initialised.")

    # ------------------------------------------------------------------ #
    #  Local training (manual — for per-client analysis)                  #
    # ------------------------------------------------------------------ #

    def _local_train(
        self,
        client: FederatedClient,
        global_weights: List[np.ndarray],
    ) -> Tuple[List[np.ndarray], int, float]:
        """
        Manual local training (outside TFF) to obtain per-client model
        weights for contribution scoring and distillation.

        Returns
        -------
        updated_weights, data_volume, local_accuracy
        """
        cfg = self.config
        # Lazily create & cache the local training model
        if self._local_model is None:
            self._local_model = tf.keras.models.clone_model(self.global_model)
            self._local_model.build(self.global_model.input_shape)
            self._local_model.compile(
                optimizer=tf.keras.optimizers.Adam(cfg.local_lr),
                loss="binary_crossentropy",
                metrics=["accuracy"],
            )
        self._local_model.set_weights(global_weights)

        if client.local_data is None:
            return global_weights, 0, 0.0

        dataset = client.local_data.batch(cfg.local_batch_size).prefetch(tf.data.AUTOTUNE)
        self._local_model.fit(dataset, epochs=cfg.local_epochs, verbose=0)
        # Evaluate to get local accuracy for client metric updates
        result = self._local_model.evaluate(dataset, verbose=0, return_dict=True)
        local_acc = result.get("accuracy", 0.0)
        return self._local_model.get_weights(), client.metrics.data_volume, local_acc

    # ------------------------------------------------------------------ #
    #  Reputation sync                                                    #
    # ------------------------------------------------------------------ #

    def _sync_reputation_to_basic_ledger(self) -> None:
        """Copy extended ledger → basic ledger used by Parts 1 & 2."""
        updated_basic = self.reputation_ledger.as_basic_ledger()
        self.basic_ledger._scores = updated_basic._scores

    # ------------------------------------------------------------------ #
    #  Single round                                                       #
    # ------------------------------------------------------------------ #

    def execute_round(
        self,
        current_round: int,
        server_val_data: tf.data.Dataset,
        proxy_data: Optional[tf.data.Dataset] = None,
        supervised_data: Optional[tf.data.Dataset] = None,
    ) -> Dict[str, Any]:
        """
        Execute one complete TFF + enhanced federated round.

        Returns a summary dict with both TFF FedAvg and enhanced accuracy.
        """
        cfg = self.config

        logger.info("── Round %d / %d ──", current_round, cfg.global_rounds)

        # ── 1. Client selection  (Part 1) ──────────────────────────── #
        selected: List[FederatedClient] = self.selector.select(
            current_round=current_round,
        )
        selected_ids = [c.client_id for c in selected]

        # Save pre-round global weights for analysis
        # Extract TFF aggregated weights
        tff_aggregated_weights = self.tff_executor.get_keras_weights()
        
        # IMPORTANT: use them as the starting weights
        global_weights_before = tff_aggregated_weights

        # ── 2. TFF federated round (FedAvg baseline) ──────────────── #
        federated_data = self.data_manager.make_federated_data(
            self.client_datasets,
            selected_ids,
            batch_size=cfg.local_batch_size,
            local_epochs=cfg.local_epochs,
        )
        tff_metrics = self.tff_executor.execute_round(federated_data)

        # Extract TFF-aggregated weights
        tff_aggregated_weights = self.tff_executor.get_keras_weights()

        # Quick TFF FedAvg accuracy check (reuse cached model)
        tff_acc = None
        if cfg.enable_comparison:
            if self._comparison_model is None:
                self._comparison_model = tf.keras.models.clone_model(self.global_model)
                self._comparison_model.build(self.global_model.input_shape)
                self._comparison_model.compile(
                    optimizer="adam", loss="binary_crossentropy",
                    metrics=["accuracy"],
                )
            self._comparison_model.set_weights(tff_aggregated_weights)
            tff_result = self._comparison_model.evaluate(
                server_val_data.batch(cfg.local_batch_size),
                verbose=0, return_dict=True,
            )
            tff_acc = tff_result.get("accuracy", 0.0)

        # ── 3. Per-client analysis (manual, for Parts 2/3/4) ──────── #
        client_updates: Dict[str, List[np.ndarray]] = {}
        data_volumes: Dict[str, int] = {}
        for client in selected:
            updated_w, n, local_acc = self._local_train(client, global_weights_before)
            client_updates[client.client_id] = updated_w
            data_volumes[client.client_id] = n
            client.metrics.local_validation_metric = local_acc

        # ── 4. Update validation & contribution aggregation (Part 2) ─ #
        records: List[ClientUpdateRecord] = self.validator.validate_updates(
            client_updates=client_updates,
            data_volumes=data_volumes,
            server_val_data=server_val_data,
        )
        enhanced_weights = self.validator.aggregate_weighted(
            records, global_weights_before,
        )

        num_accepted = sum(1 for r in records if not r.rejected)
        num_rejected = sum(1 for r in records if r.rejected)

        # Apply enhanced weights to global model
        self.global_model.set_weights(enhanced_weights)
        self.validator.global_model.set_weights(enhanced_weights)

        # ── 5. Knowledge distillation  (Part 3) ───────────────────── #
        distill_loss = None
        if cfg.enable_distillation and proxy_data is not None:
            contribution_weights = {
                r.client_id: r.contribution_weight
                for r in records
                if not r.rejected and r.contribution_weight > 0
            }

            if len(contribution_weights) >= 1:
                kd_history = run_distillation_round(
                    global_model=self.global_model,
                    client_weights={
                        cid: client_updates[cid]
                        for cid in contribution_weights
                    },
                    contribution_weights=contribution_weights,
                    proxy_data=proxy_data,
                    supervised_data=supervised_data,
                    config=cfg.distillation_config,
                )
                distill_loss = kd_history.get("loss_total", [None])[-1]

        # ── 6. Reputation update  (Part 4) ────────────────────────── #
        update_ledger_from_records(
            self.reputation_ledger, records, current_round,
        )
        self.validator.update_reputations(records)
        self._sync_reputation_to_basic_ledger()

        # ── 7. Enhanced accuracy check ────────────────────────────── #
        enhanced_result = self.global_model.evaluate(
            server_val_data.batch(cfg.local_batch_size),
            verbose=0, return_dict=True,
        )
        enhanced_acc = enhanced_result.get("accuracy", 0.0)

        if cfg.enable_comparison and tff_acc is not None:
            delta = enhanced_acc - tff_acc
            logger.info(
                "R%d  FedAvg=%.4f  Enhanced=%.4f  Δ=%+.4f  acc=%d rej=%d",
                current_round, tff_acc, enhanced_acc, delta,
                num_accepted, num_rejected,
            )
        else:
            logger.info(
                "R%d  Enhanced=%.4f  acc=%d rej=%d",
                current_round, enhanced_acc, num_accepted, num_rejected,
            )

        # ── 8. Inject enhanced weights into TFF state ─────────────── #
        self.tff_executor.set_keras_weights(
            self.global_model.get_weights(),
        )

        return {
            "round": current_round,
            "selected": selected_ids,
            "tff_fedavg_accuracy": tff_acc,
            "enhanced_accuracy": enhanced_acc,
            "num_accepted": num_accepted,
            "num_rejected": num_rejected,
            "records": records,
            "distillation_loss": distill_loss,
            "tff_metrics": tff_metrics,
        }

    # ------------------------------------------------------------------ #
    #  Full cycle                                                         #
    # ------------------------------------------------------------------ #

    def run(
        self,
        server_val_data: tf.data.Dataset,
        test_data: tf.data.Dataset,
        proxy_data: Optional[tf.data.Dataset] = None,
        supervised_data: Optional[tf.data.Dataset] = None,
    ) -> Dict[str, list]:
        """
        Run the full TFF Federated Learning cycle.

        Parameters
        ----------
        server_val_data : tf.data.Dataset
            Server validation set for update scoring.
        test_data : tf.data.Dataset
            Independent test set for full evaluation (Part 5).
        proxy_data : tf.data.Dataset | None
            Unlabelled proxy data for distillation (Part 3).
        supervised_data : tf.data.Dataset | None
            Labelled data for combined distillation loss (Part 3).

        Returns
        -------
        history : dict
        """
        cfg = self.config

        logger.info(
            "FL Cycle: %d devices, %d rounds, %d local epochs, comparison=%s",
            cfg.num_devices, cfg.global_rounds, cfg.local_epochs,
            "ON" if cfg.enable_comparison else "OFF",
        )

        # -- Baseline evaluation --------------------------------------- #
        baseline_report = self.evaluator.evaluate(
            test_data=test_data,
            batch_size=cfg.local_batch_size,
            federated_round=0,
            extra_info={"stage": "baseline", "framework": "tff"},
        )
        self.evaluator.save_report(baseline_report, tag="round_000_baseline_tff")
        logger.info(
            "Baseline — Acc: %.4f, F1: %.4f, AUC: %.4f",
            baseline_report.classification.accuracy,
            baseline_report.classification.f1_macro,
            baseline_report.classification.roc_auc,
        )
        all_reports = [baseline_report]
        cycle_start = time.time()

        # ============================================================== #
        #  MAIN LOOP                                                      #
        # ============================================================== #

        for t in range(1, cfg.global_rounds + 1):
            round_start = time.time()

            info = self.execute_round(
                current_round=t,
                server_val_data=server_val_data,
                proxy_data=proxy_data,
                supervised_data=supervised_data,
            )

            # Record history
            self.history["round"].append(t)
            self.history["tff_fedavg_accuracy"].append(info["tff_fedavg_accuracy"])
            self.history["enhanced_accuracy"].append(info["enhanced_accuracy"])
            self.history["selected_clients"].append(info["selected"])
            self.history["num_accepted"].append(info["num_accepted"])
            self.history["num_rejected"].append(info["num_rejected"])
            self.history["distillation_loss"].append(info["distillation_loss"])

            round_elapsed = time.time() - round_start
            logger.info("  └─ %.1fs", round_elapsed)

            # -- Periodic full evaluation (Part 5) --------------------- #
            is_eval_round = (
                t % cfg.eval_every == 0
                or t == 1
                or t == cfg.global_rounds
            )
            if is_eval_round:
                report = self.evaluator.evaluate(
                    test_data=test_data,
                    batch_size=cfg.local_batch_size,
                    federated_round=t,
                    extra_info={
                        "framework": "tff",
                        "tff_fedavg_acc": info["tff_fedavg_accuracy"],
                        "enhanced_acc": info["enhanced_accuracy"],
                        "accepted": info["num_accepted"],
                        "rejected": info["num_rejected"],
                    },
                )
                self.evaluator.save_report(report, tag=f"tff_round_{t:03d}")
                all_reports.append(report)
                logger.info(
                    "  Eval R%d — Acc=%.4f  F1=%.4f  AUC=%.4f",
                    t,
                    report.classification.accuracy,
                    report.classification.f1_macro,
                    report.classification.roc_auc,
                )

        total_elapsed = time.time() - cycle_start

        # ============================================================== #
        #  POST-CYCLE                                                     #
        # ============================================================== #

        logger.info("Cycle complete — %d rounds in %.1fs", cfg.global_rounds, total_elapsed)

        # Comparison report
        if len(all_reports) > 1:
            self.evaluator.save_comparison_report(all_reports)

        # Save reputation ledger
        ledger_path = Path(cfg.reports_dir) / "reputation_ledger_tff_final.json"
        self.reputation_ledger.save(str(ledger_path))

        # TF Lite export
        convert_to_tflite(self.global_model, cfg.tflite_output_path, quantise=False)
        convert_to_tflite(
            self.global_model,
            cfg.tflite_output_path.replace(".tflite", "_quantised.tflite"),
            quantise=True,
        )

        # Final summary
        self._print_summary()

        return self.history

    # ------------------------------------------------------------------ #
    #  Summary                                                            #
    # ------------------------------------------------------------------ #

    def _print_summary(self) -> None:
        """Print a compact final training summary."""
        h = self.history
        if not h["round"]:
            return

        best_idx = int(np.argmax(h["enhanced_accuracy"]))
        best_round = h["round"][best_idx]
        best_acc = h["enhanced_accuracy"][best_idx]
        final_acc = h["enhanced_accuracy"][-1]

        stats = self.reputation_ledger.statistics()

        lines = [
            "FINAL SUMMARY",
            f"  Rounds: {len(h['round'])}  |  Best acc: {best_acc:.4f} (R{best_round})  |  Final acc: {final_acc:.4f}  |  Mean acc: {np.mean(h['enhanced_accuracy']):.4f}",
        ]

        if h["tff_fedavg_accuracy"][0] is not None:
            mean_tff = np.mean([a for a in h["tff_fedavg_accuracy"] if a is not None])
            deltas = [
                e - t for e, t in zip(h["enhanced_accuracy"], h["tff_fedavg_accuracy"])
                if t is not None
            ]
            delta_str = f"  Δ={np.mean(deltas):+.4f}" if deltas else ""
            lines.append(f"  FedAvg acc: {mean_tff:.4f}{delta_str}")

        lines.append(f"  Accepted: {sum(h['num_accepted'])}  Rejected: {sum(h['num_rejected'])}  Rep μ={stats.get('mean_reputation', 0):.4f} σ={stats.get('std_reputation', 0):.4f}")

        kd = [l for l in h["distillation_loss"] if l is not None]
        if kd:
            lines.append(f"  Distillation loss: {np.mean(kd):.5f}")

        lines.append(f"  TFLite: {self.config.tflite_output_path}")
        print("\n".join(lines))


# ====================================================================== #
#  4.  ENTRY POINT  (full production run with .h5 model)                  #
# ====================================================================== #


def main() -> None:
    """
    Full TFF FL cycle with ``phase3_best.weights.h5``.

    Requires TFF and a compatible TF version.
    """
    _require_tff()

    np.random.seed(42)
    tf.random.set_seed(42)

    config = TFFCycleConfig(
        model_path="phase3_best.weights.h5",
        num_devices=100,
        local_epochs=5,
        global_rounds=50,
        clients_per_round=15,
        local_batch_size=32,
        local_lr=1e-4,
        server_lr=0.1,
        eval_every=10,
        enable_distillation=True,
        enable_comparison=True,
    )

    cycle = TFFFederatedLearningCycle(config)

    # Load model
    model = cycle.load_global_model()
    input_shape = model.input_shape[1:]
    config.input_shape = input_shape

    # Synthetic data (replace with real FF++ c23 loaders)
    logger.info("Generating synthetic data …")
    TOTAL = config.num_devices * 10
    train_ds = generate_synthetic_data(TOTAL, input_shape, seed=1)
    val_ds = generate_synthetic_data(200, input_shape, seed=2)
    test_ds = generate_synthetic_data(300, input_shape, seed=3)
    proxy_ds = generate_proxy_data(150, input_shape, seed=4)
    sup_ds = generate_synthetic_data(100, input_shape, seed=5)

    client_data = partition_data_iid_tff(train_ds, config.num_devices)

    # Build cycle
    cycle.create_clients(client_data)
    cycle.setup_tff_process()
    cycle.setup_enhancement_modules()

    # Run
    history = cycle.run(
        server_val_data=val_ds,
        test_data=test_ds,
        proxy_data=proxy_ds,
        supervised_data=sup_ds,
    )

    logger.info("Done. History keys: %s", list(history.keys()))


# ====================================================================== #
#  DEMO / SMOKE-TEST                                                      #
# ====================================================================== #

if __name__ == "__main__":
    print("\n===  TFF Federated Learning Cycle — Demo  ===\n")

    np.random.seed(42)
    tf.random.set_seed(42)

    # ---- 1. Build a tiny model --------------------------------------- #
    INPUT_DIM = 16
    demo_model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(INPUT_DIM,)),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(16, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    demo_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    print(f"Demo model — {demo_model.count_params()} params")

    # ---- 2. Config (reduced) ---------------------------------------- #
    demo_config = TFFCycleConfig(
        model_path="(in-memory demo)",
        num_devices=8,
        local_epochs=1,
        global_rounds=3,
        clients_per_round=4,
        local_batch_size=16,
        local_lr=1e-3,
        server_lr=0.1,
        eval_every=1,
        enable_distillation=True,
        enable_comparison=True,
        distillation_config=DistillationConfig(
            temperature=2.0, lam=0.5, epochs=2,
            batch_size=16, learning_rate=1e-3,
        ),
    )

    # ---- 3. Synthetic data ------------------------------------------- #
    input_shape = (INPUT_DIM,)
    N_CLI = demo_config.num_devices
    TOTAL = N_CLI * 30

    train_ds = generate_synthetic_data(TOTAL, input_shape, seed=10)
    val_ds = generate_synthetic_data(100, input_shape, seed=20)
    test_ds = generate_synthetic_data(120, input_shape, seed=30)
    proxy_ds = generate_proxy_data(80, input_shape, seed=40)
    sup_ds = generate_synthetic_data(60, input_shape, seed=50)
    client_data = partition_data_iid_tff(train_ds, N_CLI)

    # ---- 4. Build cycle ---------------------------------------------- #
    cycle = TFFFederatedLearningCycle(demo_config)
    cycle.global_model = demo_model

    cycle.create_clients(client_data)

    # ---- 5. TFF-dependent vs fallback -------------------------------- #
    if TFF_AVAILABLE:
        print("\n--- TFF available — running full TFF demo ---\n")
        cycle.setup_tff_process()
        cycle.setup_enhancement_modules()

        history = cycle.run(
            server_val_data=val_ds,
            test_data=test_ds,
            proxy_data=proxy_ds,
            supervised_data=sup_ds,
        )

        print("\nRound-by-round comparison:")
        for rnd, tff_a, enh_a in zip(
            history["round"],
            history["tff_fedavg_accuracy"],
            history["enhanced_accuracy"],
        ):
            tff_str = f"{tff_a:.4f}" if tff_a is not None else "  N/A "
            print(f"  Round {rnd}: TFF FedAvg={tff_str} | Enhanced={enh_a:.4f}")

        # Cleanup demo TFLite files
        for p in [demo_config.tflite_output_path,
                  demo_config.tflite_output_path.replace(".tflite", "_quantised.tflite")]:
            Path(p).unlink(missing_ok=True)

    else:
        print(
            "⚠  TFF not installed — running enhancement-only demo.\n"
            "   This exercises Parts 1–5 without TFF's federated layer.\n"
            "   For full TFF integration, see requirements_tff.txt.\n"
        )

        # Fallback: run the non-TFF orchestrator logic as a sanity check
        cycle.setup_enhancement_modules()

        # Simulate 3 rounds using manual local training + enhancements
        for t in range(1, demo_config.global_rounds + 1):
            selected = cycle.selector.select(current_round=t)
            selected_ids = [c.client_id for c in selected]
            global_w = cycle.global_model.get_weights()

            client_updates = {}
            data_volumes = {}
            for c in selected:
                # _local_train returns (weights, data_volume, local_acc)
                w, n, local_acc = cycle._local_train(c, global_w)
                client_updates[c.client_id] = w
                data_volumes[c.client_id] = max(n, 1)
                c.metrics.last_selected_round = t
                c.metrics.local_validation_metric = local_acc

            records = cycle.validator.validate_updates(
                client_updates, data_volumes, val_ds,
            )
            new_w = cycle.validator.aggregate_weighted(records, global_w)
            cycle.global_model.set_weights(new_w)
            cycle.validator.global_model.set_weights(new_w)

            # Distillation
            cw = {r.client_id: r.contribution_weight
                  for r in records if not r.rejected and r.contribution_weight > 0}
            if len(cw) >= 2:
                run_distillation_round(
                    cycle.global_model,
                    {cid: client_updates[cid] for cid in cw},
                    cw, proxy_ds, sup_ds,
                    demo_config.distillation_config,
                )

            update_ledger_from_records(cycle.reputation_ledger, records, t)
            cycle.validator.update_reputations(records)
            cycle._sync_reputation_to_basic_ledger()

            acc = cycle.global_model.evaluate(
                val_ds.batch(16), verbose=0, return_dict=True,
            ).get("accuracy", 0.0)
            print(f"  Round {t}: Enhanced accuracy = {acc:.4f} (selected: {selected_ids})")

        # Quick evaluation
        report = evaluate_and_report(
            cycle.global_model, test_ds,
            model_name="demo_no_tff", federated_round=demo_config.global_rounds,
        )

        print(
            f"\nFinal - Acc: {report.classification.accuracy:.4f}, "
            f"F1: {report.classification.f1_macro:.4f}, "
            f"AUC: {report.classification.roc_auc:.4f}"
        )

    print("\nDone.")

Overwriting tff_federated_cycle.py


## 3. Upload Pre-trained Model

Upload `phase3_best.weights.h5` (the EfficientNet binary classifier
for DeepFake detection). Choose **one** of the methods below.

### Option A: Upload from local machine


In [15]:
# ── Option A: Upload from your local machine ──────────────────
# from google.colab import files
# import os

# if not os.path.exists('phase3_best.weights.h5'):
   # print('Please upload phase3_best.weights.h5')
   # uploaded = files.upload()
   # print(f'Uploaded: {list(uploaded.keys())}')
#else:
   # print('Model file already present.')

# Verify
#size_mb = os.path.getsize('phase3_best.weights.h5') / (1024**2)
#print(f'Model file: phase3_best.weights.h5 ({size_mb:.1f} MB)')


### Option B: Mount Google Drive

If the model is stored in your Google Drive, mount it and copy.


In [16]:
# Replace Colab Drive mount with a robust local/upload approach for Kaggle / Jupyter
import pathlib, shutil, os, glob, time, hashlib

MODEL_FILENAME = "phase3_best.weights.h5"
TARGET = pathlib.Path(MODEL_FILENAME)

# ── ALWAYS try to find the latest model from /kaggle/input/ and overwrite ──
kaggle_hits = []
for path in glob.glob("/kaggle/input/**/*.h5", recursive=True):
    kaggle_hits.append(path)
# Prefer exact-name match
exact_matches = [p for p in kaggle_hits if p.endswith("/" + MODEL_FILENAME)]
chosen = None
if exact_matches:
    chosen = exact_matches[0]
elif kaggle_hits:
    if len(kaggle_hits) == 1:
        chosen = kaggle_hits[0]
    else:
        print("Multiple .h5 files found under /kaggle/input/. Here are the first few:")
        for i, p in enumerate(kaggle_hits[:10], 1):
            print(f"  {i}. {p}")
        chosen = kaggle_hits[0]

if chosen:
    # Always copy (overwrite) so the new model replaces any old cached version
    shutil.copy(chosen, TARGET)
    print(f"Copied {chosen} -> {TARGET.resolve()}")
elif TARGET.exists():
    print(f"Using existing {MODEL_FILENAME} in working directory (no new file in /kaggle/input/).")
else:
    # Fall back to an upload widget
    try:
        from ipywidgets import FileUpload
        from IPython.display import display, HTML

        uploader = FileUpload(accept=".h5", multiple=False)
        display(HTML("<b>Upload your .h5 file using the widget below.</b>"))
        display(uploader)
        print("\nAfter the upload finishes, run the next cell to save the uploaded file into the working directory.\n")
    except Exception as e:
        raise RuntimeError(
            "No model found in working dir or /kaggle/input/, and the interactive upload widget is unavailable. "
            "Either (a) add the file as a Kaggle Dataset (Add data → Upload), (b) upload via the UI with the widget above, "
            "or (c) place the file in the working directory and re-run this cell."
        )

# ── Verify the model file ──
if TARGET.exists():
    stat = os.stat(TARGET)
    size_mb = stat.st_size / (1024 * 1024)
    mod_time = time.ctime(stat.st_mtime)
    # Quick hash of first 1MB for fingerprinting
    with open(TARGET, "rb") as f:
        file_hash = hashlib.md5(f.read(1024 * 1024)).hexdigest()
    print(f"\n📋 Model file verification:")
    print(f"   File:     {TARGET.resolve()}")
    print(f"   Size:     {size_mb:.1f} MB")
    print(f"   Modified: {mod_time}")
    print(f"   Hash:     {file_hash} (first 1MB)")


Copied /kaggle/input/models/averageweebo101/model-v2/keras/default/1/phase3_best.weights.h5 -> /kaggle/working/phase3_best.weights.h5

📋 Model file verification:
   File:     /kaggle/working/phase3_best.weights.h5
   Size:     205.3 MB
   Modified: Wed Mar 11 15:33:43 2026
   Hash:     8771eafb42892ac5e263e780e8f99426 (first 1MB)


## 4. Quick Verification

Verify that all modules import correctly and the model loads.


In [17]:
# verify_imports_and_build_load_model.py
import importlib
import os
import sys
import shutil

modules = [
    "enhanced_client_selection",
    "update_validation",
    "knowledge_distillation",
    "client_reputation_ledger",
    "evaluation_metrics",
    "federated_learning_cycle",
    "tff_data_utils",
    "tff_learning_process",
    "tff_federated_cycle",
]

failed = []
for m in modules:
    try:
        importlib.import_module(m)
    except Exception as e:
        failed.append((m, e))

if failed:
    print("Some modules failed to import:")
    for name, err in failed:
        print(f" - {name}: {err!r}")
    raise SystemExit("Fix the imports above before continuing.")
else:
    print("All 9 modules imported successfully.")

# Verify TFF is detected
from tff_data_utils import TFF_AVAILABLE
print(f"TFF Available: {TFF_AVAILABLE!s}")
assert TFF_AVAILABLE, "TFF must be available for the full cycle!"

import tensorflow as tf
import h5py
from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras.applications.efficientnet import (
    preprocess_input as _effnet_preprocess,
)

def build_deepfake_model(input_shape=(260, 260, 3)):
    """Rebuild the EfficientNet binary classifier architecture.
    Must match the architecture used during Phase 3 training.
    """
    base = EfficientNetB2(include_top=False, weights=None, input_shape=input_shape)
    x = tf.keras.layers.GlobalAveragePooling2D()(base.output)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    model = tf.keras.Model(inputs=base.input, outputs=x)
    return model

weights_path = "phase3_best.weights.h5"
if not os.path.exists(weights_path):
    raise FileNotFoundError(f"Weights file not found at: {weights_path!r}")

# ── Inspect the .h5 file to decide the correct loading strategy ──
with h5py.File(weights_path, "r") as f:
    top_keys = list(f.keys())
    print(f"HDF5 top-level keys: {top_keys}")

model = None

# If the file contains 'model_weights' or 'model_config', it was saved with
# model.save() (full Keras model), NOT model.save_weights().
# Keras 3 misroutes it because of the .weights.h5 extension.
is_full_model = any(k in top_keys for k in ("model_weights", "model_config", "keras_version"))

if is_full_model:
    print("Detected full saved-model HDF5 (not weights-only). Renaming to .h5 for load_model().")
    full_model_path = weights_path.replace(".weights.h5", "_full_tmp.h5")
    shutil.copy(weights_path, full_model_path)
    try:
        _custom = {"preprocess_input": _effnet_preprocess}
        model = tf.keras.models.load_model(
            full_model_path, custom_objects=_custom, compile=False,
        )
        print(f"Loaded as full saved model (via {full_model_path!r})")
    except Exception as e1:
        print(f"load_model() on renamed file failed: {e1}")
    finally:
        if os.path.exists(full_model_path):
            os.remove(full_model_path)

# Strategy 2: Keras 3 native .weights.h5 (weights-only, new format)
if model is None:
    model = build_deepfake_model()
    try:
        model.load_weights(weights_path)
        print(f"Weights loaded (Keras 3 format) from {weights_path!r}")
    except Exception as e2:
        print(f"Keras 3 load_weights() failed: {e2}")
        try:
            model.load_weights(weights_path, skip_mismatch=True)
            print("Loaded weights with skip_mismatch=True (some layers may use random init).")
        except Exception as e2b:
            print(f"skip_mismatch also failed: {e2b}")
            model = None

# Strategy 3: legacy HDF5 weights (Keras 2 format mis-named .weights.h5)
if model is None:
    model = build_deepfake_model()
    legacy_path = weights_path.replace(".weights.h5", "_legacy_tmp.h5")
    shutil.copy(weights_path, legacy_path)
    try:
        model.load_weights(legacy_path)
        print(f"Weights loaded via legacy HDF5 path ({legacy_path!r})")
    except Exception as e3:
        print(f"Legacy .h5 load also failed: {e3}")
        try:
            model.load_weights(legacy_path, skip_mismatch=True)
            print("Loaded legacy weights with skip_mismatch=True (some layers may use random init).")
        except Exception as e3b:
            print(f"Legacy skip_mismatch also failed: {e3b}")
            model = None
    finally:
        if os.path.exists(legacy_path):
            os.remove(legacy_path)

if model is None:
    raise RuntimeError(
        "Could not load the model from the weights file.\n"
        "Ensure phase3_best.weights.h5 was saved with the same TF/Keras version "
        "and model architecture.\n"
        f"File HDF5 keys were: {top_keys}"
    )

model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

print(f"\nModel ready. Input shape: {model.input_shape}, Output shape: {model.output_shape}")
print(f"Total params: {model.count_params():,}")
model.summary()


All 9 modules imported successfully.
TFF Available: True
HDF5 top-level keys: ['layers', 'optimizer', 'vars']
Keras 3 load_weights() failed: A total of 186 objects could not be loaded. Example error message for object <Normalization name=normalization, built=True>:

Layer 'normalization' expected 3 variables, but received 0 variables during loading. Expected: ['mean', 'variance', 'count']

List of objects that could not be loaded:
[<Normalization name=normalization, built=True>, <Conv2D name=stem_conv, built=True>, <BatchNormalization name=stem_bn, built=True>, <DepthwiseConv2D name=block1a_dwconv, built=True>, <BatchNormalization name=block1a_bn, built=True>, <Conv2D name=block1a_se_reduce, built=True>, <Conv2D name=block1a_se_expand, built=True>, <Conv2D name=block1a_project_conv, built=True>, <BatchNormalization name=block1a_project_bn, built=True>, <DepthwiseConv2D name=block1b_dwconv, built=True>, <BatchNormalization name=block1b_bn, built=True>, <Conv2D name=block1b_se_reduce, bu

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:648: UserWarning: A total of 186 objects could not be loaded. Example error message for object <Normalization name=normalization, built=True>:

Layer 'normalization' expected 3 variables, but received 0 variables during loading. Expected: ['mean', 'variance', 'count']

List of objects that could not be loaded:
[<Normalization name=normalization, built=True>, <Conv2D name=stem_conv, built=True>, <BatchNormalization name=stem_bn, built=True>, <DepthwiseConv2D name=block1a_dwconv, built=True>, <BatchNormalization name=block1a_bn, built=True>, <Conv2D name=block1a_se_reduce, built=True>, <Conv2D name=block1a_se_expand, built=True>, <Conv2D name=block1a_project_conv, built=True>, <BatchNormalization name=block1a_project_bn, built=True>, <DepthwiseConv2D name=block1b_dwconv, built=True>, <BatchNormalization name=block1b_bn, built=True>, <Conv2D name=block1b_se_reduce, built=True>, <Conv2D name=block1b_se_expand, built=Tru

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 260, 260,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 260, 260,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 260, 260,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 261, 261,  │          0 │ normalization[0]… │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 130, 130,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 130, 130,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 130, 130,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 130, 130,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 130, 130,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 130, 130,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 130, 130,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 130, 130,  │        512 │ block1a_se_excit… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_bn  │ (None, 130, 130,  │         64 │ block1a_project_

 Total params: 7,769,978 (29.64 MB)

 Trainable params: 7,702,403 (29.38 MB)

 Non-trainable params: 67,575 (263.97 KB)

## 5. Configuration

Configure the federated learning cycle parameters.
Adjust these values based on your experiment requirements.


In [18]:
# ── Experiment Configuration ──────────────────────────────────
from tff_federated_cycle import TFFCycleConfig
from knowledge_distillation import DistillationConfig
from enhanced_client_selection import SelectionWeights
from update_validation import ContributionWeights, ClippingConfig
from client_reputation_ledger import ReputationConfig

config = TFFCycleConfig(
    # ── Core FL settings ────────────────────────────────────
    model_path='phase3_best.weights.h5',
    num_devices=100,           # Total number of simulated clients
    local_epochs=5,            # Local training epochs per client per round
    global_rounds=50,          # Total federated aggregation rounds
    clients_per_round=15,      # Clients selected each round
    local_batch_size=32,
    local_lr=1e-4,             # Client-side learning rate
    server_lr=0.1,             # Server-side learning rate (FedAvg scale)
    eval_every=10,             # Full evaluation every N rounds

    # ── TFF process settings ────────────────────────────────
    client_optimizer='adam',
    server_optimizer='sgd',
    enable_comparison=True,    # Log TFF FedAvg vs enhanced accuracy

    # ── Knowledge Distillation (Part 3) ─────────────────────
    enable_distillation=True,
    distillation_config=DistillationConfig(
        temperature=2.0,
        lam=0.5,
        epochs=3,
        batch_size=32,
        learning_rate=1e-4,
    ),

    # ── Client Selection Weights (Part 1) ───────────────────
    selection_weights=SelectionWeights(
        w_v=0.30,   # Local validation performance
        w_d=0.20,   # Data volume
        w_l=0.10,   # Latency (applied to 1 - L_i)
        w_r=0.25,   # Reputation
        w_s=0.15,   # Staleness penalty
    ),

    # ── Contribution Weights (Part 2) ───────────────────────
    contribution_weights=ContributionWeights(
        alpha=0.35,   # Validation gain
        beta=0.20,    # Similarity to global update history
        gamma=0.20,   # Data volume
        delta=0.25,   # Reputation
    ),
    clipping_config=ClippingConfig(
        clip_threshold=10.0,
        clip_value=5.0,
    ),
    harmful_threshold=0.02,

    # ── Reputation Ledger (Part 4) ──────────────────────────
    reputation_config=ReputationConfig(
        theta=0.0,
        gamma=0.10,
        decay_rate=0.99,
        floor=0.05,
        ceiling=1.0,
        initial_reputation=0.50,
        penalty_factor=0.05,
    ),

    # ── Output ──────────────────────────────────────────────
    reports_dir='reports',
    tflite_output_path='effnet_global_tff_final.tflite',
)

print('Configuration created.')
print(f'  Devices:         {config.num_devices}')
print(f'  Rounds:          {config.global_rounds}')
print(f'  Local epochs:    {config.local_epochs}')
print(f'  Clients/round:   {config.clients_per_round}')
print(f'  Distillation:    {config.enable_distillation}')
print(f'  Comparison mode: {config.enable_comparison}')


Configuration created.
  Devices:         100
  Rounds:          50
  Local epochs:    5
  Clients/round:   15
  Distillation:    True
  Comparison mode: True


## 6. Data Preparation - FaceForensics++ c23

Load the **FaceForensics++ c23** dataset from Kaggle via `kagglehub`,
extract frames from videos, preprocess them (resize to 260×260, normalize
to [0, 1]), and partition across federated clients.

**Preprocessing pipeline** (per frame):
1. Sample evenly-spaced frames from each video with OpenCV
2. Resize to (260, 260) to match the EfficientNet model input
3. Normalize pixel values: `frame / 255.0`
4. Label: 0 = Real, 1 = Fake

**Dataset splits**: 70% train / 15% validation / 15% test

> **Kaggle dataset**: `xdxd003/ff-c23`
> Contains FF++ c23 videos organized by manipulation method
> (DeepFakeDetection, Deepfakes, Face2Face, etc.) and originals.

In [19]:
# Removed

In [20]:
import os
base = "/kaggle/input"
print("exists:", os.path.isdir(base))
if os.path.isdir(base):
    for name in sorted(os.listdir(base)):
        p = os.path.join(base, name)
        print(name, "->", "dir" if os.path.isdir(p) else "file")
        # show first few children
        try:
            for child in sorted(os.listdir(p))[:8]:
                print("   ", child)
        except Exception:
            pass
else:
    print("/kaggle/input does not exist on this session.")

exists: True
datasets -> dir
    xdxd003
models -> dir
    averageweebo101


### 6a. Kaggle Notebook Variant (run THIS instead of the cell above)

When running on **Kaggle**, add **`xdxd003/ff-c23`** as an input dataset to
your notebook. It will be auto-mounted at `/kaggle/input/ff-c23/` — **no
download required**. Run the cell below instead of the previous cell.

In [21]:
# ── Locate Kaggle dataset automatically (works for BOTH old & new layouts) ──
import os

CANDIDATE_PATHS = [
    "/kaggle/input/ff-c23",                              # old style
    "/kaggle/input/datasets/xdxd003/ff-c23",              # new style (yours)
    "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23",
]

DATA_DIR = None
for p in CANDIDATE_PATHS:
    if os.path.isdir(p):
        DATA_DIR = p
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate ff-c23 dataset.\n"
        "Check that it is added to the notebook via 'Add data'."
    )

print(f"Using dataset at: {DATA_DIR}")

Using dataset at: /kaggle/input/datasets/xdxd003/ff-c23


In [22]:
import os

ROOT = "/kaggle/input/datasets/xdxd003/ff-c23"

for root, dirs, files in os.walk(ROOT):
    depth = root.replace(ROOT, "").count(os.sep)
    if depth <= 3:  # only print top few levels to avoid spam
        print(root)
        for d in dirs[:10]:
            print("   └──", d)

/kaggle/input/datasets/xdxd003/ff-c23
   └── FaceForensics++_C23
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23
   └── Face2Face
   └── csv
   └── Deepfakes
   └── DeepFakeDetection
   └── original
   └── NeuralTextures
   └── FaceShifter
   └── FaceSwap
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/Face2Face
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/csv
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/Deepfakes
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/DeepFakeDetection
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/original
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/NeuralTextures
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/FaceShifter
/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23/FaceSwap


In [23]:
from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23")

TARGETS = {
    "real": [],
    "fake": []
}

# Real videos live directly inside /original
real_dir = DATA_DIR / "original"
if real_dir.exists():
    TARGETS["real"].append(real_dir)

# Fake methods (videos directly inside each folder)
FAKE_METHODS = [
    "Deepfakes",
    "FaceSwap",
    "Face2Face",
    "NeuralTextures",
    "FaceShifter"
]

for method in FAKE_METHODS:
    path = DATA_DIR / method
    if path.exists():
        TARGETS["fake"].append(path)

print("Found:")
print("  real dirs :", len(TARGETS["real"]))
print("  fake dirs :", len(TARGETS["fake"]))

Found:
  real dirs : 1
  fake dirs : 5


In [24]:
# CELL 2 — FAST FF++ frame extraction (for flat Kaggle structure)
# - Will skip if frames already exist unless FORCE_RERUN = True
# - Preserves existing frames (no deletions)
import os, time, pathlib
import cv2
import numpy as np
from concurrent.futures import ProcessPoolExecutor, as_completed

# ---------------- USER FLAGS ----------------
FORCE_RERUN = False        # set True to re-run extraction even if frames exist
MIN_EXISTING_PER_CLASS = 1 # if >= this many files in real/fake, extraction will be skipped unless FORCE_RERUN=True

# ---------------- DATASET LOCATION (edit if needed) ----------------
DATA_DIR = "/kaggle/input/datasets/xdxd003/ff-c23/FaceForensics++_C23"

REAL_DIR = os.path.join(DATA_DIR, "original")
FAKE_DIRS = [
    os.path.join(DATA_DIR, "Deepfakes"),
    os.path.join(DATA_DIR, "FaceSwap"),
    # os.path.join(DATA_DIR, "Face2Face"),
    # os.path.join(DATA_DIR, "NeuralTextures"),
]

# ---------------- OUTPUT ----------------
OUT_FRAMES_DIR = "/kaggle/working/ffpp_frames"
IMG_SIZE = (260, 260)
MAX_WORKERS = min(8, os.cpu_count() or 1)   # Kaggle throttles above this
JPEG_QUALITY = 90

os.makedirs(f"{OUT_FRAMES_DIR}/real", exist_ok=True)
os.makedirs(f"{OUT_FRAMES_DIR}/fake", exist_ok=True)

# ---------------- UTIL ----------------
def count_jpeg_files(folder):
    if not os.path.exists(folder):
        return 0
    return sum(1 for _ in pathlib.Path(folder).glob("*.jpg"))

existing_real = count_jpeg_files(os.path.join(OUT_FRAMES_DIR, "real"))
existing_fake = count_jpeg_files(os.path.join(OUT_FRAMES_DIR, "fake"))

if not FORCE_RERUN and existing_real >= MIN_EXISTING_PER_CLASS and existing_fake >= MIN_EXISTING_PER_CLASS:
    print("ℹ️ Skipping extraction — existing frame files detected.")
    print(f"Existing frames — real: {existing_real}, fake: {existing_fake}")
    extraction_skipped = True
else:
    extraction_skipped = False

if extraction_skipped:
    print("✅ Extraction skipped. If you want to force re-extraction, set FORCE_RERUN = True and re-run this cell.")
else:
    # ---------------- SMART SAMPLING ----------------
    def choose_frame_count(total_frames):
        """
        Adaptive sampling:
        Short video  -> keep more density
        Long video   -> aggressively subsample
        """
        if total_frames < 120:
            return 8
        elif total_frames < 300:
            return 12
        elif total_frames < 600:
            return 16
        else:
            return 20   # hard cap

    def extract_video(vpath, label):
        try:
            cap = cv2.VideoCapture(vpath)
            total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            if total <= 0:
                cap.release()
                return 0

            target_frames = choose_frame_count(total)
            step = max(total // target_frames, 1)

            cls = "fake" if label else "real"
            out_dir = f"{OUT_FRAMES_DIR}/{cls}"
            base = pathlib.Path(vpath).stem

            written = 0
            frame_id = 0
            grab_id = 0

            # Sequential decode (MUCH faster than seeking)
            while True:
                ret, frame = cap.read()
                if not ret:
                    break

                if frame_id % step == 0:
                    frame = cv2.resize(frame, IMG_SIZE)
                    out_path = f"{out_dir}/{base}_{grab_id}.jpg"

                    if not os.path.exists(out_path):
                        cv2.imwrite(out_path, frame,
                                    [int(cv2.IMWRITE_JPEG_QUALITY), JPEG_QUALITY])
                        written += 1

                    grab_id += 1
                    if grab_id >= target_frames:
                        break

                frame_id += 1

            cap.release()
            return written

        except Exception as e:
            # Log for debugging but continue
            print(f"Warning: failed to extract {vpath}: {e}")
            return 0

    # ---------------- DISCOVER VIDEOS ----------------
    def list_videos(folder, label):
        if not os.path.exists(folder):
            return []
        return [(os.path.join(folder, f), label)
                for f in os.listdir(folder)
                if f.endswith(".mp4")]

    video_jobs = []
    video_jobs += list_videos(REAL_DIR, 0)

    for fd in FAKE_DIRS:
        if os.path.exists(fd):
            video_jobs += list_videos(fd, 1)

    print(f"Discovered {len(video_jobs)} videos total")

    # ---------------- PARALLEL EXECUTION ----------------
    start = time.time()
    frames_written = 0

    if len(video_jobs) == 0:
        print("⚠️ No videos found to process. Check DATA_DIR paths.")
    else:
        print(f"Starting extraction using {MAX_WORKERS} workers...")

        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as ex:
            futures = [ex.submit(extract_video, v, l) for v, l in video_jobs]

            for i, fut in enumerate(as_completed(futures), 1):
                try:
                    written = fut.result()
                except Exception as e:
                    written = 0
                    print(f"Worker exception: {e}")

                frames_written += written

                if i % 50 == 0 or i == len(video_jobs):
                    elapsed = time.time() - start
                    print(f"{i}/{len(video_jobs)} videos done — "
                          f"{frames_written} frames — {elapsed/60:.1f} min")

        elapsed_total = time.time() - start
        # final counts after extraction
        total_real = count_jpeg_files(os.path.join(OUT_FRAMES_DIR, "real"))
        total_fake = count_jpeg_files(os.path.join(OUT_FRAMES_DIR, "fake"))

        print("\n✅ Extraction finished")
        print("Total frames newly written in this run:", frames_written)
        print(f"Frames on disk — real: {total_real}, fake: {total_fake}")
        print(f"Elapsed time: {elapsed_total/60:.2f} minutes")

        if frames_written == 0 and total_real + total_fake == 0:
            print("❌ FAILURE: No frames were written and no frames exist on disk.")
        elif frames_written == 0:
            print("⚠️ No new frames were written (existing frames preserved).")
        else:
            print("✔ SUCCESS: Frames saved to", OUT_FRAMES_DIR)

# final summary for easy programmatic check
summary = {
    "extraction_skipped": extraction_skipped,
    "existing_real": existing_real,
    "existing_fake": existing_fake,
    "force_rerun": FORCE_RERUN,
}
print("\nSummary:", summary)

ℹ️ Skipping extraction — existing frame files detected.
Existing frames — real: 16844, fake: 16844
✅ Extraction skipped. If you want to force re-extraction, set FORCE_RERUN = True and re-run this cell.

Summary: {'extraction_skipped': True, 'existing_real': 16844, 'existing_fake': 16844, 'force_rerun': False}


In [25]:
# --- Recreate the Federated Learning cycle object (lost after kernel restart) ---

import sys
import os
import glob
import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess

# Make sure the repo root is on the path (Kaggle sometimes loses it)
repo_root = "/kaggle/working"
if repo_root not in sys.path:
    sys.path.append(repo_root)

# Import the TFF version (matches the TFFCycleConfig created in the config cell)
from tff_federated_cycle import TFFFederatedLearningCycle

print("Imported TFFFederatedLearningCycle successfully")

# Reuse the SAME config object you created earlier
try:
    config
except NameError:
    raise RuntimeError("Config object missing — rerun the config/setup cell before this.")

# Recreate cycle using the TFF variant
cycle = TFFFederatedLearningCycle(config)
print("Cycle rebuilt:", type(cycle))

# ── Step 1: Load the global model (required before setup_tff_process) ──
cycle.load_global_model()
MODEL_IMG_SIZE = tuple(cycle.global_model.input_shape[1:3])  # e.g. (224, 224)
print(f"Global model loaded: {cycle.global_model.count_params():,} params, input {MODEL_IMG_SIZE}")

# ── Define TFRecord loader (self-contained) ──
def parse_example(example_proto):
    feature_desc = {
        "image/encoded": tf.io.FixedLenFeature([], tf.string),
        "image/format": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.float32),
    }
    parsed = tf.io.parse_single_example(example_proto, feature_desc)
    image = tf.io.decode_jpeg(parsed["image/encoded"], channels=3)
    image = tf.image.resize(image, MODEL_IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = effnet_preprocess(image)
    label = parsed["label"]
    return image, label

def load_client_dataset_unbatched(tfrecord_path):
    """Return an UNBATCHED dataset — make_federated_data applies .batch() later."""
    ds = tf.data.TFRecordDataset(
        tfrecord_path,
        compression_type="GZIP",
        num_parallel_reads=1,
    )
    ds = ds.map(parse_example, num_parallel_calls=1)
    return ds

# ── Step 2: Build client_data dict from TFRecord shards (UNBATCHED) ──
CLIENT_DIR = "/kaggle/working/ffpp_tfrecord_clients"
client_tfrecord_files = sorted(glob.glob(os.path.join(CLIENT_DIR, "*.tfrecord")))
assert len(client_tfrecord_files) > 0, f"No .tfrecord files found in {CLIENT_DIR}"

client_data = {}
for fp in client_tfrecord_files:
    cid = os.path.splitext(os.path.basename(fp))[0]  # e.g. "client_0"
    client_data[cid] = load_client_dataset_unbatched(fp)

print(f"Built client_data dict: {len(client_data)} clients")

# ── Step 3: Create federated clients ──
cycle.create_clients(client_data)
print(f"Created {len(cycle.clients)} federated clients")

2026-03-11 15:33:51,439 | INFO     | Loading global model from phase3_best.weights.h5 …


Imported TFFFederatedLearningCycle successfully
Cycle rebuilt: <class 'tff_federated_cycle.TFFFederatedLearningCycle'>


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:648: UserWarning: A total of 186 objects could not be loaded. Example error message for object <Normalization name=normalization_1, built=True>:

Layer 'normalization_1' expected 3 variables, but received 0 variables during loading. Expected: ['mean', 'variance', 'count']

List of objects that could not be loaded:
[<Normalization name=normalization_1, built=True>, <Conv2D name=stem_conv, built=True>, <BatchNormalization name=stem_bn, built=True>, <DepthwiseConv2D name=block1a_dwconv, built=True>, <BatchNormalization name=block1a_bn, built=True>, <Conv2D name=block1a_se_reduce, built=True>, <Conv2D name=block1a_se_expand, built=True>, <Conv2D name=block1a_project_conv, built=True>, <BatchNormalization name=block1a_project_bn, built=True>, <DepthwiseConv2D name=block1b_dwconv, built=True>, <BatchNormalization name=block1b_bn, built=True>, <Conv2D name=block1b_se_reduce, built=True>, <Conv2D name=block1b_se_expand, bui

Global model loaded: 7,769,978 params, input (260, 260)
Built client_data dict: 100 clients


2026-03-11 15:34:17,727 | INFO     | Created 100 federated clients.


Created 100 federated clients


In [26]:
import os
import glob
import tensorflow as tf
from tqdm import tqdm

frames_root = "/kaggle/working/ffpp_frames"
out_root = "/kaggle/working/ffpp_tfrecord_clients"

os.makedirs(out_root, exist_ok=True)

NUM_CLIENTS = 100

image_paths = sorted(glob.glob(frames_root + "/**/*.jpg", recursive=True))
print("Total frames:", len(image_paths))

def get_label(path):
    return 1.0 if "FAKE" in path.upper() else 0.0

def serialize_example(img_bytes, label):
    feature = {
        "image/encoded": tf.train.Feature(bytes_list=tf.train.BytesList(value=[img_bytes])),
        "image/format": tf.train.Feature(bytes_list=tf.train.BytesList(value=[b"jpeg"])),
        "label": tf.train.Feature(float_list=tf.train.FloatList(value=[label])),
    }
    example = tf.train.Example(features=tf.train.Features(feature=feature))
    return example.SerializeToString()

# distribute evenly
splits = [[] for _ in range(NUM_CLIENTS)]
for i, p in enumerate(image_paths):
    splits[i % NUM_CLIENTS].append(p)

print("Writing GZIP-compressed TFRecords...")

# Write with GZIP compression so downstream cells (load_client_dataset) can read them
tfrecord_options = tf.io.TFRecordOptions(compression_type="GZIP")

for cid, paths in enumerate(tqdm(splits)):
    tfrecord_path = f"{out_root}/client_{cid}.tfrecord"
    with tf.io.TFRecordWriter(tfrecord_path, options=tfrecord_options) as writer:
        for p in paths:
            with open(p, "rb") as f:
                img_bytes = f.read()
            label = get_label(p)
            writer.write(serialize_example(img_bytes, label))

print("TFRecord rebuild complete.")

Total frames: 33688
Writing GZIP-compressed TFRecords...


100%|██████████| 100/100 [00:29<00:00,  3.45it/s]

TFRecord rebuild complete.


In [27]:
import tensorflow as tf
import glob

files = sorted(glob.glob("/kaggle/working/ffpp_tfrecord_clients/client_*.tfrecord"))

print("Validating first 3 files...")

for fp in files[:3]:
    print("Testing:", fp)
    ds = tf.data.TFRecordDataset(fp, compression_type="GZIP")
    for _ in ds.take(1):
        pass
    print("  OK")

Validating first 3 files...
Testing: /kaggle/working/ffpp_tfrecord_clients/client_0.tfrecord
  OK
Testing: /kaggle/working/ffpp_tfrecord_clients/client_1.tfrecord
  OK
Testing: /kaggle/working/ffpp_tfrecord_clients/client_10.tfrecord
  OK


In [28]:
# CELL 5.5 — rebuild client file list after kernel restart

import glob
import os

CLIENT_DIR = "/kaggle/working/ffpp_tfrecord_clients"

assert os.path.exists(CLIENT_DIR), "TFRecord directory missing!"

client_files = sorted(glob.glob(os.path.join(CLIENT_DIR, "*.tfrecord")))

print("Discovered", len(client_files), "client shards")
print("First 5:", client_files[:5])

Discovered 100 client shards
First 5: ['/kaggle/working/ffpp_tfrecord_clients/client_0.tfrecord', '/kaggle/working/ffpp_tfrecord_clients/client_1.tfrecord', '/kaggle/working/ffpp_tfrecord_clients/client_10.tfrecord', '/kaggle/working/ffpp_tfrecord_clients/client_11.tfrecord', '/kaggle/working/ffpp_tfrecord_clients/client_12.tfrecord']


## 7. Initialize TFF Process & Enhancement Modules

Build the TFF Weighted FedAvg learning process and wire up
all five enhancement modules.


In [29]:
# CELL 6 — loader to read TFRecord client shards (UPDATED FOR JPEG STORAGE)

import tensorflow as tf
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess

IMG_SIZE = (260, 260)   # must match the size used during frame extraction

def parse_example(example_proto):
    feature_desc = {
        "image/encoded": tf.io.FixedLenFeature([], tf.string),
        "image/format": tf.io.FixedLenFeature([], tf.string),
        "label": tf.io.FixedLenFeature([], tf.float32),
    }

    parsed = tf.io.parse_single_example(example_proto, feature_desc)

    image = tf.io.decode_jpeg(parsed["image/encoded"], channels=3)
    image = tf.image.resize(image, IMG_SIZE)

    image = tf.cast(image, tf.float32)
    image = effnet_preprocess(image)

    label = parsed["label"]
    return image, label


def load_client_dataset(tfrecord_path, batch_size=4):  # keep small for memory safety
    ds = tf.data.TFRecordDataset(
        tfrecord_path,
        compression_type="GZIP",     # IMPORTANT (we wrote gzip!)
        num_parallel_reads=1
    )

    ds = ds.map(parse_example, num_parallel_calls=1)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(1)

    return ds


# Example usage
example_client = load_client_dataset(client_files[0])
for x, y in example_client.take(1):
    print(x.shape, y.shape)

# sanity check — force one batch (verifies decode works)
for x, y in example_client.take(1):
    print("Batch shape:", x.shape, "Labels shape:", y.shape)

(4, 260, 260, 3) (4,)
Batch shape: (4, 260, 260, 3) Labels shape: (4,)


In [30]:
# ── Reload all %%writefile modules so the kernel picks up any edits ───
import importlib
import flwr_adapter
import evaluation_metrics
import enhanced_client_selection
import update_validation
import knowledge_distillation
import client_reputation_ledger
import tff_data_utils
import tff_learning_process
import tff_federated_cycle

for mod in [
    flwr_adapter,
    evaluation_metrics,
    enhanced_client_selection,
    update_validation,
    knowledge_distillation,
    client_reputation_ledger,
    tff_data_utils,
    tff_learning_process,
    tff_federated_cycle,
]:
    importlib.reload(mod)

print("All modules reloaded from disk.")

# ── Rebuild cycle with freshly-reloaded classes ──────────────────────
from tff_federated_cycle import TFFFederatedLearningCycle
cycle = TFFFederatedLearningCycle(config)
cycle.load_global_model()
cycle.create_clients(client_data)

# ── Setup TFF federated learning process ─────────────────────
cycle.setup_tff_process()
print('TFF Learning Process initialised.')

# ── Setup enhancement modules (Parts 1-5) ────────────────────
cycle.setup_enhancement_modules()
print('Enhancement modules (Parts 1-5) initialised.')
print('\n✅ Ready to run the federated learning cycle.')

2026-03-11 15:34:47,092 | INFO     | Loading global model from phase3_best.weights.h5 …


All modules reloaded from disk.


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:648: UserWarning: A total of 186 objects could not be loaded. Example error message for object <Normalization name=normalization_2, built=True>:

Layer 'normalization_2' expected 3 variables, but received 0 variables during loading. Expected: ['mean', 'variance', 'count']

List of objects that could not be loaded:
[<Normalization name=normalization_2, built=True>, <Conv2D name=stem_conv, built=True>, <BatchNormalization name=stem_bn, built=True>, <DepthwiseConv2D name=block1a_dwconv, built=True>, <BatchNormalization name=block1a_bn, built=True>, <Conv2D name=block1a_se_reduce, built=True>, <Conv2D name=block1a_se_expand, built=True>, <Conv2D name=block1a_project_conv, built=True>, <BatchNormalization name=block1a_project_bn, built=True>, <DepthwiseConv2D name=block1b_dwconv, built=True>, <BatchNormalization name=block1b_bn, built=True>, <Conv2D name=block1b_se_reduce, built=True>, <Conv2D name=block1b_se_expand, bui

TFF Learning Process initialised.
Enhancement modules (Parts 1-5) initialised.

✅ Ready to run the federated learning cycle.


## 8. Run the Full TFF Federated Learning Cycle

Execute the complete federated training loop:
- Each round: Client Selection → TFF FedAvg → Update Validation →
  Knowledge Distillation → Reputation Update → Evaluation
- Comparison mode logs both TFF FedAvg and enhanced accuracy
- Full evaluation reports saved every `eval_every` rounds

> **Runtime:** With 100 clients, 50 rounds, and EfficientNet,
> this may take significant time. Consider reducing
> `global_rounds` or `num_devices` for initial testing.


## 8a. Checkpoint & Memory Management Utilities

These helpers allow the federated training loop to:
1. **Save checkpoints** every N rounds (model weights, history, reputation ledger, round number)
2. **Resume from the latest checkpoint** if the session is interrupted — simply re-run the training cell
3. **Monitor and manage memory** to avoid Kaggle OOM crashes (GPU 16 GiB, CPU RAM 29 GiB, Disk 57.6 GiB)

In [31]:
# ── Checkpoint & Memory Management ────────────────────────────
import os, gc, json, time, logging
import numpy as np
import tensorflow as tf

logger = logging.getLogger(__name__)

# ═══════════════════════════════════════════════════════════════
#  CONFIGURATION
# ═══════════════════════════════════════════════════════════════
CHECKPOINT_DIR   = "/kaggle/working/fl_checkpoints"
CHECKPOINT_EVERY = 2          # save every N rounds

# Kaggle resource limits (used for monitoring/warnings)
GPU_MEM_LIMIT_GB  = 15.0      # leave 1 GiB headroom from 16 GiB
CPU_RAM_LIMIT_GB  = 27.0      # leave 2 GiB headroom from 29 GiB
DISK_LIMIT_GB     = 55.0      # leave ~2.5 GiB headroom from 57.6 GiB

# ═══════════════════════════════════════════════════════════════
#  MEMORY MONITORING
# ═══════════════════════════════════════════════════════════════
def _bytes_to_gb(b):
    return b / (1024 ** 3)

def get_memory_usage():
    """Return a dict with current CPU RAM, GPU, and disk usage in GiB."""
    info = {}

    # CPU RAM via /proc/meminfo (works on Kaggle Linux)
    try:
        with open("/proc/meminfo") as f:
            meminfo = {}
            for line in f:
                parts = line.split()
                meminfo[parts[0].rstrip(":")] = int(parts[1]) * 1024  # kB → bytes
        total = meminfo.get("MemTotal", 0)
        avail = meminfo.get("MemAvailable", 0)
        used = total - avail
        info["cpu_ram_used_gb"] = _bytes_to_gb(used)
        info["cpu_ram_total_gb"] = _bytes_to_gb(total)
        info["cpu_ram_avail_gb"] = _bytes_to_gb(avail)
    except Exception:
        info["cpu_ram_used_gb"] = -1

    # GPU memory
    try:
        gpus = tf.config.list_physical_devices("GPU")
        if gpus:
            mem_info = tf.config.experimental.get_memory_info("GPU:0")
            info["gpu_used_gb"] = _bytes_to_gb(mem_info["current"])
            info["gpu_peak_gb"] = _bytes_to_gb(mem_info["peak"])
        else:
            info["gpu_used_gb"] = 0
            info["gpu_peak_gb"] = 0
    except Exception:
        info["gpu_used_gb"] = -1
        info["gpu_peak_gb"] = -1

    # Disk usage via os.statvfs
    try:
        st = os.statvfs("/kaggle/working")
        disk_total = st.f_blocks * st.f_frsize
        disk_free = st.f_bavail * st.f_frsize
        disk_used = disk_total - disk_free
        info["disk_used_gb"] = _bytes_to_gb(disk_used)
        info["disk_total_gb"] = _bytes_to_gb(disk_total)
        info["disk_free_gb"] = _bytes_to_gb(disk_free)
    except Exception:
        info["disk_used_gb"] = -1

    return info

def log_memory(prefix=""):
    """Log current memory usage and warn if approaching limits."""
    m = get_memory_usage()
    parts = []
    warnings = []

    if m.get("cpu_ram_used_gb", -1) >= 0:
        parts.append(f"CPU RAM: {m['cpu_ram_used_gb']:.1f}/{m['cpu_ram_total_gb']:.1f} GiB")
        if m["cpu_ram_used_gb"] > CPU_RAM_LIMIT_GB:
            warnings.append(f"⚠ CPU RAM {m['cpu_ram_used_gb']:.1f} GiB exceeds soft limit {CPU_RAM_LIMIT_GB} GiB")

    if m.get("gpu_used_gb", -1) >= 0:
        parts.append(f"GPU: {m['gpu_used_gb']:.2f} GiB (peak {m['gpu_peak_gb']:.2f})")
        if m["gpu_used_gb"] > GPU_MEM_LIMIT_GB:
            warnings.append(f"⚠ GPU memory {m['gpu_used_gb']:.2f} GiB exceeds soft limit {GPU_MEM_LIMIT_GB} GiB")

    if m.get("disk_used_gb", -1) >= 0:
        parts.append(f"Disk: {m['disk_used_gb']:.1f}/{m['disk_total_gb']:.1f} GiB")
        if m["disk_used_gb"] > DISK_LIMIT_GB:
            warnings.append(f"⚠ Disk usage {m['disk_used_gb']:.1f} GiB exceeds soft limit {DISK_LIMIT_GB} GiB")

    tag = f"[{prefix}] " if prefix else ""
    logger.info("%sMemory — %s", tag, " | ".join(parts))
    for w in warnings:
        logger.warning(w)
    return m

def cleanup_memory():
    """Force garbage collection and clear TF caches where possible."""
    gc.collect()
    try:
        tf.keras.backend.clear_session.__wrapped__  # check availability
    except AttributeError:
        pass
    gc.collect()

# ═══════════════════════════════════════════════════════════════
#  CHECKPOINT SAVE / LOAD
# ═══════════════════════════════════════════════════════════════
def save_checkpoint(cycle, round_num, all_reports=None):
    """
    Save a full training checkpoint: model weights, history,
    reputation ledger, and metadata.
    """
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # 1. Model weights
    weights_path = os.path.join(CHECKPOINT_DIR, "model_weights.weights.h5")
    cycle.global_model.save_weights(weights_path)

    # 2. History (JSON-serialisable)
    history_path = os.path.join(CHECKPOINT_DIR, "history.json")
    # Convert any numpy types to plain Python for JSON
    serialisable_history = {}
    for k, v in cycle.history.items():
        serialisable_history[k] = [
            float(x) if isinstance(x, (np.floating, float)) else
            int(x) if isinstance(x, (np.integer, int)) else
            x for x in v
        ]
    with open(history_path, "w") as f:
        json.dump(serialisable_history, f, indent=2, default=str)

    # 3. Reputation ledger
    ledger_path = os.path.join(CHECKPOINT_DIR, "reputation_ledger.json")
    cycle.reputation_ledger.save(ledger_path)

    # 4. Metadata
    meta = {
        "round": round_num,
        "global_rounds": cycle.config.global_rounds,
        "timestamp": time.time(),
        "num_reports": len(all_reports) if all_reports else 0,
    }
    meta_path = os.path.join(CHECKPOINT_DIR, "checkpoint_meta.json")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    logger.info("💾 Checkpoint saved at round %d → %s", round_num, CHECKPOINT_DIR)


def load_checkpoint(cycle):
    """
    Try to load a checkpoint. Returns (start_round, all_reports_count)
    if found, or (0, 0) if no checkpoint exists.

    Restores: model weights, history, reputation ledger, TFF state.
    """
    meta_path = os.path.join(CHECKPOINT_DIR, "checkpoint_meta.json")
    if not os.path.exists(meta_path):
        logger.info("No checkpoint found — starting from scratch.")
        return 0, 0

    # Load metadata
    with open(meta_path) as f:
        meta = json.load(f)
    saved_round = meta["round"]

    # Verify all files exist
    weights_path = os.path.join(CHECKPOINT_DIR, "model_weights.weights.h5")
    history_path = os.path.join(CHECKPOINT_DIR, "history.json")
    ledger_path  = os.path.join(CHECKPOINT_DIR, "reputation_ledger.json")

    for p in [weights_path, history_path, ledger_path]:
        if not os.path.exists(p):
            logger.warning("Checkpoint incomplete (missing %s) — starting fresh.", p)
            return 0, 0

    # 1. Restore model weights
    cycle.global_model.load_weights(weights_path)
    logger.info("  ✓ Model weights restored")

    # 2. Restore history
    with open(history_path) as f:
        cycle.history = json.load(f)
    logger.info("  ✓ History restored (%d rounds)", len(cycle.history.get("round", [])))

    # 3. Restore reputation ledger
    from client_reputation_ledger import ClientReputationLedger
    cycle.reputation_ledger = ClientReputationLedger.load(ledger_path)
    cycle.basic_ledger = cycle.reputation_ledger.as_basic_ledger()
    # Update references in selector and validator
    cycle.selector.reputation_ledger = cycle.basic_ledger
    cycle.validator.reputation_ledger = cycle.basic_ledger
    logger.info("  ✓ Reputation ledger restored")

    # 4. Sync model weights into TFF executor and validator
    cycle.tff_executor.set_keras_weights(cycle.global_model.get_weights())
    cycle.validator.global_model.set_weights(cycle.global_model.get_weights())
    logger.info("  ✓ TFF executor & validator weights synced")

    logger.info("🔄 Resuming from checkpoint at round %d", saved_round)
    return saved_round, meta.get("num_reports", 0)


# ═══════════════════════════════════════════════════════════════
#  MAIN TRAINING WRAPPER WITH CHECKPOINTS + MEMORY MANAGEMENT
# ═══════════════════════════════════════════════════════════════
def run_with_checkpoints(
    cycle,
    server_val_data,
    test_data,
    proxy_data=None,
    supervised_data=None,
    force_fresh_start=False,
):
    """
    Drop-in replacement for cycle.run() that adds:
      • Checkpoint save every CHECKPOINT_EVERY rounds
      • Automatic resume from the latest checkpoint
      • Memory monitoring and garbage collection after every round

    Set force_fresh_start=True to ignore any existing checkpoint
    and train from the currently loaded model weights (e.g. after
    uploading a new phase3_best.weights.h5).
    """
    from evaluation_metrics import evaluate_and_report
    from pathlib import Path

    cfg = cycle.config

    logger.info(
        "FL Cycle (with checkpoints): %d devices, %d rounds, "
        "checkpoint every %d rounds",
        cfg.num_devices, cfg.global_rounds, CHECKPOINT_EVERY,
    )

    # ── Try to resume from checkpoint ──────────────────────────
    if force_fresh_start:
        logger.info("force_fresh_start=True — ignoring any existing checkpoint.")
        resume_round, saved_reports_count = 0, 0
    else:
        resume_round, saved_reports_count = load_checkpoint(cycle)

    all_reports = []

    # ── Baseline evaluation (only if starting fresh) ───────────
    if resume_round == 0:
        log_memory("Pre-baseline")
        baseline_report = cycle.evaluator.evaluate(
            test_data=test_data,
            batch_size=cfg.local_batch_size,
            federated_round=0,
            extra_info={"stage": "baseline", "framework": "tff"},
        )
        cycle.evaluator.save_report(baseline_report, tag="round_000_baseline_tff")
        logger.info(
            "Baseline — Acc: %.4f, F1: %.4f, AUC: %.4f",
            baseline_report.classification.accuracy,
            baseline_report.classification.f1_macro,
            baseline_report.classification.roc_auc,
        )
        all_reports.append(baseline_report)
    else:
        logger.info("Skipping baseline (resuming from round %d)", resume_round)

    cycle_start = time.time()
    start_round = resume_round + 1

    # ════════════════════════════════════════════════════════════
    #  MAIN LOOP (resumes from checkpoint if available)
    # ════════════════════════════════════════════════════════════
    for t in range(start_round, cfg.global_rounds + 1):
        round_start = time.time()

        # Memory check before round
        mem = get_memory_usage()
        if mem.get("cpu_ram_avail_gb", 999) < 2.0:
            logger.warning("⚠ Low CPU RAM (%.1f GiB free) — forcing GC before round %d",
                          mem["cpu_ram_avail_gb"], t)
            cleanup_memory()

        # ── Execute the round ──────────────────────────────────
        info = cycle.execute_round(
            current_round=t,
            server_val_data=server_val_data,
            proxy_data=proxy_data,
            supervised_data=supervised_data,
        )

        # Record history
        cycle.history["round"].append(t)
        cycle.history["tff_fedavg_accuracy"].append(info["tff_fedavg_accuracy"])
        cycle.history["enhanced_accuracy"].append(info["enhanced_accuracy"])
        cycle.history["selected_clients"].append(info["selected"])
        cycle.history["num_accepted"].append(info["num_accepted"])
        cycle.history["num_rejected"].append(info["num_rejected"])
        cycle.history["distillation_loss"].append(info["distillation_loss"])

        round_elapsed = time.time() - round_start
        logger.info("  └─ %.1fs", round_elapsed)

        # ── Periodic full evaluation (Part 5) ─────────────────
        is_eval_round = (
            t % cfg.eval_every == 0
            or t == 1
            or t == cfg.global_rounds
        )
        if is_eval_round:
            report = cycle.evaluator.evaluate(
                test_data=test_data,
                batch_size=cfg.local_batch_size,
                federated_round=t,
                extra_info={
                    "framework": "tff",
                    "tff_fedavg_acc": info["tff_fedavg_accuracy"],
                    "enhanced_acc": info["enhanced_accuracy"],
                    "accepted": info["num_accepted"],
                    "rejected": info["num_rejected"],
                },
            )
            cycle.evaluator.save_report(report, tag=f"tff_round_{t:03d}")
            all_reports.append(report)
            logger.info(
                "  Eval R%d — Acc=%.4f  F1=%.4f  AUC=%.4f",
                t,
                report.classification.accuracy,
                report.classification.f1_macro,
                report.classification.roc_auc,
            )

        # ── Checkpoint save ────────────────────────────────────
        if t % CHECKPOINT_EVERY == 0 or t == cfg.global_rounds:
            save_checkpoint(cycle, t, all_reports)

        # ── Memory cleanup after every round ───────────────────
        cleanup_memory()
        if t % 5 == 0 or t == 1:
            log_memory(f"R{t}")

    total_elapsed = time.time() - cycle_start

    # ════════════════════════════════════════════════════════════════════
    #  POST-CYCLE
    # ════════════════════════════════════════════════════════════════════
    logger.info("Cycle complete — %d rounds in %.1fs", cfg.global_rounds, total_elapsed)

    # Comparison report
    if len(all_reports) > 1:
        cycle.evaluator.save_comparison_report(all_reports)

    # Save final reputation ledger
    from pathlib import Path as _Path
    ledger_path = _Path(cfg.reports_dir) / "reputation_ledger_tff_final.json"
    cycle.reputation_ledger.save(str(ledger_path))

    # TF Lite export
    from tff_federated_cycle import convert_to_tflite
    convert_to_tflite(cycle.global_model, cfg.tflite_output_path, quantise=False)
    convert_to_tflite(
        cycle.global_model,
        cfg.tflite_output_path.replace(".tflite", "_quantised.tflite"),
        quantise=True,
    )

    # Final summary and memory/log prints
    cycle._print_summary()
    print(f"   Limits — GPU: {GPU_MEM_LIMIT_GB} GiB | CPU RAM: {CPU_RAM_LIMIT_GB} GiB | Disk: {DISK_LIMIT_GB} GiB")

    log_memory("Final")
    print(f"   Save every:       {CHECKPOINT_EVERY} rounds")
    print(f"   Checkpoint dir:   {CHECKPOINT_DIR}")

    logger.info("✅ Checkpoint & memory management utilities loaded.")
    return cycle.history

In [32]:
# ── Build val_ds, test_ds, proxy_ds, sup_ds from extracted frames ──

import os, glob, tensorflow as tf, numpy as np

# Use model's actual input shape (loaded in cell 45 via cycle.load_global_model())
MODEL_IMG_SIZE = tuple(cycle.global_model.input_shape[1:3])  # e.g. (224, 224)
print(f"Model input size: {MODEL_IMG_SIZE}")

FRAMES_DIR = "/kaggle/working/ffpp_frames"

# Gather all frame paths + labels
all_paths, all_labels = [], []
for p in sorted(glob.glob(FRAMES_DIR + "/**/*.jpg", recursive=True)):
    label = 1.0 if "fake" in p.lower() else 0.0
    all_paths.append(p)
    all_labels.append(label)

print(f"Total frames: {len(all_paths)}")

# Shuffle deterministically
rng = np.random.RandomState(42)
idx = rng.permutation(len(all_paths))
all_paths  = [all_paths[i] for i in idx]
all_labels = [all_labels[i] for i in idx]

# Split: 15% val, 10% test, 3% proxy (unlabelled), 2% supervised distillation
n = len(all_paths)
n_val   = max(1, int(n * 0.15))
n_test  = max(1, int(n * 0.10))
n_proxy = max(1, int(n * 0.03))
n_sup   = max(1, int(n * 0.02))

val_paths,   val_labels   = all_paths[:n_val],                all_labels[:n_val]
test_paths,  test_labels  = all_paths[n_val:n_val+n_test],    all_labels[n_val:n_val+n_test]
proxy_paths               = all_paths[n_val+n_test:n_val+n_test+n_proxy]
sup_paths,   sup_labels   = all_paths[n_val+n_test+n_proxy:n_val+n_test+n_proxy+n_sup], \
                            all_labels[n_val+n_test+n_proxy:n_val+n_test+n_proxy+n_sup]

print(f"val={len(val_paths)}, test={len(test_paths)}, proxy={len(proxy_paths)}, sup={len(sup_paths)}")

# Helper: load image — no external preprocessing; the model's built-in
# preprocess_input layer handles normalisation automatically.
def _load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.io.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, MODEL_IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img, label

def _load_image_only(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, MODEL_IMG_SIZE)
    img = tf.cast(img, tf.float32)
    return img

# Build UNBATCHED tf.data.Datasets — cycle.run() and evaluate() batch internally
val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_ds = val_ds.map(_load_image, num_parallel_calls=tf.data.AUTOTUNE)

test_ds = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_ds = test_ds.map(_load_image, num_parallel_calls=tf.data.AUTOTUNE)

# proxy_ds is UNLABELLED (images only) — used for knowledge distillation
proxy_ds = tf.data.Dataset.from_tensor_slices(proxy_paths)
proxy_ds = proxy_ds.map(_load_image_only, num_parallel_calls=tf.data.AUTOTUNE)

# sup_ds is labelled — supervised component of distillation loss
sup_ds = tf.data.Dataset.from_tensor_slices((sup_paths, sup_labels))
sup_ds = sup_ds.map(_load_image, num_parallel_calls=tf.data.AUTOTUNE)

print("val_ds, test_ds, proxy_ds, sup_ds ready (unbatched)")

# Quick shape check (take single elements)
for x, y in val_ds.take(1):
    print(f"  val_ds  element: image={x.shape}, label={y.shape}")
for x, y in test_ds.take(1):
    print(f"  test_ds element: image={x.shape}, label={y.shape}")
for x in proxy_ds.take(1):
    print(f"  proxy_ds element: image={x.shape} (unlabelled)")
for x, y in sup_ds.take(1):
    print(f"  sup_ds  element: image={x.shape}, label={y.shape}")

Model input size: (260, 260)
Total frames: 33688
val=5053, test=3368, proxy=1010, sup=673
val_ds, test_ds, proxy_ds, sup_ds ready (unbatched)
  val_ds  element: image=(260, 260, 3), label=()
  test_ds element: image=(260, 260, 3), label=()
  proxy_ds element: image=(260, 260, 3) (unlabelled)
  sup_ds  element: image=(260, 260, 3), label=()


# NOTE: RUN CELL BELOW ONLY FOR FIRST TIME

In [33]:
# ── Run the full federated learning cycle ────────────────────
#history = run_with_checkpoints(
   # cycle,
   # server_val_data=val_ds,
   # test_data=test_ds,
   # proxy_data=proxy_ds,
   # supervised_data=sup_ds,
#)

#print('\n✅ Federated Learning Cycle complete!')
#print(f'History keys: {list(history.keys())}')

# ONLY RUN CELL BELOW IF AND ONLY IF SESSION/KERNEL RESTARTED AND ALL OTHER VARIABLES ARE ALREADY SAVED

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  QUICK RESTART & RESUME — Run ONLY this cell after a Kaggle restart
#  (With persistence ON, all files in /kaggle/working/ survive)
#
#  This cell:
#   1. Re-configures GPU / mixed precision
#   2. Re-registers the custom Keras preprocessing
#   3. Re-imports & reloads all %%writefile modules
#   4. Rebuilds config, cycle, client_data, datasets
#   5. Defines checkpoint & memory management utilities
#   6. Runs training — auto-resumes from the last checkpoint
# ══════════════════════════════════════════════════════════════════

import sys, os, gc, glob, json, time, logging, importlib
import numpy as np

# ── 1. GPU / Accelerator setup ────────────────────────────────
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/usr/local/cuda"
tf.get_logger().setLevel(logging.ERROR)
logging.getLogger("absl").setLevel(logging.ERROR)

strategy = None
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver()
    tf.config.experimental_connect_to_cluster(resolver)
    tf.tpu.experimental.initialize_tpu_system(resolver)
    strategy = tf.distribute.TPUStrategy(resolver)
    ACCELERATOR = "TPU"
except (ValueError, tf.errors.NotFoundError):
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        strategy = tf.distribute.MirroredStrategy()
        ACCELERATOR = "GPU"
        for gpu in gpus:
            try:
                tf.config.experimental.set_memory_growth(gpu, True)
            except RuntimeError:
                pass
    else:
        strategy = tf.distribute.get_strategy()
        ACCELERATOR = "CPU"

if ACCELERATOR == "TPU":
    tf.keras.mixed_precision.set_global_policy("mixed_bfloat16")
elif ACCELERATOR == "GPU":
    tf.keras.mixed_precision.set_global_policy("mixed_float16")
    tf.config.optimizer.set_jit(True)

tf.config.optimizer.set_experimental_options({'layout_optimizer': False})

print(f"Accelerator: {ACCELERATOR} | Strategy: {strategy.__class__.__name__}")

# ── 2. Re-register custom Keras preprocessing ────────────────
from tensorflow.keras.saving import register_keras_serializable

@register_keras_serializable(package="custom_preproc")
def preprocess_input(x):
    # Must match the custom_objects key used in load_model();
    # EfficientNet's preprocess_input is identity in TF2 because
    # the model already contains built-in Rescaling/Normalization layers.
    return tf.keras.applications.efficientnet.preprocess_input(x)

print("✔ preprocess_input registered")

# ── 3. Re-import & reload all modules ────────────────────────
repo_root = "/kaggle/working"
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import flwr_adapter, evaluation_metrics, enhanced_client_selection
import update_validation, knowledge_distillation, client_reputation_ledger
import tff_data_utils, tff_learning_process, tff_federated_cycle

for mod in [flwr_adapter, evaluation_metrics, enhanced_client_selection,
            update_validation, knowledge_distillation, client_reputation_ledger,
            tff_data_utils, tff_learning_process, tff_federated_cycle]:
    importlib.reload(mod)

print("✔ All modules reloaded from disk")

# ── 4. Rebuild config ────────────────────────────────────────
from tff_federated_cycle import TFFCycleConfig, TFFFederatedLearningCycle, convert_to_tflite
from knowledge_distillation import DistillationConfig
from enhanced_client_selection import SelectionWeights
from update_validation import ContributionWeights, ClippingConfig
from client_reputation_ledger import ReputationConfig

config = TFFCycleConfig(
    model_path='phase3_best.weights.h5',
    num_devices=100, local_epochs=5, global_rounds=50,
    clients_per_round=15, local_batch_size=32,
    local_lr=5e-4, server_lr=0.1, eval_every=10,
    client_optimizer='adam', server_optimizer='sgd',
    enable_comparison=True, enable_distillation=True,
    distillation_config=DistillationConfig(
        temperature=2.0, lam=0.5, epochs=3, batch_size=32, learning_rate=1e-4),
    selection_weights=SelectionWeights(
        w_v=0.30, w_d=0.20, w_l=0.10, w_r=0.25, w_s=0.15),
    contribution_weights=ContributionWeights(
        alpha=0.35, beta=0.20, gamma=0.20, delta=0.25),
    clipping_config=ClippingConfig(clip_threshold=10.0, clip_value=5.0),
    harmful_threshold=0.02,
    reputation_config=ReputationConfig(
        theta=0.0, gamma=0.10, decay_rate=0.99, floor=0.05,
        ceiling=1.0, initial_reputation=0.50, penalty_factor=0.05),
    reports_dir='reports',
    tflite_output_path='effnet_global_tff_final.tflite',
)
print(f"✔ Config: {config.global_rounds} rounds, {config.num_devices} devices")

# ── 5. Rebuild cycle object ──────────────────────────────────
from tensorflow.keras.applications.efficientnet import preprocess_input as effnet_preprocess

# Verify which model file will be loaded
import hashlib as _hl
_model_stat = os.stat(config.model_path)
with open(config.model_path, "rb") as _mf:
    _model_hash = _hl.md5(_mf.read(1024 * 1024)).hexdigest()
print(f"📋 Model file: {config.model_path}")
print(f"   Size: {_model_stat.st_size / 1e6:.1f} MB | Modified: {time.ctime(_model_stat.st_mtime)} | Hash: {_model_hash}")

cycle = TFFFederatedLearningCycle(config)
cycle.load_global_model()

# ── Weight fingerprint: verify the new model was actually loaded ──
def _weight_fingerprint(model):
    """Quick fingerprint: L2 norm + mean of all weights."""
    w = model.get_weights()
    flat = np.concatenate([x.flatten() for x in w])
    return np.linalg.norm(flat), float(np.mean(flat))

_fp_norm, _fp_mean = _weight_fingerprint(cycle.global_model)
print(f"🔑 Model weight fingerprint after load_global_model():")
print(f"   L2 norm = {_fp_norm:.6f},  mean = {_fp_mean:.8f}")
print(f"   (Record these values. If they change after checkpoint loading, the checkpoint is overriding your new model.)")

# ── Freeze BatchNorm layers (important for EfficientNet in FL) ──
bn_count = 0
for layer in cycle.global_model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
        bn_count += 1

print(f"✔ Frozen {bn_count} BatchNorm layers for FL stability")

# Recompile after changing layer trainability
cycle.global_model.compile(
    optimizer=tf.keras.optimizers.Adam(config.local_lr),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

MODEL_IMG_SIZE = tuple(cycle.global_model.input_shape[1:3])
print(f"✔ Model loaded: {cycle.global_model.count_params():,} params, input {MODEL_IMG_SIZE}")

# ── 6. Rebuild client_data from persisted TFRecords ──────────
def _parse_example(example_proto):
    feature_desc = {
        "image/encoded": tf.io.FixedLenFeature([], tf.string),
        "image/format":  tf.io.FixedLenFeature([], tf.string),
        "label":         tf.io.FixedLenFeature([], tf.float32),
    }
    parsed = tf.io.parse_single_example(example_proto, feature_desc)
    image = tf.io.decode_jpeg(parsed["image/encoded"], channels=3)
    image = tf.image.resize(image, MODEL_IMG_SIZE)
    image = tf.cast(image, tf.float32)
    # No external preprocessing — the model has a built-in preprocess_input layer
    return image, parsed["label"]

CLIENT_DIR = "/kaggle/working/ffpp_tfrecord_clients"
client_files = sorted(glob.glob(os.path.join(CLIENT_DIR, "*.tfrecord")))
assert len(client_files) > 0, f"No .tfrecord files in {CLIENT_DIR} — persistence may not be on!"

client_data = {}
for fp in client_files:
    cid = os.path.splitext(os.path.basename(fp))[0]
    ds = tf.data.TFRecordDataset(fp, compression_type="GZIP", num_parallel_reads=1)
    client_data[cid] = ds.map(_parse_example, num_parallel_calls=1)

cycle.create_clients(client_data)
print(f"✔ {len(client_data)} clients rebuilt from TFRecords")

# ── 7. Setup TFF process & enhancement modules ──────────────
cycle.setup_tff_process()
cycle.setup_enhancement_modules()
print("✔ TFF process & enhancement modules initialised")

# ── 8. Rebuild val/test/proxy/sup datasets ───────────────────
FRAMES_DIR = "/kaggle/working/ffpp_frames"
all_paths, all_labels = [], []
for p in sorted(glob.glob(FRAMES_DIR + "/**/*.jpg", recursive=True)):
    all_paths.append(p)
    all_labels.append(1.0 if "fake" in p.lower() else 0.0)

assert len(all_paths) > 0, f"No frames in {FRAMES_DIR} — persistence may not be on!"

# deterministic shuffle
rng = np.random.RandomState(42)
idx = rng.permutation(len(all_paths))
all_paths  = [all_paths[i] for i in idx]
all_labels = [all_labels[i] for i in idx]

n = len(all_paths)
n_val   = max(1, int(n * 0.15))
n_test  = max(1, int(n * 0.10))
n_proxy = max(1, int(n * 0.03))
n_sup   = max(1, int(n * 0.02))

# split
val_paths  = all_paths[:n_val];                                           val_labels  = all_labels[:n_val]
test_paths = all_paths[n_val:n_val + n_test];                             test_labels = all_labels[n_val:n_val + n_test]
proxy_paths = all_paths[n_val + n_test:n_val + n_test + n_proxy]
sup_paths   = all_paths[n_val + n_test + n_proxy:n_val + n_test + n_proxy + n_sup]
sup_labels  = all_labels[n_val + n_test + n_proxy:n_val + n_test + n_proxy + n_sup]

# loaders
def _load_image(path, label):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.cast(tf.image.resize(img, MODEL_IMG_SIZE), tf.float32)
    return img, label

def _load_image_only(path):
    img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
    img = tf.cast(tf.image.resize(img, MODEL_IMG_SIZE), tf.float32)
    return img

# datasets (use AUTOTUNE)
val_ds   = tf.data.Dataset.from_tensor_slices((val_paths, val_labels)).map(_load_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds  = tf.data.Dataset.from_tensor_slices((test_paths, test_labels)).map(_load_image, num_parallel_calls=tf.data.AUTOTUNE)
proxy_ds = tf.data.Dataset.from_tensor_slices(proxy_paths).map(_load_image_only, num_parallel_calls=tf.data.AUTOTUNE)
sup_ds   = tf.data.Dataset.from_tensor_slices((sup_paths, sup_labels)).map(_load_image, num_parallel_calls=tf.data.AUTOTUNE)

print(f"✔ Datasets built: val={len(val_paths)}, test={len(test_paths)}, proxy={len(proxy_paths)}, sup={len(sup_paths)}")

# ── 8a. Diagnostics: verify model architecture & preprocessing ──
print("\n" + "="*60)
print("  DIAGNOSTIC: Model architecture")
print("="*60)
cycle.global_model.summary(print_fn=print)

print("\n" + "="*60)
print("  DIAGNOSTIC: Baseline accuracy (loaded model on val)")
print("="*60)
_diag_result = cycle.global_model.evaluate(
    val_ds.batch(config.local_batch_size), verbose=1, return_dict=True)
print(f"  Loaded model val accuracy: {_diag_result.get('accuracy', 0.0):.4f}")

print("\n" + "="*60)
print("  DIAGNOSTIC: Clone model accuracy (checks preprocessing consistency)")
print("="*60)
_diag_clone = tf.keras.models.clone_model(cycle.global_model)
_diag_clone.build(cycle.global_model.input_shape)
_diag_clone.set_weights(cycle.global_model.get_weights())
_diag_clone.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
_diag_clone_result = _diag_clone.evaluate(
    val_ds.batch(config.local_batch_size), verbose=1, return_dict=True)
print(f"  Clone  model val accuracy: {_diag_clone_result.get('accuracy', 0.0):.4f}")
print(f"  Loaded model val accuracy: {_diag_result.get('accuracy', 0.0):.4f}")
if abs(_diag_clone_result.get('accuracy', 0) - _diag_result.get('accuracy', 0)) > 0.01:
    print("  ⚠ MISMATCH: clone vs loaded accuracy differ — preprocessing inconsistency!")
else:
    print("  ✅ Loaded and cloned model accuracies match — preprocessing is consistent.")
del _diag_clone, _diag_result, _diag_clone_result
gc.collect()
print("="*60 + "\n")

# ══════════════════════════════════════════════════════════════
#  9. CHECKPOINT & MEMORY MANAGEMENT
# ══════════════════════════════════════════════════════════════
CHECKPOINT_DIR   = "/kaggle/working/fl_checkpoints"
CHECKPOINT_EVERY = 1

GPU_MEM_LIMIT_GB  = 15.0
CPU_RAM_LIMIT_GB  = 27.0
DISK_LIMIT_GB     = 55.0

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-8s | %(message)s")

def _bytes_to_gb(b):
    return b / (1024 ** 3)

def get_memory_usage():
    info = {}
    try:
        with open("/proc/meminfo") as f:
            meminfo = {}
            for line in f:
                parts = line.split()
                meminfo[parts[0].rstrip(":")] = int(parts[1]) * 1024
        total, avail = meminfo.get("MemTotal", 0), meminfo.get("MemAvailable", 0)
        info["cpu_ram_used_gb"] = _bytes_to_gb(total - avail)
        info["cpu_ram_total_gb"] = _bytes_to_gb(total)
        info["cpu_ram_avail_gb"] = _bytes_to_gb(avail)
    except Exception:
        info["cpu_ram_used_gb"] = -1
    try:
        if tf.config.list_physical_devices("GPU"):
            mi = tf.config.experimental.get_memory_info("GPU:0")
            info["gpu_used_gb"] = _bytes_to_gb(mi["current"])
            info["gpu_peak_gb"] = _bytes_to_gb(mi["peak"])
        else:
            info["gpu_used_gb"] = info["gpu_peak_gb"] = 0
    except Exception:
        info["gpu_used_gb"] = info["gpu_peak_gb"] = -1
    try:
        st = os.statvfs("/kaggle/working")
        dt, df = st.f_blocks * st.f_frsize, st.f_bavail * st.f_frsize
        info["disk_used_gb"] = _bytes_to_gb(dt - df)
        info["disk_total_gb"] = _bytes_to_gb(dt)
        info["disk_free_gb"] = _bytes_to_gb(df)
    except Exception:
        info["disk_used_gb"] = -1
    return info

def log_memory(prefix=""):
    m = get_memory_usage()
    parts, warns = [], []
    if m.get("cpu_ram_used_gb", -1) >= 0:
        parts.append(f"CPU RAM: {m['cpu_ram_used_gb']:.1f}/{m['cpu_ram_total_gb']:.1f} GiB")
        if m["cpu_ram_used_gb"] > CPU_RAM_LIMIT_GB:
            warns.append(f"⚠ CPU RAM {m['cpu_ram_used_gb']:.1f} GiB > {CPU_RAM_LIMIT_GB}")
    if m.get("gpu_used_gb", -1) >= 0:
        parts.append(f"GPU: {m['gpu_used_gb']:.2f} GiB (peak {m['gpu_peak_gb']:.2f})")
        if m["gpu_used_gb"] > GPU_MEM_LIMIT_GB:
            warns.append(f"⚠ GPU {m['gpu_used_gb']:.2f} GiB > {GPU_MEM_LIMIT_GB}")
    if m.get("disk_used_gb", -1) >= 0:
        parts.append(f"Disk: {m['disk_used_gb']:.1f}/{m['disk_total_gb']:.1f} GiB")
        if m["disk_used_gb"] > DISK_LIMIT_GB:
            warns.append(f"⚠ Disk {m['disk_used_gb']:.1f} GiB > {DISK_LIMIT_GB}")
    tag = f"[{prefix}] " if prefix else ""
    logger.info("%sMemory — %s", tag, " | ".join(parts))
    for w in warns:
        logger.warning(w)
    return m

def cleanup_memory():
    gc.collect()
    gc.collect()

def save_checkpoint(cycle_obj, round_num, all_reports=None):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    cycle_obj.global_model.save_weights(os.path.join(CHECKPOINT_DIR, "model_weights.weights.h5"))
    ser_hist = {}
    for k, v in cycle_obj.history.items():
        ser_hist[k] = [float(x) if isinstance(x, (np.floating, float)) else
                       int(x) if isinstance(x, (np.integer, int)) else x for x in v]
    with open(os.path.join(CHECKPOINT_DIR, "history.json"), "w") as f:
        json.dump(ser_hist, f, indent=2, default=str)
    cycle_obj.reputation_ledger.save(os.path.join(CHECKPOINT_DIR, "reputation_ledger.json"))
    meta = {"round": round_num, "global_rounds": cycle_obj.config.global_rounds,
            "timestamp": time.time(), "num_reports": len(all_reports) if all_reports else 0}
    with open(os.path.join(CHECKPOINT_DIR, "checkpoint_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    logger.info("💾 Checkpoint saved at round %d", round_num)

def load_checkpoint(cycle_obj):
    meta_path = os.path.join(CHECKPOINT_DIR, "checkpoint_meta.json")
    if not os.path.exists(meta_path):
        logger.info("No checkpoint found — starting from scratch.")
        return 0, 0
    with open(meta_path) as f:
        meta = json.load(f)
    saved_round = meta["round"]
    wp = os.path.join(CHECKPOINT_DIR, "model_weights.weights.h5")
    hp = os.path.join(CHECKPOINT_DIR, "history.json")
    lp = os.path.join(CHECKPOINT_DIR, "reputation_ledger.json")
    for p in [wp, hp, lp]:
        if not os.path.exists(p):
            logger.warning("Checkpoint incomplete (missing %s) — starting fresh.", p)
            return 0, 0
    cycle_obj.global_model.load_weights(wp)
    with open(hp) as f:
        cycle_obj.history = json.load(f)
    from client_reputation_ledger import ClientReputationLedger as _CRL
    cycle_obj.reputation_ledger = _CRL.load(lp)
    cycle_obj.basic_ledger = cycle_obj.reputation_ledger.as_basic_ledger()
    cycle_obj.selector.reputation_ledger = cycle_obj.basic_ledger
    cycle_obj.validator.reputation_ledger = cycle_obj.basic_ledger
    cycle_obj.tff_executor.set_keras_weights(cycle_obj.global_model.get_weights())
    cycle_obj.validator.global_model.set_weights(cycle_obj.global_model.get_weights())
    logger.info("🔄 Resuming from checkpoint at round %d", saved_round)
    return saved_round, meta.get("num_reports", 0)

# ── run_with_checkpoints (fixed) ───────────────────────────────────────────────
def run_with_checkpoints(cycle_obj, server_val_data, test_data,
                         proxy_data=None, supervised_data=None,
                         force_fresh_start=False):
    cfg = cycle_obj.config
    logger.info("FL Cycle: %d devices, %d rounds, checkpoint every %d",
                cfg.num_devices, cfg.global_rounds, CHECKPOINT_EVERY)

    if force_fresh_start:
        logger.info("force_fresh_start=True — ignoring any existing checkpoint.")
        resume_round, _ = 0, 0
    else:
        resume_round, _ = load_checkpoint(cycle_obj)
    all_reports = []

    # Baseline evaluation (only when starting fresh)
    if resume_round == 0:
        log_memory("Pre-baseline")
        baseline_report = cycle_obj.evaluator.evaluate(
            test_data=test_data,
            batch_size=cfg.local_batch_size,
            federated_round=0,
            extra_info={"stage": "baseline", "framework": "tff"}
        )
        cycle_obj.evaluator.save_report(baseline_report, tag="round_000_baseline_tff")
        logger.info("Baseline — Acc: %.4f, F1: %.4f, AUC: %.4f",
                    baseline_report.classification.accuracy,
                    baseline_report.classification.f1_macro,
                    baseline_report.classification.roc_auc)
        all_reports.append(baseline_report)
    else:
        logger.info("Skipping baseline (resuming from round %d)", resume_round)

    cycle_start = time.time()

    # Always initialise prev_weights from the current model so the
    # weight-diff diagnostic works even when resuming from a checkpoint.
    prev_weights = [w.copy() for w in cycle_obj.global_model.get_weights()]

    for t in range(resume_round + 1, cfg.global_rounds + 1):
        round_start = time.time()

        mem = get_memory_usage()
        if mem.get("cpu_ram_avail_gb", 999) < 2.0:
            logger.warning("⚠ Low RAM (%.1f GiB free) — GC before round %d",
                           mem["cpu_ram_avail_gb"], t)
            cleanup_memory()

        info = cycle_obj.execute_round(
            current_round=t,
            server_val_data=server_val_data,
            proxy_data=proxy_data,
            supervised_data=supervised_data
        )

        curr_weights = cycle_obj.global_model.get_weights()

        import numpy as np
        diff = np.linalg.norm(
            np.concatenate([w.flatten() for w in curr_weights]) -
            np.concatenate([w.flatten() for w in prev_weights])
        )
        
        print(f"🔍 Round {t} weight change L2:", diff)
        
        prev_weights = [w.copy() for w in curr_weights]

        # Update history
        cycle_obj.history.setdefault("round", []).append(t)
        cycle_obj.history.setdefault("tff_fedavg_accuracy", []).append(info.get("tff_fedavg_accuracy"))
        cycle_obj.history.setdefault("enhanced_accuracy", []).append(info.get("enhanced_accuracy"))
        cycle_obj.history.setdefault("selected_clients", []).append(info.get("selected"))
        cycle_obj.history.setdefault("num_accepted", []).append(info.get("num_accepted"))
        cycle_obj.history.setdefault("num_rejected", []).append(info.get("num_rejected"))
        cycle_obj.history.setdefault("distillation_loss", []).append(info.get("distillation_loss"))

        logger.info("  └─ Round %d finished in %.1fs", t, time.time() - round_start)

        # Periodic evaluation
        is_eval = (t % cfg.eval_every == 0) or (t == 1) or (t == cfg.global_rounds)
        if is_eval:
            rpt = cycle_obj.evaluator.evaluate(
                test_data=test_data,
                batch_size=cfg.local_batch_size,
                federated_round=t,
                extra_info={
                    "framework": "tff",
                    "tff_fedavg_acc": info.get("tff_fedavg_accuracy"),
                    "enhanced_acc": info.get("enhanced_accuracy"),
                    "accepted": info.get("num_accepted"),
                    "rejected": info.get("num_rejected")
                }
            )
            cycle_obj.evaluator.save_report(rpt, tag=f"tff_round_{t:03d}")
            all_reports.append(rpt)
            logger.info("  Eval R%d — Acc=%.4f F1=%.4f AUC=%.4f", t,
                        rpt.classification.accuracy,
                        rpt.classification.f1_macro,
                        rpt.classification.roc_auc)

        # Save checkpoint periodically and at final round
        if (t % CHECKPOINT_EVERY == 0) or (t == cfg.global_rounds):
            save_checkpoint(cycle_obj, t, all_reports)

        # Per-round cleanup
        cleanup_memory()
        log_memory(f"R{t}")

    total = time.time() - cycle_start
    logger.info("Cycle complete — %d rounds in %.1fs", cfg.global_rounds, total)

    # Post-run aggregation / reports
    if len(all_reports) > 1:
        try:
            cycle_obj.evaluator.save_comparison_report(all_reports)
        except Exception as e:
            logger.warning("Failed to save comparison report: %s", e)

    # Save the reputation ledger final
    try:
        from pathlib import Path as _P
        cycle_obj.reputation_ledger.save(str(_P(cfg.reports_dir) / "reputation_ledger_tff_final.json"))
    except Exception as e:
        logger.warning("Failed to save reputation ledger: %s", e)

    # Convert to tflite (normal and quantised)
    try:
        convert_to_tflite(cycle_obj.global_model, cfg.tflite_output_path, quantise=False)
    except Exception as e:
        logger.warning("TFLite conversion (float) failed: %s", e)

    try:
        convert_to_tflite(
            cycle_obj.global_model,
            cfg.tflite_output_path.replace(".tflite", "_quantised.tflite"),
            quantise=True
        )
    except Exception as e:
        logger.warning("TFLite conversion (quantised) failed: %s", e)

    # Final summary and memory logging
    try:
        cycle_obj._print_summary()
    except Exception:
        # _print_summary is optional — ignore if not present
        pass

    log_memory("Final")

    return cycle_obj.history

# ── 10. RUN — auto-resume from checkpoint if one exists ─────────────────────────
# Set force_fresh_start=True when using a new model file to prevent
# old checkpoint weights from overriding the freshly loaded model.
FORCE_FRESH_START = True   # ← set to False to resume from checkpoint

history = run_with_checkpoints(
    cycle_obj=cycle,
    server_val_data=val_ds,
    test_data=test_ds,
    proxy_data=proxy_ds,
    supervised_data=sup_ds,
    force_fresh_start=FORCE_FRESH_START,
)

print(f'✔ Federated cycle finished. History keys: {list(history.keys())}')

2026-03-11 23:48:20.182643: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773272900.360787      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773272900.412830      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773272900.865088      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773272900.865130      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773272900.865133      55 computation_placer.cc:177] computation placer alr

Accelerator: GPU | Strategy: MirroredStrategy
✔ preprocess_input registered
✔ All modules reloaded from disk
✔ Config: 50 rounds, 100 devices
📋 Model file: phase3_best.weights.h5
   Size: 215.3 MB | Modified: Wed Mar 11 23:42:55 2026 | Hash: 8771eafb42892ac5e263e780e8f99426


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:648: UserWarning: A total of 186 objects could not be loaded. Example error message for object <Normalization name=normalization, built=True>:

Layer 'normalization' expected 3 variables, but received 0 variables during loading. Expected: ['mean', 'variance', 'count']

List of objects that could not be loaded:
[<Normalization name=normalization, built=True>, <Conv2D name=stem_conv, built=True>, <BatchNormalization name=stem_bn, built=True>, <DepthwiseConv2D name=block1a_dwconv, built=True>, <BatchNormalization name=block1a_bn, built=True>, <Conv2D name=block1a_se_reduce, built=True>, <Conv2D name=block1a_se_expand, built=True>, <Conv2D name=block1a_project_conv, built=True>, <BatchNormalization name=block1a_project_bn, built=True>, <DepthwiseConv2D name=block1b_dwconv, built=True>, <BatchNormalization name=block1b_bn, built=True>, <Conv2D name=block1b_se_reduce, built=True>, <Conv2D name=block1b_se_expand, built=Tru

🔑 Model weight fingerprint after load_global_model():
   L2 norm = 404.062295,  mean = 0.00867402
   (Record these values. If they change after checkpoint loading, the checkpoint is overriding your new model.)
✔ Frozen 69 BatchNorm layers for FL stability
✔ Model loaded: 7,769,978 params, input (260, 260)


2026-03-11 23:49:12,903 | INFO     | Created 100 federated clients.


...

Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 260, 260,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 260, 260,  │          0 │ input_layer[0][0] │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 260, 260,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 261, 261,  │          0 │

I0000 00:00:1773272961.168191     120 service.cc:152] XLA service 0x79f264002af0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1773272961.168247     120 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1773272962.448609     120 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1773272980.500417     120 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


158/158 ━━━━━━━━━━━━━━━━━━━━ 67s 210ms/step - accuracy: 0.5028 - loss: 0.6931
  Loaded model val accuracy: 0.4946

  DIAGNOSTIC: Clone model accuracy (checks preprocessing consistency)
158/158 ━━━━━━━━━━━━━━━━━━━━ 39s 124ms/step - accuracy: 0.5028 - loss: 0.6931
  Clone  model val accuracy: 0.4946
  Loaded model val accuracy: 0.4946
  ✅ Loaded and cloned model accuracies match — preprocessing is consistent.


2026-03-11 23:51:03,908 | INFO     | FL Cycle: 100 devices, 50 rounds, checkpoint every 1
2026-03-11 23:51:03,908 | INFO     | force_fresh_start=True — ignoring any existing checkpoint.
2026-03-11 23:51:03,910 | INFO     | [Pre-baseline] Memory — CPU RAM: 3.1/31.4 GiB | GPU: 0.15 GiB (peak 1.97) | Disk: 2.7/19.5 GiB
2026-03-11 23:51:03,910 | INFO     | Starting evaluation for 'effnet_global_tff' …


2026-03-11 23:52:13,254 | INFO     | Classification — Acc: 0.5042 | F1-macro: 0.3352 | ROC-AUC: 0.5000
2026-03-11 23:53:00,076 | INFO     | Latency — mean: 441.12 ms | p95: 460.45 ms | p99: 487.53 ms
2026-03-11 23:53:00,207 | INFO     | Model size — params: 7,769,978 | disk: 29.64 MB
2026-03-11 23:53:00,209 | INFO     | Reports saved → effnet_global_tff_20260311_235300_round_000_baseline_tff.json  &  effnet_global_tff_20260311_235300_round_000_baseline_tff.txt
2026-03-11 23:53:00,210 | INFO     | Baseline — Acc: 0.5042, F1: 0.3352, AUC: 0.5000
2026-03-11 23:53:00,377 | INFO     | ── Round 1 / 50 ──
2026-03-11 23:53:00,379 | INFO     | Round 1 — selected 15 / 100 clients: ['client_24', 'client_82', 'client_70', 'client_50', 'client_31', 'client_59', 'client_32', 'client_75', 'client_35', 'client_39', 'client_23', 'client_57', 'client_7', 'client_26', 'client_15']
/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:160: UserWarning: Your input ran out of data; in

In [ ]:
# ── Clear old checkpoints & reports (run this when switching to a new model) ──
#import os, shutil

#CHECKPOINT_DIR = "/kaggle/working/fl_checkpoints"
#REPORTS_DIR = "/kaggle/working/reports"

#for d in [CHECKPOINT_DIR, REPORTS_DIR]:
#    if os.path.exists(d):
#        n_files = sum(len(files) for _, _, files in os.walk(d))
#        shutil.rmtree(d)
#        print(f"🗑  Deleted {d} ({n_files} files)")
#    else:
#        print(f"   {d} — already clean")

#print("\n✅ Old checkpoints & reports cleared. The next training run will start fresh.")


## 9. Results Visualization

Visualise the training history with matplotlib.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import json, os

# ── Recover history if cell 57 failed partway ──────────────────
if 'history' not in dir() or history is None:
    # Try 1: grab from the cycle object
    if 'cycle' in dir() and hasattr(cycle, 'history') and cycle.history:
        history = cycle.history
        print("ℹ Recovered history from cycle.history")
    else:
        # Try 2: load from the most recent checkpoint
        _ckpt_path = "/kaggle/working/fl_checkpoints/history.json"
        if os.path.exists(_ckpt_path):
            with open(_ckpt_path) as _f:
                history = json.load(_f)
            print(f"ℹ Loaded history from checkpoint ({_ckpt_path})")
        else:
            raise RuntimeError(
                "Cannot find training history. "
                "Re-run cell 57 successfully, or ensure a checkpoint exists at "
                f"{_ckpt_path}"
            )

# ── 9a. Accuracy: TFF FedAvg vs Enhanced ────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

rounds = history['round']
tff_acc = history['tff_fedavg_accuracy']
enh_acc = history['enhanced_accuracy']

# Plot 1: Accuracy comparison
ax = axes[0, 0]
if tff_acc[0] is not None:
    ax.plot(rounds, tff_acc, 'b--o', label='TFF FedAvg', markersize=3, alpha=0.7)
ax.plot(rounds, enh_acc, 'r-s', label='Enhanced (Ours)', markersize=3, alpha=0.7)
ax.set_xlabel('Federated Round')
ax.set_ylabel('Accuracy')
ax.set_title('TFF FedAvg vs Enhanced Accuracy')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Accuracy improvement (delta)
ax = axes[0, 1]
if tff_acc[0] is not None:
    deltas = [e - t for e, t in zip(enh_acc, tff_acc) if t is not None]
    ax.bar(rounds[:len(deltas)], deltas, color=['green' if d >= 0 else 'red' for d in deltas], alpha=0.7)
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xlabel('Federated Round')
    ax.set_ylabel('Accuracy Improvement (Δ)')
    ax.set_title('Enhanced − FedAvg Accuracy Delta')
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'Comparison mode disabled', ha='center', va='center', transform=ax.transAxes)

# Plot 3: Accepted vs Rejected updates
ax = axes[1, 0]
ax.bar(rounds, history['num_accepted'], label='Accepted', color='green', alpha=0.7)
ax.bar(rounds, history['num_rejected'], bottom=history['num_accepted'],
       label='Rejected', color='red', alpha=0.7)
ax.set_xlabel('Federated Round')
ax.set_ylabel('Number of Updates')
ax.set_title('Accepted vs Rejected Client Updates')
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 4: Distillation loss
ax = axes[1, 1]
kd_losses = history['distillation_loss']
kd_rounds = [r for r, l in zip(rounds, kd_losses) if l is not None]
kd_values = [l for l in kd_losses if l is not None]
if kd_values:
    ax.plot(kd_rounds, kd_values, 'g-^', label='KD Loss', markersize=4)
    ax.set_xlabel('Federated Round')
    ax.set_ylabel('Distillation Loss')
    ax.set_title('Knowledge Distillation Loss Over Rounds')
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, 'No distillation data', ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.savefig('results_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Results plot saved to results_overview.png')

In [ ]:
# ── 9b. Reputation Distribution ──────────────────────────────
import matplotlib.pyplot as plt

reps = cycle.reputation_ledger.all_reputations()
rep_values = list(reps.values())
stats = cycle.reputation_ledger.statistics()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(rep_values, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(stats['mean_reputation'], color='red', linestyle='--',
           label=f"Mean: {stats['mean_reputation']:.3f}")
ax.axvline(stats['median_reputation'], color='orange', linestyle='--',
           label=f"Median: {stats['median_reputation']:.3f}")
ax.set_xlabel('Reputation Score')
ax.set_ylabel('Number of Clients')
ax.set_title('Final Client Reputation Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Top/Bottom clients
ax = axes[1]
ranked = cycle.reputation_ledger.ranked()
top_10 = ranked[:10]
bottom_10 = ranked[-10:]
combined = top_10 + bottom_10
names = [c[0] for c in combined]
values = [c[1] for c in combined]
colors = ['green'] * len(top_10) + ['red'] * len(bottom_10)
ax.barh(range(len(combined)), values, color=colors, alpha=0.7)
ax.set_yticks(range(len(combined)))
ax.set_yticklabels(names, fontsize=7)
ax.set_xlabel('Reputation Score')
ax.set_title('Top 10 & Bottom 10 Clients by Reputation')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('reputation_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'\nReputation stats:')
for k, v in stats.items():
    print(f'  {k}: {v:.4f}' if isinstance(v, float) else f'  {k}: {v}')


In [ ]:
# ── 9c. Round-by-round Summary Table ─────────────────────────
print(f"{'Round':>6} | {'TFF Acc':>8} | {'Enh Acc':>8} | "
      f"{'Delta':>8} | {'Accepted':>8} | {'Rejected':>8} | {'KD Loss':>10}")
print('-' * 72)

for i, rnd in enumerate(history['round']):
    tff_a = history['tff_fedavg_accuracy'][i]
    enh_a = history['enhanced_accuracy'][i]
    tff_str = f'{tff_a:.4f}' if tff_a is not None else '  N/A '
    delta = f'{enh_a - tff_a:+.4f}' if tff_a is not None else '  N/A '
    kd = history['distillation_loss'][i]
    kd_str = f'{kd:.5f}' if kd is not None else '    N/A   '
    print(f'{rnd:>6} | {tff_str:>8} | {enh_a:>8.4f} | '
          f'{delta:>8} | {history["num_accepted"][i]:>8} | '
          f'{history["num_rejected"][i]:>8} | {kd_str:>10}')


## 10. Final Evaluation Report

Run a comprehensive final evaluation on the test set.


In [ ]:
from evaluation_metrics import evaluate_and_report
import os, glob, json
import numpy as np
import tensorflow as tf

# ── Recover cycle + datasets if not in scope (e.g. after kernel restart) ──
if 'cycle' not in dir():
    print("ℹ 'cycle' not found — rebuilding from checkpoint …")
    from tff_federated_cycle import TFFCycleConfig, TFFFederatedLearningCycle
    from knowledge_distillation import DistillationConfig
    from enhanced_client_selection import SelectionWeights
    from update_validation import ContributionWeights, ClippingConfig
    from client_reputation_ledger import ReputationConfig

    config = TFFCycleConfig(
        model_path='phase3_best.weights.h5',
        num_devices=100, local_epochs=5, global_rounds=50,
        clients_per_round=15, local_batch_size=32,
        local_lr=5e-4, server_lr=0.1, eval_every=10,
        client_optimizer='adam', server_optimizer='sgd',
        enable_comparison=True, enable_distillation=True,
        distillation_config=DistillationConfig(
            temperature=2.0, lam=0.5, epochs=3, batch_size=32, learning_rate=1e-4),
        selection_weights=SelectionWeights(
            w_v=0.30, w_d=0.20, w_l=0.10, w_r=0.25, w_s=0.15),
        contribution_weights=ContributionWeights(
            alpha=0.35, beta=0.20, gamma=0.20, delta=0.25),
        clipping_config=ClippingConfig(clip_threshold=10.0, clip_value=5.0),
        harmful_threshold=0.02,
        reputation_config=ReputationConfig(
            theta=0.0, gamma=0.10, decay_rate=0.99, floor=0.05,
            ceiling=1.0, initial_reputation=0.50, penalty_factor=0.05),
        reports_dir='reports',
        tflite_output_path='effnet_global_tff_final.tflite',
    )

    cycle = TFFFederatedLearningCycle(config)
    cycle.load_global_model()

    # Restore weights from checkpoint if available
    _ckpt_weights = "/kaggle/working/fl_checkpoints/model_weights.weights.h5"
    if os.path.exists(_ckpt_weights):
        cycle.global_model.load_weights(_ckpt_weights)
        print(f"✔ Model weights restored from {_ckpt_weights}")
    else:
        print("⚠ No checkpoint weights found — using base model weights")

if 'test_ds' not in dir():
    print("ℹ 'test_ds' not found — rebuilding datasets …")
    MODEL_IMG_SIZE = tuple(cycle.global_model.input_shape[1:3])

    FRAMES_DIR = "/kaggle/working/ffpp_frames"
    all_paths, all_labels = [], []
    for p in sorted(glob.glob(FRAMES_DIR + "/**/*.jpg", recursive=True)):
        all_paths.append(p)
        all_labels.append(1.0 if "fake" in p.lower() else 0.0)

    assert len(all_paths) > 0, f"No frames in {FRAMES_DIR} — persistence may not be on!"

    rng = np.random.RandomState(42)
    idx = rng.permutation(len(all_paths))
    all_paths  = [all_paths[i] for i in idx]
    all_labels = [all_labels[i] for i in idx]

    n = len(all_paths)
    n_val  = max(1, int(n * 0.15))
    n_test = max(1, int(n * 0.10))

    test_paths  = all_paths[n_val:n_val + n_test]
    test_labels = all_labels[n_val:n_val + n_test]

    def _load_image(path, label):
        img = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        img = tf.cast(tf.image.resize(img, MODEL_IMG_SIZE), tf.float32)
        return img, label

    test_ds = tf.data.Dataset.from_tensor_slices(
        (test_paths, test_labels)
    ).map(_load_image, num_parallel_calls=tf.data.AUTOTUNE)
    print(f"✔ test_ds rebuilt: {len(test_paths)} samples")

# Full evaluation on the test set
final_report = evaluate_and_report(
    model=cycle.global_model,
    test_data=test_ds,
    model_name='effnet_global_tff_final',
    reports_dir=config.reports_dir,
    federated_round=config.global_rounds,
    extra_info={
        'framework': 'tensorflow_federated',
        'num_devices': config.num_devices,
        'total_rounds': config.global_rounds,
        'local_epochs': config.local_epochs,
        'enhancement_modules': [
            'enhanced_client_selection',
            'update_validation',
            'knowledge_distillation',
            'client_reputation_ledger',
        ],
    },
    tag='final',
)

print('\n' + '=' * 60)
print('  FINAL EVALUATION RESULTS')
print('=' * 60)
cls = final_report.classification
print(f'  Accuracy:         {cls.accuracy:.4f}')
print(f'  F1 Score (macro): {cls.f1_macro:.4f}')
print(f'  F1 Score (wt.):   {cls.f1_weighted:.4f}')
print(f'  Precision:        {cls.precision_macro:.4f}')
print(f'  Recall:           {cls.recall_macro:.4f}')
print(f'  ROC-AUC:          {cls.roc_auc:.4f}')
print(f'\n  Confusion Matrix:')
if cls.confusion_matrix:
    print(f'    TN={cls.confusion_matrix[0][0]:>5}  FP={cls.confusion_matrix[0][1]:>5}')
    print(f'    FN={cls.confusion_matrix[1][0]:>5}  TP={cls.confusion_matrix[1][1]:>5}')

lat = final_report.latency
print(f'\n  Inference Latency:')
print(f'    Mean: {lat.mean_ms:.2f} ms | P95: {lat.p95_ms:.2f} ms')

sz = final_report.model_size
print(f'\n  Model Size:')
print(f'    Parameters: {sz.total_params:,}')
print(f'    File size:  {sz.file_size_mb:.2f} MB')
print('=' * 60)

In [ ]:
# Visualization cell — run this immediately after the cell that defines `final_report`
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Safely extract objects (works if final_report exists in the current scope) ---
cls = getattr(final_report, 'classification', None)
lat = getattr(final_report, 'latency', None)
sz  = getattr(final_report, 'model_size', None)

# --- Classification metrics (barh) ---
metrics = {}
if cls is not None:
    metrics['Accuracy']           = getattr(cls, 'accuracy', np.nan)
    metrics['F1 (macro)']        = getattr(cls, 'f1_macro', np.nan)
    metrics['F1 (weighted)']     = getattr(cls, 'f1_weighted', np.nan)
    metrics['Precision (macro)'] = getattr(cls, 'precision_macro', np.nan)
    metrics['Recall (macro)']    = getattr(cls, 'recall_macro', np.nan)
    metrics['ROC-AUC']           = getattr(cls, 'roc_auc', np.nan)

metrics_df = pd.DataFrame.from_dict(metrics, orient='index', columns=['value'])

if not metrics_df.empty:
    plt.figure(figsize=(7, 4.2))
    plt.barh(metrics_df.index, metrics_df['value'])
    plt.xlim(0, 1)
    plt.xlabel('Score')
    plt.title('Classification metrics')
    # annotate values
    for i, v in enumerate(metrics_df['value']):
        if np.isfinite(v):
            plt.text(v + 0.01, i, f'{v:.3f}', va='center')
    plt.tight_layout()
    plt.show()
else:
    print("No classification metrics found in final_report.classification")

# --- Confusion matrix (2x2 heatmap) ---
cm = None
if cls is not None:
    cm = getattr(cls, 'confusion_matrix', None)

if cm:
    cm_arr = np.array(cm)
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm_arr, interpolation='nearest')
    ax.set_title('Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Negative', 'Positive']); ax.set_yticklabels(['Negative', 'Positive'])
    # annotate each cell
    for (i, j), val in np.ndenumerate(cm_arr):
        ax.text(j, i, int(val), ha='center', va='center')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()
else:
    print("No confusion_matrix found (or it is empty) in final_report.classification")

# --- Latency (mean vs P95) ---
if lat is not None:
    mean_ms = getattr(lat, 'mean_ms', np.nan)
    p95_ms  = getattr(lat, 'p95_ms', np.nan)
    plt.figure(figsize=(4.2, 3.2))
    plt.bar(['mean', 'p95'], [mean_ms, p95_ms])
    plt.ylabel('Latency (ms)')
    plt.title('Inference latency')
    for i, v in enumerate([mean_ms, p95_ms]):
        if np.isfinite(v):
            # small vertical offset for label
            offset = max(0.01, 0.01 * abs(v))
            plt.text(i, v + offset, f'{v:.2f} ms', ha='center')
    plt.tight_layout()
    plt.show()
else:
    print("No latency info found in final_report.latency")

# --- Model size (Parameters in millions, File size MB) ---
if sz is not None:
    params = getattr(sz, 'total_params', None)
    file_mb = getattr(sz, 'file_size_mb', None)

    labels = []
    values = []
    if params is not None:
        labels.append('Parameters (M)')
        values.append(params / 1_000_000.0)
    if file_mb is not None:
        labels.append('File size (MB)')
        values.append(file_mb)

    if values:
        fig, ax = plt.subplots(figsize=(5.2, 2.6))
        ax.bar(labels, values)
        ax.set_title('Model size')
        for i, v in enumerate(values):
            if np.isfinite(v):
                ax.text(i, v + max(0.01, 0.01 * abs(v)), f'{v:,.2f}', ha='center')
        plt.tight_layout()
        plt.show()
    else:
        print("Model size fields are missing in final_report.model_size")
else:
    print("No model_size found in final_report")

# --- Tabular summary for quick copy/paste or inspection ---
# Combine top-level summary info into a small DataFrame
summary_data = []
if metrics_df is not None and not metrics_df.empty:
    for k, v in metrics_df['value'].items():
        summary_data.append({'metric': k, 'value': v})

if lat is not None:
    summary_data.append({'metric': 'latency_mean_ms', 'value': getattr(lat, 'mean_ms', np.nan)})
    summary_data.append({'metric': 'latency_p95_ms',  'value': getattr(lat, 'p95_ms', np.nan)})

if sz is not None:
    summary_data.append({'metric': 'total_params',   'value': getattr(sz, 'total_params', np.nan)})
    summary_data.append({'metric': 'file_size_mb',   'value': getattr(sz, 'file_size_mb', np.nan)})

if summary_data:
    summary_df = pd.DataFrame(summary_data)
    print("\nSummary table:")
    display(summary_df)
else:
    print("No summary data available to display.")

## 11. Export & Download Results

Download the trained model, TF Lite exports, reports, and
reputation ledger.


In [ ]:
import os
import zipfile

# ── Save final Keras model ───────────────────────────────────
cycle.global_model.save('effnet_global_tff_trained.h5')
print('Saved: effnet_global_tff_trained.h5')

# ── List all output files ────────────────────────────────────
output_files = []

# Model files
for f in ['effnet_global_tff_trained.h5',
          config.tflite_output_path,
          config.tflite_output_path.replace('.tflite', '_quantised.tflite')]:
    if os.path.exists(f):
        output_files.append(f)

# Reports
if os.path.isdir('reports'):
    for f in os.listdir('reports'):
        output_files.append(os.path.join('reports', f))

# Visualisation plots
for f in ['results_overview.png', 'reputation_distribution.png']:
    if os.path.exists(f):
        output_files.append(f)

print(f'\nOutput files ({len(output_files)} total):')
for f in output_files:
    size = os.path.getsize(f) if os.path.exists(f) else 0
    print(f'  {f} ({size / 1024:.1f} KB)')

# ── Create ZIP archive ───────────────────────────────────────
zip_name = 'fl_cycle_results.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in output_files:
        zf.write(f)
print(f'\nCreated: {zip_name} ({os.path.getsize(zip_name) / 1024:.1f} KB)')


In [ ]:
# ── Download results ─────────────────────────────────────────
from google.colab import files
files.download('fl_cycle_results.zip')
print('Download initiated.')


---

## Appendix A: Quick Demo (Reduced Settings)

If the full 100-device / 50-round cycle takes too long, use this
cell for a quick smoke-test with reduced parameters.


In [ ]:
# ── Quick Demo: 8 clients, 3 rounds ─────────────────────────
# Uncomment and run this cell for a faster test.
# Comment out Section 8 above to avoid running the full cycle.

# from tff_federated_cycle import TFFFederatedLearningCycle, TFFCycleConfig
# from tff_data_utils import generate_synthetic_data, generate_proxy_data, partition_data_iid_tff
# from knowledge_distillation import DistillationConfig
# import numpy as np, tensorflow as tf
#
# np.random.seed(42); tf.random.set_seed(42)
#
# quick_config = TFFCycleConfig(
#     model_path='phase3_best.weights.h5',
#     num_devices=8, local_epochs=1, global_rounds=3,
#     clients_per_round=4, local_batch_size=16,
#     local_lr=1e-3, eval_every=1,
#     enable_distillation=True, enable_comparison=True,
#     distillation_config=DistillationConfig(
#         temperature=3.0, lam=0.7, epochs=2,
#         batch_size=16, learning_rate=1e-3),
# )
#
# quick_cycle = TFFFederatedLearningCycle(quick_config)
# model = quick_cycle.load_global_model()
# input_shape = model.input_shape[1:]
# quick_config.input_shape = input_shape
#
# train = generate_synthetic_data(8 * 30, input_shape, seed=10)
# val   = generate_synthetic_data(100, input_shape, seed=20)
# test  = generate_synthetic_data(120, input_shape, seed=30)
# proxy = generate_proxy_data(80, input_shape, seed=40)
# sup   = generate_synthetic_data(60, input_shape, seed=50)
#
# client_data = partition_data_iid_tff(train, 8)
# quick_cycle.create_clients(client_data)
# quick_cycle.setup_tff_process()
# quick_cycle.setup_enhancement_modules()
#
# history = quick_cycle.run(
#     server_val_data=val, test_data=test,
#     proxy_data=proxy, supervised_data=sup,
# )
#
# for r, t_a, e_a in zip(history['round'], history['tff_fedavg_accuracy'], history['enhanced_accuracy']):
#     t_s = f'{t_a:.4f}' if t_a else 'N/A'
#     print(f'  Round {r}: FedAvg={t_s} | Enhanced={e_a:.4f}')
